# Additional chronological evaluation: 2021–2022

This notebook evaluates the unchanged lagged hybrid, compact U-Net and fixed **75% hybrid + 25% U-Net** combination on one additional two-year block. It tests whether the development improvement persists in another period, without searching weights or changing architecture.

**Status: completed on Kaggle, version `354026629`.** The fixed blend reduced pooled RMSE from **1.796784 to 1.795289 mm/day (0.0832%)**, but increased MAE and absolute bias and worsened RMSE in 2022. The operational hybrid remains unchanged. Read the [complete result and decision](RESULTS.md). The failed first attempt and its input-coverage repair remain recorded below.

## What this evaluation can establish

The records reviewed for this step contain development scores for 2007–2020, competition feedback for 2023–2024, and a separate retrospective 2025 evaluation. The 2021–2022 observations were already used by earlier final training runs. No separate 2021–2022 comparison was found in those reviewed records; absence of a recorded score does not prove the period was never inspected.

For that reason this is an **additional retrospective chronological evaluation, not an untouched independent holdout**. Freezing predictions before scoring is an execution safeguard, not proof that historical targets were unknown to the project. A convincing future-skill assessment still requires prospectively frozen predictions with verified source availability.

The existing workflow deliberately supports its seven original blocks only. This extension has a separate locked adapter and notebook so that adding a period cannot silently reinterpret the original run or bypass its baseline-reproduction checks.

## Fixed experimental design

| Item | Setting |
|---|---|
| Evaluation | January 2021–December 2022, 24 months |
| Outer training and rainfall reference | October 1990–September 2020, 360 months |
| Inner epoch-selection validation | October 2018–September 2020 |
| Inner training/reference cutoff | June 2018; four-month label lag |
| Atmospheric inputs | T−4 |
| Ocean indices | T−3 |
| Seasonal initialization | T−1, from the existing audited SEAS5/CFSv2 snapshots |
| Network | Original channels 16/32/64, seed 42, AdamW, maximum 40 epochs, patience 6 |
| Final neural fitting | Fresh initialization; epoch count selected only inside the outer training window |
| Hybrid | Same two LightGBMs and local ridge, refitted on the stated outer window |
| Combination | Exactly 75% hybrid + 25% U-Net; no new clipping or recalibration |
| Primary comparison | Pooled full-grid RMSE of the blend against the refitted hybrid |

All models remain fixed throughout the 24 evaluation months. Source fields update for each target month according to the lags. The final operational model trained through December 2022 is **not** reused for this earlier block. The original [U-Net protocol](../spatial_unet/protocol.json) is hash-pinned by the [extension protocol](protocol.json).

## Run in Kaggle

1. Import [temporal_extension.ipynb](temporal_extension.ipynb) into a **new private notebook**.
2. Attach the official competition dataset, `worcap-seas5-dados`, and `worcap-cfsv2-dados`. Each seasonal input needs its manifest and both `desenvolvimento` and `somente_ajuste_final` NetCDF files. The `teste` partitions and SST are not used. The seven old model outputs are unnecessary because this experiment refits the models with a new cutoff.
3. Enable a GPU. Use the original baseline environment: NumPy 2.0.2, pandas 2.3.3, xarray 2025.12.0 and LightGBM 4.6.0. PyTorch and the NetCDF readers must also be present. Preflight stops on incompatible baseline versions or unavailable CUDA; it does not replace packages automatically. Actual PyTorch/GPU versions are recorded.
4. Run cells in order. The check cell runs small synthetic tests and checks the input hashes. The fitting cell trains the hybrid, selects neural epochs internally, refits the network and freezes predictions. The evaluation cell then verifies that package before decoding the target rainfall.
5. Read `report.html`, download `temporal_extension_reports.zip`, and save the notebook version **with all outputs**. The small ZIP contains reports and verification metadata; the full saved output also retains models and prediction maps.

No Internet, CDS key, new downloads or GitHub login are needed when the prepared datasets are attached. This step performs one new outer block, with one inner neural fit and one final neural fit. It can take longer than evaluation of saved maps. The model prints each epoch's timing and progress.

If auto-discovery finds duplicate inputs, set the three directory variables explicitly. Existing runs are not overwritten. A failed fitting run is preserved and needs a new output directory. If predictions froze successfully but the session ended before evaluation, use the restore cell to point to the complete saved run; do not refit it just to obtain the report. An interrupted evaluation leaves `opened.json` and must not be represented as unopened data.

## Local entry points

Use the [existing model dependencies](../spatial_unet/requirements.txt); avoid installing evaluation-only pins over a training environment.

```bash
python -m unittest research.temporal_extension.test_extension research.spatial_unet.test_spatial -v
python -m research.temporal_extension.experiment check --official /data/official --seas5 /data/seas5 --cfsv2 /data/cfsv2 --device cuda
python -m research.temporal_extension.experiment freeze --output /outputs/new_run --official /data/official --seas5 /data/seas5 --cfsv2 /data/cfsv2 --device cuda
python -m research.temporal_extension.experiment evaluate --output /outputs/new_run --observations /data/official/treino_tp.nc
```

The fit path decodes only the training rainfall window. Forecast inference uses an adapter with no target-rainfall access. Source files are still byte-hashed in full for provenance, including the shared rainfall archive; hashing does not decode those future values into model inputs. Scaling and climatologies are fitted inside the appropriate training window. Existing monthly-map inference is checked against the target-free adapter on synthetic data with nonzero network predictions.

## Artifacts and interpretation

`signature.json`, `plan.json`, `split.json`, `source_calendar.csv` and `code/` record the experiment before fitting. The model folders preserve parameters, scalers, references and histories. `predictions.nc` contains all four forecast fields; `frozen.json` records their verified package hashes before evaluation. The local record is not an independent timestamp or tamper-proof ledger.

The separate `evaluation/` directory contains pooled global, annual, calendar-month and latitude-band metrics, monthly error sums, the opening record, a plot and a portable HTML report. Lower RMSE alone does not hide worsened MAE or absolute bias. Two years provide limited temporal diversity, the supplied grid includes ocean, and consolidated source files do not establish historical publication vintages.

No result automatically replaces the operational hybrid, changes the ongoing source gate or issues a live forecast. Record the result once, including a negative result, before planning another hypothesis.

## Preparation checks

The original fourteen synthetic checks passed from the notebook's extracted source bundle in an isolated directory. They cover the original spatial model and this extension, including target-free inference equivalence, training-window selection, rejection of modified frozen predictions, pooled-error metrics and complete report generation. The [initial validation record](evidence/validation.json) describes that preparation, not a successful real-data run.

### Missing-index fix

Kaggle version `354007635` stopped at the first hybrid forecast with `KeyError: 2020-10-01`. The bundled development index table ended in September 2020, so its structural checks were insufficient for the new forecast calendar. No outer scores were produced. This was an input-loading defect, not a model result.

The corrected loader uses the already preserved [NOAA archive](../../competition/metadata/NOAA/indices_noaa.csv), pins its SHA-256, and checks all required T−3 origins before fitting. All 526 shared months match the development table exactly; the 360 training origins are unchanged. Only July 1990–September 2022 is selected. There is no download, interpolation, new feature or protocol/weight change. `indices.json` records the source, overlap check and selected coverage.

Two regression checks verify first/last forecast coverage and reject the old truncated table before fitting. The [repair validation record](evidence/coverage_fix_validation.json) records the results. The failed Kaggle version remains preserved; rerunning this correction uses a fresh output directory.


In [ ]:
from pathlib import Path
import hashlib, json, sys, runpy, shutil
BASE = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()/'outputs'
RUN_ROOT = BASE/'temporal_extension'
CODE_ROOT = RUN_ROOT/'code'
CODE_ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE = {'competition/metadata/NOAA/indices_noaa.csv': 'time_origem,nino34,nino12,tna,tsa\n1976-12-01,0.64,0.25,-0.60,-0.47\n1977-01-01,0.84,0.29,-0.40,-0.53\n1977-02-01,0.55,-0.14,-0.40,-0.27\n1977-03-01,0.40,-0.38,-0.27,-0.34\n1977-04-01,-0.25,-0.68,-0.25,-0.34\n1977-05-01,-0.03,-0.45,-0.28,-0.39\n1977-06-01,0.36,0.08,-0.20,-0.41\n1977-07-01,0.45,0.03,-0.42,-0.40\n1977-08-01,0.25,-0.52,-0.62,-0.19\n1977-09-01,0.44,-0.50,-0.61,-0.12\n1977-10-01,0.89,-0.19,-0.66,-0.31\n1977-11-01,1.03,-0.29,-0.57,-0.27\n1977-12-01,1.09,-0.56,-0.44,-0.34\n1978-01-01,0.72,-0.07,-0.13,-0.52\n1978-02-01,0.51,-0.35,0.02,-0.56\n1978-03-01,0.06,-0.77,0.09,-0.80\n1978-04-01,-0.37,-0.37,0.10,-0.96\n1978-05-01,-0.50,-0.54,-0.13,-0.99\n1978-06-01,-0.45,-0.86,-0.39,-0.78\n1978-07-01,-0.48,-0.40,-0.60,-0.33\n1978-08-01,-0.62,-0.64,-0.66,-0.40\n1978-09-01,-0.60,-0.30,-0.59,-0.50\n1978-10-01,-0.34,-0.43,-0.35,-0.61\n1978-11-01,-0.15,0.04,-0.30,-0.61\n1978-12-01,0.07,0.23,-0.48,-0.43\n1979-01-01,0.05,0.23,-0.08,-0.49\n1979-02-01,0.09,-0.58,0.10,-0.51\n1979-03-01,0.13,-0.25,-0.01,-0.65\n1979-04-01,0.17,0.51,0.14,-0.30\n1979-05-01,-0.09,0.27,0.15,-0.27\n1979-06-01,0.04,0.76,0.18,-0.21\n1979-07-01,-0.20,0.45,0.01,-0.12\n1979-08-01,0.21,0.50,-0.25,0.07\n1979-09-01,0.48,1.02,-0.27,0.14\n1979-10-01,0.32,0.67,-0.32,-0.17\n1979-11-01,0.35,0.12,-0.20,-0.12\n1979-12-01,0.69,-0.34,-0.03,-0.28\n1980-01-01,0.55,-0.10,0.15,-0.48\n1980-02-01,0.33,-0.25,0.23,-0.61\n1980-03-01,0.09,0.02,0.02,-0.61\n1980-04-01,0.20,-0.00,0.40,-0.42\n1980-05-01,0.13,0.25,0.50,-0.38\n1980-06-01,0.37,0.21,0.23,-0.39\n1980-07-01,0.02,-0.44,-0.08,-0.31\n1980-08-01,-0.23,-0.44,-0.04,-0.16\n1980-09-01,-0.12,-0.11,-0.17,-0.04\n1980-10-01,-0.11,-0.52,-0.38,-0.25\n1980-11-01,0.10,-0.68,-0.28,-0.43\n1980-12-01,0.36,-1.13,-0.31,-0.86\n1981-01-01,-0.46,-1.29,0.03,-0.61\n1981-02-01,-0.45,-0.83,-0.01,-0.73\n1981-03-01,-0.02,-0.19,0.45,-0.99\n1981-04-01,-0.17,-0.31,0.18,-0.83\n1981-05-01,-0.11,-0.22,0.16,-0.75\n1981-06-01,-0.15,-0.06,-0.06,-0.53\n1981-07-01,-0.43,-0.56,-0.12,-0.48\n1981-08-01,-0.18,-0.55,-0.24,-0.07\n1981-09-01,-0.07,-0.24,-0.23,0.05\n1981-10-01,-0.03,-0.52,-0.39,0.12\n1981-11-01,-0.27,-0.55,-0.18,0.08\n1981-12-01,0.07,-0.17,-0.02,-0.16\n1982-01-01,-0.04,-0.12,-0.06,-0.30\n1982-02-01,-0.13,-0.35,-0.24,-0.47\n1982-03-01,-0.02,-0.72,-0.23,-0.56\n1982-04-01,0.24,-0.51,-0.32,-0.66\n1982-05-01,0.65,0.08,-0.24,-0.71\n1982-06-01,0.92,0.27,-0.40,-0.76\n1982-07-01,0.64,0.60,-0.54,-0.70\n1982-08-01,0.93,1.09,-0.64,-0.77\n1982-09-01,1.39,2.01,-0.66,-0.46\n1982-10-01,2.00,2.28,-0.77,-0.33\n1982-11-01,2.07,3.08,-0.77,-0.46\n1982-12-01,2.33,3.20,-0.79,-0.43\n1983-01-01,2.43,2.73,-0.61,-0.01\n1983-02-01,2.22,2.03,-0.19,0.11\n1983-03-01,1.69,2.34,0.28,-0.48\n1983-04-01,1.12,3.36,0.29,-0.56\n1983-05-01,1.12,4.10,-0.03,-0.80\n1983-06-01,0.62,4.56,0.10,-0.76\n1983-07-01,-0.11,3.93,-0.05,-0.56\n1983-08-01,-0.13,3.11,-0.33,-0.28\n1983-09-01,-0.50,2.07,-0.56,-0.11\n1983-10-01,-1.03,1.02,-0.58,-0.31\n1983-11-01,-1.13,0.56,-0.26,-0.12\n1983-12-01,-0.95,0.22,-0.06,0.12\n1984-01-01,-0.80,-0.29,-0.30,0.35\n1984-02-01,-0.53,-0.77,-0.38,0.38\n1984-03-01,-0.48,-0.41,-0.25,0.44\n1984-04-01,-0.60,-0.25,-0.34,0.19\n1984-05-01,-0.64,-1.11,-0.36,-0.08\n1984-06-01,-0.90,-0.92,-0.61,0.14\n1984-07-01,-0.40,-0.54,-0.63,0.29\n1984-08-01,-0.40,-0.52,-0.62,0.57\n1984-09-01,-0.35,-0.36,-0.62,0.69\n1984-10-01,-0.83,-0.61,-0.92,0.36\n1984-11-01,-1.18,-0.18,-0.77,0.13\n1984-12-01,-1.53,-0.50,-0.79,-0.01\n1985-01-01,-0.89,-0.77,-0.68,-0.02\n1985-02-01,-0.91,-1.13,-0.55,0.25\n1985-03-01,-0.96,-0.59,-0.77,-0.21\n1985-04-01,-0.98,-1.38,-0.81,-0.16\n1985-05-01,-0.90,-1.41,-0.91,-0.10\n1985-06-01,-0.76,-1.25,-0.54,-0.27\n1985-07-01,-0.61,-1.04,-0.32,0.02\n1985-08-01,-0.49,-1.02,-0.42,-0.10\n1985-09-01,-0.57,-0.83,-0.54,-0.13\n1985-10-01,-0.64,-0.82,-0.57,0.05\n1985-11-01,-0.43,-0.92,-0.39,0.06\n1985-12-01,-0.40,-0.75,-0.54,0.03\n1986-01-01,-0.79,-0.59,-0.86,-0.14\n1986-02-01,-0.64,-0.35,-0.80,0.20\n1986-03-01,-0.51,-0.84,-0.76,0.04\n1986-04-01,-0.41,-0.86,-0.80,-0.03\n1986-05-01,-0.47,-0.82,-0.74,-0.08\n1986-06-01,-0.08,-0.90,-0.84,0.10\n1986-07-01,0.16,-0.14,-0.65,-0.06\n1986-08-01,0.24,0.17,-0.74,0.01\n1986-09-01,0.59,-0.13,-0.34,0.16\n1986-10-01,0.90,0.14,-0.63,-0.08\n1986-11-01,1.04,0.28,-0.91,-0.35\n1986-12-01,0.98,0.53,-0.98,-0.24\n1987-01-01,1.19,0.77,-0.57,-0.32\n1987-02-01,1.17,1.02,-0.35,-0.19\n1987-03-01,1.24,1.15,0.06,-0.18\n1987-04-01,0.89,1.51,0.17,-0.27\n1987-05-01,0.91,1.35,0.17,-0.06\n1987-06-01,1.24,1.06,0.37,0.11\n1987-07-01,1.34,1.19,0.21,0.25\n1987-08-01,1.48,0.91,0.22,0.24\n1987-09-01,1.53,1.31,0.02,0.17\n1987-10-01,1.36,1.50,0.02,0.19\n1987-11-01,1.29,0.90,0.05,0.45\n1987-12-01,1.05,0.89,0.22,0.13\n1988-01-01,0.69,0.09,-0.09,0.30\n1988-02-01,0.35,-0.27,-0.20,0.39\n1988-03-01,0.29,-0.52,0.12,0.43\n1988-04-01,-0.49,-0.90,0.22,0.28\n1988-05-01,-1.05,-1.08,0.08,0.37\n1988-06-01,-1.46,-1.96,0.10,0.42\n1988-07-01,-1.54,-1.54,-0.00,0.20\n1988-08-01,-1.44,-1.59,-0.31,0.30\n1988-09-01,-1.33,-1.43,-0.32,0.16\n1988-10-01,-2.09,-1.26,-0.63,0.15\n1988-11-01,-2.18,-0.65,-0.54,-0.05\n1988-12-01,-1.98,-0.97,-0.52,-0.07\n1989-01-01,-1.95,-0.28,-0.67,0.08\n1989-02-01,-1.37,0.05,-0.66,-0.04\n1989-03-01,-1.33,-0.45,-0.82,0.00\n1989-04-01,-1.11,-0.49,-1.04,-0.02\n1989-05-01,-0.80,-1.16,-0.86,-0.01\n1989-06-01,-0.64,-0.73,-0.42,0.00\n1989-07-01,-0.47,-0.44,-0.03,0.00\n1989-08-01,-0.58,-0.46,0.05,0.19\n1989-09-01,-0.38,-0.47,-0.32,0.06\n1989-10-01,-0.40,-0.29,-0.30,0.06\n1989-11-01,-0.30,-0.38,-0.35,-0.24\n1989-12-01,-0.13,-0.43,-0.07,-0.51\n1990-01-01,0.02,-0.33,-0.33,-0.68\n1990-02-01,0.36,-0.27,-0.28,-0.14\n1990-03-01,0.20,-0.41,-0.20,-0.03\n1990-04-01,0.26,-0.33,-0.10,0.04\n1990-05-01,0.32,-0.23,-0.07,-0.40\n1990-06-01,0.00,-0.05,-0.27,-0.23\n1990-07-01,0.15,-0.87,-0.22,-0.37\n1990-08-01,0.17,-0.61,-0.04,-0.33\n1990-09-01,0.12,-0.28,0.02,-0.22\n1990-10-01,0.13,-0.73,-0.01,-0.13\n1990-11-01,0.08,-0.55,0.02,-0.01\n1990-12-01,0.32,-0.44,-0.12,-0.04\n1991-01-01,0.51,-0.09,-0.39,-0.21\n1991-02-01,0.32,-0.10,-0.29,-0.20\n1991-03-01,0.10,0.00,-0.37,-0.21\n1991-04-01,0.21,-0.68,-0.54,-0.00\n1991-05-01,0.44,-0.01,-0.62,0.14\n1991-06-01,0.65,0.24,-0.67,0.13\n1991-07-01,0.62,0.60,-0.67,-0.20\n1991-08-01,0.43,0.00,-0.62,-0.52\n1991-09-01,0.22,0.16,-0.52,-0.51\n1991-10-01,0.91,0.42,-0.53,-0.59\n1991-11-01,1.14,0.64,-0.46,-0.52\n1991-12-01,1.62,0.45,-0.47,-0.34\n1992-01-01,1.61,0.36,-0.24,-0.61\n1992-02-01,1.62,0.44,-0.14,-0.53\n1992-03-01,1.50,0.92,-0.34,-0.74\n1992-04-01,1.41,1.72,-0.44,-0.88\n1992-05-01,1.27,1.83,-0.39,-1.01\n1992-06-01,0.53,0.71,-0.36,-1.11\n1992-07-01,0.27,-0.05,-0.29,-0.86\n1992-08-01,-0.15,-0.24,-0.56,-0.86\n1992-09-01,-0.14,-0.50,-0.58,-0.70\n1992-10-01,-0.35,-0.26,-0.48,-0.74\n1992-11-01,-0.14,-0.25,-0.61,-0.97\n1992-12-01,0.04,-0.31,-0.52,-0.63\n1993-01-01,0.28,0.03,-0.50,-0.59\n1993-02-01,0.42,0.20,-0.43,-0.39\n1993-03-01,0.47,0.53,-0.38,-0.44\n1993-04-01,0.92,0.83,-0.22,-0.11\n1993-05-01,0.93,1.13,-0.25,-0.32\n1993-06-01,0.64,0.88,-0.18,-0.34\n1993-07-01,0.33,0.32,-0.44,-0.21\n1993-08-01,0.16,0.21,-0.45,-0.16\n1993-09-01,0.26,0.12,-0.50,0.30\n1993-10-01,0.36,0.32,-0.56,0.47\n1993-11-01,0.28,-0.23,-0.65,0.57\n1993-12-01,0.19,-0.29,-0.70,0.24\n1994-01-01,0.04,-0.07,-0.83,-0.01\n1994-02-01,-0.16,-0.33,-0.78,-0.14\n1994-03-01,-0.06,-1.13,-0.76,-0.10\n1994-04-01,0.03,-1.22,-0.62,-0.13\n1994-05-01,0.14,-0.91,-0.70,-0.23\n1994-06-01,0.27,-0.59,-0.67,-0.49\n1994-07-01,0.16,-0.82,-0.67,-0.52\n1994-08-01,0.52,-0.90,-0.63,-0.37\n1994-09-01,0.40,-0.05,-0.62,-0.30\n1994-10-01,0.90,0.65,-0.50,-0.27\n1994-11-01,1.14,0.59,-0.34,-0.11\n1994-12-01,1.21,0.68,-0.42,-0.07\n1995-01-01,1.10,0.91,-0.30,0.07\n1995-02-01,0.87,0.24,-0.22,0.34\n1995-03-01,0.49,-0.57,-0.21,0.39\n1995-04-01,0.27,-0.99,-0.04,0.29\n1995-05-01,0.03,-1.01,0.10,0.24\n1995-06-01,0.08,-0.53,0.23,0.33\n1995-07-01,0.03,-0.25,0.31,0.09\n1995-08-01,-0.38,-0.52,0.25,-0.26\n1995-09-01,-0.57,-0.28,0.03,-0.19\n1995-10-01,-0.72,-0.41,-0.06,-0.27\n1995-11-01,-0.78,-0.26,0.20,-0.06\n1995-12-01,-0.72,-0.69,0.17,-0.26\n1996-01-01,-0.65,-0.61,0.49,-0.15\n1996-02-01,-0.66,-0.32,0.22,0.16\n1996-03-01,-0.48,-0.07,0.07,0.12\n1996-04-01,-0.34,-1.28,0.15,0.34\n1996-05-01,-0.40,-0.81,0.16,0.43\n1996-06-01,-0.12,-1.07,-0.06,0.56\n1996-07-01,-0.14,-0.98,-0.15,0.41\n1996-08-01,-0.30,-0.46,-0.36,0.15\n1996-09-01,-0.34,-0.53,-0.22,-0.01\n1996-10-01,-0.28,-0.80,-0.34,-0.10\n1996-11-01,-0.30,-0.79,-0.28,-0.06\n1996-12-01,-0.43,-1.03,-0.15,-0.33\n1997-01-01,-0.43,-0.56,0.06,-0.55\n1997-02-01,-0.24,-0.03,0.16,-0.60\n1997-03-01,-0.06,0.82,-0.00,-0.82\n1997-04-01,0.34,0.98,0.12,-0.81\n1997-05-01,0.87,2.16,0.23,-0.89\n1997-06-01,1.15,3.17,0.29,-0.98\n1997-07-01,1.60,3.63,0.06,-0.53\n1997-08-01,1.94,3.98,-0.17,-0.29\n1997-09-01,2.10,4.05,-0.12,0.06\n1997-10-01,2.29,4.00,0.16,0.47\n1997-11-01,2.42,4.12,0.27,0.60\n1997-12-01,2.30,4.27,0.15,0.76\n1998-01-01,2.42,3.55,0.19,0.38\n1998-02-01,2.08,2.88,0.59,0.47\n1998-03-01,1.49,2.88,0.59,0.34\n1998-04-01,0.90,3.00,0.43,0.23\n1998-05-01,0.68,3.01,0.39,0.20\n1998-06-01,-0.39,2.00,0.42,0.35\n1998-07-01,-0.73,1.31,0.29,0.59\n1998-08-01,-0.83,0.94,0.33,0.40\n1998-09-01,-0.82,0.31,0.13,0.26\n1998-10-01,-1.19,0.12,0.07,-0.01\n1998-11-01,-1.23,-0.21,0.12,0.14\n1998-12-01,-1.51,-0.14,0.00,-0.15\n1999-01-01,-1.53,-0.23,-0.35,-0.20\n1999-02-01,-1.41,0.05,-0.38,-0.22\n1999-03-01,-0.92,0.30,-0.38,-0.02\n1999-04-01,-0.81,-0.78,-0.28,0.14\n1999-05-01,-0.87,-0.43,-0.09,0.18\n1999-06-01,-0.95,-0.55,-0.10,0.18\n1999-07-01,-0.84,-0.65,-0.10,0.38\n1999-08-01,-0.98,-0.51,0.01,0.35\n1999-09-01,-0.84,-0.97,0.01,0.09\n1999-10-01,-1.03,-0.76,-0.18,0.05\n1999-11-01,-1.41,-1.11,-0.18,0.15\n1999-12-01,-1.54,-1.06,-0.02,0.10\n2000-01-01,-1.79,-0.89,-0.42,-0.12\n2000-02-01,-1.53,-0.47,-0.28,0.12\n2000-03-01,-1.26,-0.56,-0.09,0.05\n2000-04-01,-0.80,0.46,-0.17,0.06\n2000-05-01,-0.80,-0.29,-0.27,-0.05\n2000-06-01,-0.75,-0.65,-0.40,-0.11\n2000-07-01,-0.57,-0.57,-0.27,-0.10\n2000-08-01,-0.36,-0.59,-0.19,-0.00\n2000-09-01,-0.39,-0.34,-0.36,-0.13\n2000-10-01,-0.55,-0.49,-0.42,-0.12\n2000-11-01,-0.75,-0.70,-0.43,-0.11\n2000-12-01,-0.92,-0.55,-0.49,-0.27\n2001-01-01,-0.88,-0.76,-0.38,-0.17\n2001-02-01,-0.63,-0.33,-0.32,-0.10\n2001-03-01,-0.48,0.35,-0.01,-0.06\n2001-04-01,-0.30,0.42,-0.11,0.07\n2001-05-01,-0.30,-0.64,-0.14,0.12\n2001-06-01,-0.11,-0.86,-0.01,0.04\n2001-07-01,0.01,-0.70,0.02,-0.11\n2001-08-01,-0.07,-0.77,0.00,-0.06\n2001-09-01,-0.28,-1.20,0.13,-0.04\n2001-10-01,-0.26,-1.14,0.06,-0.03\n2001-11-01,-0.28,-0.94,0.31,-0.02\n2001-12-01,-0.46,-1.12,0.37,-0.25\n2002-01-01,-0.14,-0.94,0.53,-0.19\n2002-02-01,0.00,-0.30,0.35,-0.33\n2002-03-01,0.11,0.62,0.21,-0.06\n2002-04-01,0.14,0.33,-0.18,-0.00\n2002-05-01,0.21,0.18,-0.40,0.15\n2002-06-01,0.68,-0.40,-0.38,0.02\n2002-07-01,0.57,-0.44,-0.33,0.22\n2002-08-01,0.70,-0.26,-0.38,-0.19\n2002-09-01,0.82,0.04,-0.31,-0.23\n2002-10-01,1.16,0.57,-0.07,-0.09\n2002-11-01,1.41,0.78,-0.09,-0.25\n2002-12-01,1.41,0.69,-0.09,0.08\n2003-01-01,0.98,0.10,0.02,0.22\n2003-02-01,0.64,-0.21,-0.17,0.30\n2003-03-01,0.48,-0.49,-0.11,0.28\n2003-04-01,-0.03,-0.90,-0.15,0.11\n2003-05-01,-0.52,-1.35,-0.14,0.02\n2003-06-01,-0.19,-0.91,-0.02,-0.21\n2003-07-01,0.14,-0.58,0.09,0.14\n2003-08-01,0.05,0.08,0.26,0.30\n2003-09-01,0.15,-0.43,0.28,0.34\n2003-10-01,0.46,-0.07,0.38,0.31\n2003-11-01,0.39,0.29,0.17,0.36\n2003-12-01,0.32,0.31,0.20,0.30\n2004-01-01,0.26,-0.03,0.25,0.07\n2004-02-01,0.17,-0.20,0.34,-0.05\n2004-03-01,-0.10,-0.34,0.20,-0.36\n2004-04-01,0.06,-0.45,0.30,-0.57\n2004-05-01,0.10,-0.96,-0.03,-0.47\n2004-06-01,0.14,-0.69,0.02,-0.39\n2004-07-01,0.41,-0.70,0.18,-0.18\n2004-08-01,0.66,-0.44,0.35,0.02\n2004-09-01,0.67,-0.10,0.21,0.18\n2004-10-01,0.73,0.11,0.20,-0.08\n2004-11-01,0.62,0.39,0.32,0.06\n2004-12-01,0.71,0.27,0.24,0.17\n2005-01-01,0.56,0.03,0.22,0.28\n2005-02-01,0.26,-0.61,0.20,0.22\n2005-03-01,0.28,-1.15,0.53,0.26\n2005-04-01,0.28,-0.36,0.65,-0.02\n2005-05-01,0.30,0.22,0.81,-0.40\n2005-06-01,0.22,-0.15,0.67,-0.57\n2005-07-01,-0.01,-0.06,0.57,-0.57\n2005-08-01,-0.04,-0.30,0.41,-0.19\n2005-09-01,-0.08,-0.61,0.35,-0.23\n2005-10-01,-0.15,-1.05,0.12,-0.22\n2005-11-01,-0.44,-1.41,0.09,-0.26\n2005-12-01,-0.75,-1.19,0.26,-0.29\n2006-01-01,-0.98,-0.69,0.16,-0.30\n2006-02-01,-0.71,0.33,0.01,-0.04\n2006-03-01,-0.73,-0.27,0.02,0.06\n2006-04-01,-0.30,-0.89,0.26,-0.00\n2006-05-01,-0.11,-0.39,0.23,-0.11\n2006-06-01,0.09,-0.19,0.28,0.10\n2006-07-01,0.03,0.18,0.09,0.05\n2006-08-01,0.37,0.51,0.20,0.13\n2006-09-01,0.62,0.78,0.31,0.13\n2006-10-01,0.76,1.20,0.31,0.01\n2006-11-01,0.98,0.91,0.40,-0.10\n2006-12-01,1.10,0.62,0.18,0.05\n2007-01-01,0.59,0.57,0.21,-0.09\n2007-02-01,0.12,0.06,0.38,-0.06\n2007-03-01,-0.15,-0.65,0.17,-0.31\n2007-04-01,-0.16,-1.05,0.24,-0.07\n2007-05-01,-0.39,-1.47,0.01,-0.06\n2007-06-01,-0.16,-1.40,-0.08,0.12\n2007-07-01,-0.37,-0.94,-0.18,-0.02\n2007-08-01,-0.57,-1.36,-0.33,-0.04\n2007-09-01,-1.04,-1.34,-0.27,0.12\n2007-10-01,-1.41,-1.65,-0.19,-0.11\n2007-11-01,-1.58,-1.95,-0.07,-0.08\n2007-12-01,-1.61,-1.63,-0.21,-0.06\n2008-01-01,-1.79,-0.80,-0.10,-0.14\n2008-02-01,-1.70,-0.16,-0.00,0.09\n2008-03-01,-1.17,0.33,0.12,0.25\n2008-04-01,-0.89,0.50,-0.06,0.33\n2008-05-01,-0.64,-0.32,0.07,0.47\n2008-06-01,-0.44,0.26,0.15,0.39\n2008-07-01,-0.04,0.47,0.17,0.40\n2008-08-01,-0.04,0.67,0.12,0.31\n2008-09-01,-0.28,0.32,0.20,0.13\n2008-10-01,-0.30,-0.17,0.01,0.05\n2008-11-01,-0.37,-0.21,-0.04,0.13\n2008-12-01,-0.90,-0.33,-0.03,0.15\n2009-01-01,-1.00,-0.17,-0.17,0.11\n2009-02-01,-0.71,-0.31,-0.55,0.08\n2009-03-01,-0.72,-0.85,-0.49,0.20\n2009-04-01,-0.25,0.00,-0.47,0.21\n2009-05-01,0.17,0.17,-0.43,0.44\n2009-06-01,0.49,0.54,-0.19,0.28\n2009-07-01,0.69,0.70,0.17,0.14\n2009-08-01,0.62,0.66,0.07,0.04\n2009-09-01,0.68,0.22,-0.03,0.04\n2009-10-01,0.96,0.22,0.20,0.23\n2009-11-01,1.49,0.14,0.20,0.11\n2009-12-01,1.81,0.49,0.25,0.43\n2010-01-01,1.43,0.36,0.55,0.46\n2010-02-01,1.18,0.04,0.90,0.41\n2010-03-01,1.07,-0.04,1.12,0.54\n2010-04-01,0.56,0.51,1.25,0.37\n2010-05-01,-0.15,0.39,1.15,0.27\n2010-06-01,-0.62,0.14,0.87,0.54\n2010-07-01,-0.89,-1.05,0.74,0.35\n2010-08-01,-1.33,-1.23,0.63,0.29\n2010-09-01,-1.56,-1.29,0.37,0.08\n2010-10-01,-1.65,-1.53,0.35,0.21\n2010-11-01,-1.57,-1.39,0.39,0.22\n2010-12-01,-1.63,-1.03,0.67,0.15\n2011-01-01,-1.70,-0.61,0.73,0.12\n2011-02-01,-1.26,0.07,0.35,0.32\n2011-03-01,-0.98,-0.43,0.25,0.38\n2011-04-01,-0.74,-0.05,0.17,0.18\n2011-05-01,-0.53,0.25,0.20,-0.04\n2011-06-01,-0.25,0.36,0.35,-0.11\n2011-07-01,-0.23,0.27,0.15,0.02\n2011-08-01,-0.66,-0.20,0.15,0.00\n2011-09-01,-0.76,-1.01,-0.01,0.09\n2011-10-01,-0.93,-0.71,0.02,0.06\n2011-11-01,-1.09,-0.80,-0.10,-0.23\n2011-12-01,-1.05,-0.68,-0.06,-0.74\n2012-01-01,-0.93,-0.39,-0.09,-0.66\n2012-02-01,-0.61,0.40,-0.19,-0.78\n2012-03-01,-0.48,-0.02,-0.41,-0.43\n2012-04-01,-0.29,0.48,-0.33,-0.64\n2012-05-01,-0.18,0.43,-0.11,-0.38\n2012-06-01,0.14,0.99,0.07,-0.27\n2012-07-01,0.44,0.82,0.02,-0.29\n2012-08-01,0.66,0.27,0.19,-0.28\n2012-09-01,0.44,0.04,0.34,-0.13\n2012-10-01,0.23,-0.39,0.35,-0.19\n2012-11-01,0.33,-0.46,0.44,-0.16\n2012-12-01,-0.13,-0.61,0.26,-0.09\n2013-01-01,-0.42,-0.70,0.19,0.05\n2013-02-01,-0.40,-0.32,0.22,-0.02\n2013-03-01,-0.14,-0.38,0.55,-0.09\n2013-04-01,-0.08,-1.32,0.37,0.04\n2013-05-01,-0.28,-1.44,0.44,0.13\n2013-06-01,-0.33,-1.44,0.07,-0.03\n2013-07-01,-0.28,-1.36,0.06,-0.03\n2013-08-01,-0.29,-0.76,0.22,0.05\n2013-09-01,-0.09,-0.77,0.23,-0.12\n2013-10-01,-0.24,-0.16,0.23,-0.18\n2013-11-01,-0.02,-0.50,0.02,-0.26\n2013-12-01,-0.09,-0.25,-0.02,-0.16\n2014-01-01,-0.42,0.37,-0.12,-0.23\n2014-02-01,-0.45,-0.67,-0.25,-0.05\n2014-03-01,-0.07,-0.51,-0.50,-0.13\n2014-04-01,0.28,-0.25,-0.36,-0.08\n2014-05-01,0.45,0.74,-0.27,0.14\n2014-06-01,0.48,1.17,-0.29,0.15\n2014-07-01,0.13,1.04,-0.13,-0.09\n2014-08-01,0.14,0.67,-0.10,0.05\n2014-09-01,0.37,0.15,0.13,0.11\n2014-10-01,0.48,0.34,0.13,-0.06\n2014-11-01,0.89,0.57,0.04,-0.50\n2014-12-01,0.77,0.14,0.05,-0.38\n2015-01-01,0.59,-0.16,-0.08,0.10\n2015-02-01,0.57,-0.62,-0.06,0.15\n2015-03-01,0.48,-0.01,-0.27,-0.11\n2015-04-01,0.90,0.64,-0.40,0.05\n2015-05-01,1.04,1.59,-0.13,0.11\n2015-06-01,1.28,2.19,-0.31,0.07\n2015-07-01,1.56,2.32,-0.12,-0.19\n2015-08-01,1.87,2.08,0.09,-0.31\n2015-09-01,2.01,2.48,0.35,-0.19\n2015-10-01,2.21,2.38,0.51,0.06\n2015-11-01,2.57,2.21,0.18,0.19\n2015-12-01,2.56,2.09,0.13,0.44\n2016-01-01,2.56,1.82,0.25,0.47\n2016-02-01,2.11,1.09,0.28,0.32\n2016-03-01,1.60,1.10,0.24,0.25\n2016-04-01,1.05,0.29,0.17,0.08\n2016-05-01,0.45,0.45,0.18,0.01\n2016-06-01,0.06,0.52,0.15,0.37\n2016-07-01,-0.25,-0.06,0.10,0.37\n2016-08-01,-0.48,0.19,0.15,0.33\n2016-09-01,-0.46,0.25,0.01,0.11\n2016-10-01,-0.75,-0.03,0.12,0.24\n2016-11-01,-0.63,-0.08,0.15,0.10\n2016-12-01,-0.51,0.02,0.23,0.41\n2017-01-01,-0.34,0.33,0.24,0.25\n2017-02-01,-0.01,0.93,0.06,-0.18\n2017-03-01,-0.09,1.56,-0.03,-0.06\n2017-04-01,0.22,0.49,0.31,0.09\n2017-05-01,0.30,0.25,0.38,0.12\n2017-06-01,0.22,0.20,0.35,0.26\n2017-07-01,0.22,-0.46,0.32,0.10\n2017-08-01,-0.18,-0.76,0.33,0.39\n2017-09-01,-0.56,-1.28,0.15,0.12\n2017-10-01,-0.52,-1.32,0.03,0.19\n2017-11-01,-0.84,-1.44,0.22,0.02\n2017-12-01,-0.85,-1.54,0.25,-0.09\n2018-01-01,-0.86,-1.25,0.03,0.29\n2018-02-01,-0.73,-0.43,-0.27,-0.27\n2018-03-01,-0.73,-1.30,-0.13,-0.18\n2018-04-01,-0.36,-0.81,-0.33,-0.03\n2018-05-01,-0.12,-0.46,-0.60,-0.07\n2018-06-01,0.12,-0.47,-0.72,0.15\n2018-07-01,0.27,-0.08,-0.48,0.12\n2018-08-01,0.05,-0.54,-0.32,0.21\n2018-09-01,0.30,0.12,-0.04,0.21\n2018-10-01,0.84,0.30,-0.13,0.31\n2018-11-01,1.00,0.69,-0.32,0.21\n2018-12-01,0.97,0.79,-0.33,0.40\n2019-01-01,0.51,0.67,-0.17,0.41\n2019-02-01,0.62,0.66,-0.04,0.41\n2019-03-01,0.81,-0.18,-0.15,0.67\n2019-04-01,0.67,0.36,-0.17,0.34\n2019-05-01,0.61,0.24,-0.08,0.31\n2019-06-01,0.66,-0.11,0.06,0.27\n2019-07-01,0.41,-0.36,0.09,0.24\n2019-08-01,0.19,-0.29,0.09,0.22\n2019-09-01,0.11,-0.68,0.20,0.33\n2019-10-01,0.55,-0.71,0.01,0.25\n2019-11-01,0.74,0.02,-0.05,0.87\n2019-12-01,0.51,-0.19,-0.04,0.80\n2020-01-01,0.64,-0.12,0.01,0.95\n2020-02-01,0.48,-0.03,0.32,0.57\n2020-03-01,0.36,-0.44,0.58,0.31\n2020-04-01,0.49,0.03,0.46,0.41\n2020-05-01,-0.12,-0.24,0.32,0.53\n2020-06-01,-0.21,-0.43,0.47,0.29\n2020-07-01,-0.04,-1.09,0.38,0.29\n2020-08-01,-0.42,-0.43,0.25,0.30\n2020-09-01,-0.66,-0.85,0.15,0.09\n2020-10-01,-1.19,-0.99,0.19,0.17\n2020-11-01,-1.01,-0.87,0.08,-0.06\n2020-12-01,-0.98,-0.76,0.17,-0.38\n2021-01-01,-1.04,-0.58,0.27,-0.12\n2021-02-01,-0.94,-0.74,0.23,-0.33\n2021-03-01,-0.72,-0.50,-0.01,-0.21\n2021-04-01,-0.55,-0.96,0.16,0.06\n2021-05-01,-0.41,-0.73,0.04,0.41\n2021-06-01,-0.06,0.08,-0.10,0.58\n2021-07-01,-0.20,-0.14,-0.02,0.86\n2021-08-01,-0.38,-0.17,-0.07,0.72\n2021-09-01,-0.50,-0.29,0.16,0.43\n2021-10-01,-0.78,-0.60,0.03,0.42\n2021-11-01,-0.88,-0.98,0.11,0.32\n2021-12-01,-1.07,-1.29,0.16,0.00\n2022-01-01,-0.77,-0.90,0.36,0.14\n2022-02-01,-0.68,-1.14,0.39,0.46\n2022-03-01,-0.84,-0.59,-0.07,0.22\n2022-04-01,-0.90,-1.04,-0.11,0.15\n2022-05-01,-1.02,-1.04,0.14,0.01\n2022-06-01,-0.77,-0.82,0.23,0.30\n2022-07-01,-0.56,-0.54,0.05,0.08\n2022-08-01,-0.96,-0.23,0.14,0.18\n2022-09-01,-1.06,-0.77,0.25,0.20\n2022-10-01,-0.99,-1.37,0.08,0.30\n2022-11-01,-0.76,-0.92,0.02,0.33\n2022-12-01,-0.86,-0.67,0.18,0.34\n2023-01-01,-0.78,-0.35,0.23,0.49\n2023-02-01,-0.62,-0.14,0.04,0.28\n2023-03-01,-0.13,1.00,0.49,0.14\n2023-04-01,0.24,2.50,0.62,0.19\n2023-05-01,0.47,1.69,0.83,0.44\n2023-06-01,0.95,2.39,1.17,0.26\n2023-07-01,1.20,2.45,1.14,0.28\n2023-08-01,1.56,2.55,1.02,0.30\n2023-09-01,1.65,2.04,1.06,0.25\n2023-10-01,1.59,1.80,0.89,0.19\n2023-11-01,2.01,1.94,0.78,0.73\n2023-12-01,1.81,1.37,0.88,1.05\n2024-01-01,1.71,1.14,1.17,0.99\n2024-02-01,1.47,1.08,1.41,1.14\n2024-03-01,1.10,-0.06,1.21,1.33\n2024-04-01,0.93,-0.11,1.17,1.10\n2024-05-01,0.41,-0.41,1.22,0.82\n2024-06-01,0.25,-0.48,1.09,0.54\n2024-07-01,0.20,-0.44,0.87,0.33\n2024-08-01,0.02,-0.62,0.79,0.19\n2024-09-01,-0.11,-0.58,0.66,0.30\n2024-10-01,-0.24,-0.47,0.61,0.28\n2024-11-01,-0.05,-0.14,0.67,0.22\n', 'research/common/__init__.py': '"""Audited input loading and fixed-reference model helpers."""\n', 'research/common/baseline.py': '"""Fit and predict the fixed lagged hybrid, without changing its architecture."""\nfrom pathlib import Path\nimport gc\nimport numpy as np\nimport xarray as xr\nfrom .inputs import require, sha256, write_json\n\n\ndef fit(data, cutoff, folder):\n    folder = Path(folder)\n    folder.mkdir(parents=True, exist_ok=True)\n    m = data.lib\n    train = data.rain.sel(time=slice(None, cutoff))\n    X, y, ids, dates, climate, atmosphere, sums, counts, seas, cfs = m.amostrar_arvores_multissistema(\n        train, data.atmosphere, data.seas, data.cfs, cutoff)\n    models = []\n    for name, features in [(\'arvores_seas5\', m.FEATURES_SEAS5), (\'arvores_multissistema\', m.FEATURES_MULTISSISTEMA)]:\n        dataset = m.lgb.Dataset(np.ascontiguousarray(X[:, :len(features)]), label=y, feature_name=features)\n        model = m.lgb.train(m.PARAMETROS, dataset, num_boost_round=m.ARVORES)\n        require(model.feature_name() == features, \'Baseline feature order changed.\')\n        model.save_model(str(folder / (name + \'.txt\')))\n        models.append(model)\n        del dataset\n    ridge = m.mos_fit(train, data.seas, data.cfs, cutoff)\n    np.savez_compressed(folder / \'ridge.npz\', **ridge)\n    np.savez_compressed(folder / \'clima_atmosfera_indices.npz\', **atmosphere)\n    for name, field in [(\'chuva\', climate), (\'seas5\', seas), (\'cfsv2\', cfs)]:\n        field.to_netcdf(folder / (\'clima_\' + name + \'.nc\'))\n    audit = dict(training_start=str(dates[0].date()), training_end=str(dates[-1].date()),\n        training_months=len(dates), examples=len(y), parameters=m.PARAMETROS,\n        tree_count=m.ARVORES, weights=[.375, .375, .25], residual_scales=[.9, .875],\n        sample_hash=__import__(\'hashlib\').sha256(ids.tobytes()).hexdigest(), inputs=data.audit)\n    del X, y, ids\n    gc.collect()\n    write_json(folder / \'fit.json\', audit)\n    return dict(models=models, ridge=ridge, climate=climate, atmosphere=atmosphere, seas=seas, cfs=cfs, audit=audit)\n\n\ndef predict(data, state, dates):\n    m = data.lib\n    components = []\n    for model, name, scale in zip(state[\'models\'], m.NOMES_MODELOS, [.9, .875]):\n        anomalies = m.prever_par(model, name, data.atmosphere, data.seas, data.cfs,\n            state[\'atmosphere\'], state[\'climate\'], state[\'seas\'], state[\'cfs\'], dates)\n        components.append(m.reconstruir_chuva(state[\'climate\'], anomalies, scale).values.astype(\'float64\'))\n    ridge = m.mos_predict(state[\'ridge\'], data.seas, data.cfs, dates)\n    result = .75 * (.5 * components[0] + .5 * components[1]) + .25 * ridge\n    return xr.DataArray(result, dims=(\'time\', \'lat\', \'lon\'),\n        coords=dict(time=dates, lat=data.rain.lat, lon=data.rain.lon), attrs=dict(units=\'mm/day\'))\n', 'research/common/inputs.py': '"""Load the existing audited inputs; no downloads, imputation or new sources."""\nfrom pathlib import Path\nimport hashlib\nimport importlib.metadata\nimport importlib.util\nimport json\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\n\nROOT = Path(__file__).resolve().parents[2]\nLIBRARY = ROOT / \'research/lagged_sources/library.py\'\nREQUIRED = {\'numpy\': \'2.0.2\', \'pandas\': \'2.3.3\', \'xarray\': \'2025.12.0\', \'lightgbm\': \'4.6.0\'}\n\n\ndef require(ok, message):\n    if not bool(ok):\n        raise ValueError(message)\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for b in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef write_json(path, obj):\n    Path(path).write_text(json.dumps(obj, indent=2, sort_keys=True, allow_nan=False, default=str) + \'\\n\', encoding=\'utf-8\')\n\n\ndef library(path=LIBRARY):\n    # Separate module globals: callers cannot change another run\'s index table.\n    spec = importlib.util.spec_from_file_location(\'fixed_lag_library\', path)\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    return module\n\n\ndef find_unique(filename, supplied=None):\n    if supplied:\n        p = Path(supplied)\n        require((p / filename).is_file(), f\'Missing {filename} in {p}\')\n        return p.resolve()\n    candidates = sorted({p.parent.resolve() for p in Path(\'/kaggle/input\').rglob(filename)})\n    require(len(candidates) == 1, f\'Attach exactly one input containing {filename}, or set its directory explicitly. Found {len(candidates)}.\')\n    return candidates[0]\n\n\ndef preflight(official=None, seas5=None, cfsv2=None, final=False):\n    paths = dict(official=find_unique(\'treino_tp.nc\', official),\n                 seas5=find_unique(\'seas5_51_manifesto.json\', seas5),\n                 cfsv2=find_unique(\'cfsv2_manifesto.json\', cfsv2))\n    expected = json.loads((ROOT / \'research/lagged_sources/official_hashes.json\').read_text())\n    files = {}\n    for row in expected:\n        if not row[\'nome\'].startswith(\'treino_\'):\n            continue\n        p = paths[\'official\'] / row[\'nome\']\n        require(p.is_file(), f\'Missing official input: {p.name}\')\n        require(sha256(p) == row[\'sha256\'], f\'Official input hash mismatch: {p.name}\')\n        files[p.name] = row[\'sha256\']\n    for source, name in [(\'seas5\', \'seas5_51_manifesto.json\'), (\'cfsv2\', \'cfsv2_manifesto.json\')]:\n        manifest = paths[source] / name\n        files[name] = sha256(manifest)\n        m = json.loads(manifest.read_text(encoding=\'utf-8\'))\n        for phase in [\'desenvolvimento\'] + ([\'somente_ajuste_final\'] if final else []):\n            row = m[\'arquivos\'][phase]\n            p = paths[source] / row[\'arquivo\']\n            require(p.is_file() and sha256(p) == row[\'sha256\'], f\'Missing or altered {source}: {phase}\')\n            files[p.name] = row[\'sha256\']\n    versions = {n: importlib.metadata.version(n) for n in REQUIRED}\n    require(versions == REQUIRED, f\'The fixed baseline needs {REQUIRED}; found {versions}.\')\n    return paths, dict(files=files, versions=versions, final_training=final)\n\n\nclass Inputs:\n    def __init__(self, official=None, seas5=None, cfsv2=None, output=\'outputs\', final=False):\n        self.paths, self.audit = preflight(official, seas5, cfsv2, final)\n        self.lib = m = library()\n        m.PASTA = self.paths[\'official\']\n        m.PASTA_SEAS5_MANUAL = self.paths[\'seas5\']\n        m.PASTA_CFSV2_MANUAL = self.paths[\'cfsv2\']\n        m.PASTA_SEAS5, m.MANIFESTO_SEAS5 = m.localizar_seas5()\n        m.PASTA_CFSV2, m.MANIFESTO_CFSV2 = m.localizar_cfsv2()\n        m.SAIDA = Path(output)\n        m.SAIDA.mkdir(parents=True, exist_ok=True)\n        end = \'2022-12-01\' if final else \'2020-12-01\'\n        start = \'1993-01-01\' if final else \'1976-10-01\'\n        with xr.open_dataset(m.PASTA / \'treino_tp.nc\') as ds:\n            require(ds.tp.attrs.get(\'units\') == \'mm/day\', \'Unexpected rainfall units.\')\n            self.rain = ds.tp.sel(time=slice(start, end)).transpose(\'time\', \'lat\', \'lon\').load()\n        require(pd.DatetimeIndex(self.rain.time.values).equals(pd.date_range(start, end, freq=\'MS\')), \'Incomplete rainfall calendar.\')\n        require(np.array_equal(self.rain.lat, np.arange(-60, 15.25, .25)) and\n                np.array_equal(self.rain.lon, np.arange(-90, -24.75, .25)), \'Different target grid.\')\n        require(np.isfinite(self.rain.values).all() and (self.rain.values >= 0).all(), \'Invalid rainfall.\')\n        atm_start = pd.Timestamp(start) - pd.DateOffset(months=4)\n        atm_end = pd.Timestamp(end) - pd.DateOffset(months=4)\n        self.atmosphere, _ = m.carregar_atmosfera(self.rain, atm_start, atm_end)\n        index_path = ROOT / (\'competition/metadata/NOAA/indices_noaa.csv\' if final else \'research/lagged_sources/ocean_indices.csv\')\n        m.INDICES_OC = pd.read_csv(index_path, parse_dates=[\'time_origem\']).set_index(\'time_origem\')\n        m.conferir_indices(m.INDICES_OC)\n        self.audit[\'indices_sha256\'] = sha256(index_path)\n        self.audit[\'library_sha256\'] = sha256(LIBRARY)\n        phases = [\'desenvolvimento\'] + ([\'somente_ajuste_final\'] if final else [])\n        self.seas = xr.concat([m.carregar_seas5(p, self.rain) for p in phases], dim=\'time\').sel(time=slice(start, end))\n        self.cfs = xr.concat([m.carregar_cfsv2(p, self.rain) for p in phases], dim=\'time\').sel(time=slice(start, end))\n        self.audit.update(rainfall_read=[start, end], forecast_partitions=phases,\n                          test_partition_read=False, sst_used=False)\n\n\nclass Context:\n    """Training-only climatologies, with a bounded reference window for inner selection."""\n    def __init__(self, data, reference, training):\n        self.data = data\n        m = data.lib\n        self.reference = pd.DatetimeIndex(reference)\n        self.training = pd.DatetimeIndex(training)\n        require(24 <= len(self.training) <= 360 and len(self.reference) <= 360, \'Invalid training window.\')\n        require(self.training.isin(self.reference).all() and self.reference.is_monotonic_increasing and\n                self.reference.is_unique and self.training.is_unique, \'Invalid training/reference dates.\')\n        self.cutoff = self.reference[-1]\n        rain = data.rain.sel(time=self.reference).values\n        self.counts = np.array([(self.reference.month == month).sum() for month in range(1, 13)])\n        require((self.counts > 1).all(), \'At least two reference years per month are needed.\')\n        self.sums = np.stack([rain[self.reference.month == month].sum(0, dtype=np.float64) for month in range(1, 13)])\n        self.climate = xr.DataArray((self.sums / self.counts[:, None, None]).astype(\'float32\'),\n            dims=(\'month\', \'lat\', \'lon\'), coords=dict(month=np.arange(1, 13), lat=data.rain.lat, lon=data.rain.lon), attrs=dict(units=\'mm/day\'))\n        origins = m.origem_mensal(self.reference, 4)\n        self.atm = m.ajustar_clima_atmosfera(data.atmosphere, origins)\n        origins = m.origem_mensal(self.reference, 3)\n        vals = m.INDICES_OC.loc[origins, m.INDICES_NOMES].to_numpy(dtype=np.float64)\n        self.atm[\'_clima_indices\'] = np.stack([vals[origins.month == month].mean(0) for month in range(1, 13)])\n        self.seasonal = []\n        for forecasts in [data.seas, data.cfs]:\n            v = forecasts.sel(time=self.training)\n            require(pd.DatetimeIndex(v.time_origem.values).equals(m.origem_mensal(self.training, 1)), \'Wrong seasonal initialization.\')\n            means = np.stack([v.values[self.training.month == month].mean(0, dtype=np.float64) for month in range(1, 13)]).astype(\'float32\')\n            require(np.isfinite(means).all(), \'Missing seasonal reference month.\')\n            self.seasonal.append(xr.DataArray(means, dims=self.climate.dims, coords=self.climate.coords))\n\n    def features(self, target, training=False):\n        d, m = self.data, self.data.lib\n        t = pd.Timestamp(target)\n        require(t in self.training if training else t > self.cutoff, \'Month crosses the training boundary.\')\n        x = np.column_stack([m.matriz_mes(d.atmosphere, self.atm, self.climate, t),\n            m.valores_seas5(d.seas, t), m.valores_anomalia_seas5(d.seas, self.seasonal[0], t),\n            m.valores_cfsv2(d.cfs, t), m.valores_anomalia_cfsv2(d.cfs, self.seasonal[1], t)]).astype(\'float32\')\n        if training:\n            # Exclude the entire target map, including neighbors seen by convolutions.\n            y = d.rain.sel(time=t).values\n            month = t.month - 1\n            x[:, 22] = ((self.sums[month] - y.astype(\'float64\')) / (self.counts[month] - 1)).ravel().astype(\'float32\')\n        require(np.isfinite(x).all(), \'Nonfinite features.\')\n        return x.T.reshape(31, d.rain.sizes[\'lat\'], d.rain.sizes[\'lon\'])\n\n    def save(self, folder):\n        folder = Path(folder)\n        folder.mkdir(parents=True, exist_ok=True)\n        self.climate.to_netcdf(folder / \'rainfall_climatology.nc\')\n        np.savez_compressed(folder / \'references.npz\', sums=self.sums, counts=self.counts,\n                            seas5=self.seasonal[0].values, cfsv2=self.seasonal[1].values, **self.atm)\n        write_json(folder / \'context.json\', dict(reference=self.reference.strftime(\'%Y-%m\').tolist(),\n                                                training=self.training.strftime(\'%Y-%m\').tolist(), features=self.data.lib.FEATURES_MULTISSISTEMA))\n', 'research/common/synthetic_fixture.py': '"""Small synthetic fields for tests; never operational inputs."""\nfrom types import SimpleNamespace\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\nfrom .inputs import library\n\n\ndef fixture():\n    m = library()\n    dates = pd.date_range(\'1976-06-01\', \'2009-01-01\', freq=\'MS\')\n    rng = np.random.default_rng(71)\n    coords = dict(time=dates, lat=np.linspace(-60, 15, 9), lon=np.linspace(-90, -25, 13))\n    def field():\n        return xr.DataArray(rng.uniform(1, 5, (len(dates), 9, 13)).astype(\'float32\'), dims=(\'time\', \'lat\', \'lon\'), coords=coords)\n    rain, seas, cfs = field(), field(), field()\n    origins = m.origem_mensal(dates, 1)\n    seas = seas.assign_coords(time_origem=(\'time\', origins.values))\n    cfs = cfs.assign_coords(time_origem=(\'time\', origins.values), inicializacao_mais_recente=(\'time\', origins.values))\n    atmosphere = {name: field() for name in m.VARIAVEIS}\n    m.INDICES_OC = pd.DataFrame(rng.normal(size=(len(dates), 4)), index=dates, columns=m.INDICES_NOMES)\n    return SimpleNamespace(lib=m, rain=rain, seas=seas, cfs=cfs, atmosphere=atmosphere)\n', 'research/lagged_sources/library.py': 'from pathlib import Path\nfrom datetime import datetime, timezone\nfrom time import perf_counter\nimport base64, gzip, io, json, hashlib, gc, platform, importlib.metadata\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\nimport lightgbm as lgb\nfrom sklearn.decomposition import PCA\nfrom threadpoolctl import threadpool_limits\nfrom IPython.display import display, Markdown, Image, FileLink\nDATASET, SYSTEM, SCHEMA = (\'seasonal-monthly-single-levels\', \'51\', \'worcap_seas5_v1\')\nPASTA_SEAS5_MANUAL = PASTA_CFSV2_MANUAL = None\nPONTOS_POR_MES = 5000\nANOS_TREINO = 30\nARVORES = 300\nSEMENTE = 42\nCORTE_FINAL = \'2022-12-01\'\nFIM_DESENVOLVIMENTO = \'2020-12-01\'\nPARAMETROS = {\'objective\': \'regression\', \'metric\': \'rmse\', \'learning_rate\': 0.05, \'num_leaves\': 31, \'min_data_in_leaf\': 150, \'lambda_l2\': 10.0, \'feature_fraction\': 0.9, \'bagging_fraction\': 0.8, \'bagging_freq\': 1, \'num_threads\': 4, \'seed\': SEMENTE, \'deterministic\': True, \'force_col_wise\': True, \'verbosity\': -1}\nBLOCOS = [{\'nome\': \'H1\', \'corte\': \'2006-12-01\', \'inicio\': \'2007-01-01\'}, {\'nome\': \'H2\', \'corte\': \'2008-12-01\', \'inicio\': \'2009-01-01\'}, {\'nome\': \'H3\', \'corte\': \'2010-12-01\', \'inicio\': \'2011-01-01\'}, {\'nome\': \'H4\', \'corte\': \'2012-12-01\', \'inicio\': \'2013-01-01\'}, {\'nome\': \'A\', \'corte\': \'2014-12-01\', \'inicio\': \'2015-01-01\'}, {\'nome\': \'B\', \'corte\': \'2016-12-01\', \'inicio\': \'2017-01-01\'}, {\'nome\': \'C\', \'corte\': \'2018-12-01\', \'inicio\': \'2019-01-01\'}]\nVARIAVEIS = [\'cloud_cover\', \'geopotential_850\', \'rel_hum_850\', \'shum_850\', \'surface_pressure\', \'t2\', \'temperature_850\', \'u_850\', \'v_850\']\nINDICES_NOMES = [\'nino34\', \'nino12\', \'tna\', \'tsa\']\nFONTES_INDICES = {\'fonte\': \'NOAA Physical Sciences Laboratory\', \'captura_utc\': \'2026-09-21T23:00:40.933121+00:00\', \'unidade\': \'degC\', \'tipo\': \'monthly sea surface temperature anomaly indices\', \'interpretacao_temporal\': \'Retrospective series with observation month strictly T-1. These are not archived real-time vintages.\', \'inicio\': \'1976-12-01\', \'fim\': \'2024-11-01\', \'meses\': 576, \'colunas\': [\'time_origem\', \'nino34\', \'nino12\', \'tna\', \'tsa\'], \'fontes\': [{\'indice\': \'nino34\', \'url\': \'https://psl.noaa.gov/data/timeseries/month/data/nino34.long.anom.data\', \'arquivo_bruto\': \'nino34.data\', \'sha256_bruto\': \'46e501b91f624ef85b525d73fbb427f9d04d30411ba6e5c7cff579b35fb4ff01\', \'inicio_declarado\': 1870, \'fim_declarado\': 2026, \'rodape\': \'-99.99\\n  NINA34\\n 5N-5S 170W-120W \\n HadISST \\n  Anomaly from 1981-2010\\n https://psl.noaa.gov/data/timeseries/month/\\n  units=degC\'}, {\'indice\': \'nino12\', \'url\': \'https://psl.noaa.gov/data/timeseries/month/data/nino12.long.anom.data\', \'arquivo_bruto\': \'nino12.data\', \'sha256_bruto\': \'b13ad5ad085cd13e70ec5d82662b509d522204e38a8eefd133f14be270e51928\', \'inicio_declarado\': 1870, \'fim_declarado\': 2026, \'rodape\': \'-99.99\\n  NINA12\\n 0N-10S 270W-280E \\n HadISST \\n  Anomaly from 1981-2010\\n https://psl.noaa.gov/data/timeseries/month/\\n  units=degC\'}, {\'indice\': \'tna\', \'url\': \'https://psl.noaa.gov/data/correlation/tna.data\', \'arquivo_bruto\': \'tna.data\', \'sha256_bruto\': \'387c4125f377e57bcb683a849aefcd76da7d7f3cdcb2ac68ab086ceadf300e8b\', \'inicio_declarado\': 1948, \'fim_declarado\': 2026, \'rodape\': \'-99.99\\n  TNAa\\n Tropical Northern Atlantic Index\\n SST Anom 5.5N-23.5N; 15W-57.5W\\n  Using HadISST1.1: Climo 1991-2020\\n  NOAA/PSL\\n  https://psl.noaa.gov/data/timeseries/month/\'}, {\'indice\': \'tsa\', \'url\': \'https://psl.noaa.gov/data/correlation/tsa.data\', \'arquivo_bruto\': \'tsa.data\', \'sha256_bruto\': \'700c0dffda0f19c1254ba482808b345eaa0538f5bf09fc3bc025f46a5fbe3a69\', \'inicio_declarado\': 1948, \'fim_declarado\': 2026, \'rodape\': \'-99.99\\n  TSAa\\n  NOAA/PSL\\n  https://psl.noaa.gov/data/timeseries/month/\'}], \'sha256_csv\': \'b6f444fbec48681c1e5b46a5c22aa197ad91c7fbbdaff82c3ed3a9c63f7c15a4\', \'transformacao_no_modelo\': \'Monthly recentering within each fold, using only the 360 training origins; no standardization or moving averages.\'}\nINDICES_CSV_GZIP_BASE64 = \'H4sIAAAAAAACCm1cy44tuY3cz7ecGuj9+JqBF4bRC7eBmf5/zElGBMkse1N90YpSSqL4CJKqv/7459//51//+8c//v7Pz59//PmvPuw/tX3++vNvn7/+72//Ve9eP7X9lPop/73G90ebn5/vP8vzc+wHsL+jABwDXBsywOwEcIZpv1xHIJqm6EDg//aDMfv3IGI8iB9fwYl/O2QSUrp9wQabAfslZOE7fX1/FMyBvVQCNhcyPz5Pw3oJOADg09PGlv2s+sblFLbPadMvW1dtQNTCA7v4PUMsWyeXUe1IK1eAM507nxik8kVgzH4bX8RxHBfLttUV++3asWoiJBdbXbctN0xHAMVSbPq9n3/a904hQFLpmz+/37AdX00hoeAg5ohl3EvIIgQSOzgJ+84+hGxBTlwe3MLeCTmEQBprxKkOrfYKgt/EinCAhEA0dor2m50nYzIkpBJS7cBGTOUInmt5jqP1tG5b63XRlOkAXMZxCdAMWNzxY52VAEoGJ4nLyL1NIgYRGxKW4vXCcdeW+zFlsw35DbuuK+VR6/0oTIXCaAmSyVeDpDDpmt9QFbtjhQt9joXjUpTz3ONGnX/WiXGpSW+P9cEFg6ZxgspjtJtbW2izVuCm64ZIqVLPxTrFJUHTVPwUhgCcondfPwRmwn4A3UVl6iMd8nHKASsrxa3caARMFyU/MGkFOS6btXWWWIDZtGecJgufhQ3AbepaglSDdxEQaOIiRKqBY6z49wYQEFcNDML00VRPQniauKu00XHzH0RzA/xTeXt7hdo/iEqJPL9ikHbdEONEK0US9qLH5d+dkK7LTdP8EdqMzoOQ5cIme/X7fTTHfO23PaKtMISTiBWW4EdWEj+nJnHD1cNO44xxw2pIB5/HTWyQ0SZE0oEVx2DrZkGAcOGUnoRzP9SFB6FzhV/lhT/mB4kIsxXCb3FLWsiGl6fFhs2sPBAJpybTiQVbvPBA3sKBf2rdHc8DkdLAh9bQ/SUEtWbN8OQG3pUAqs1tH+0Zerc1w05xzUreaWsrtF+3u5+FV9nayIUP/p5zk/WFw+EnIJX2aH2jIuztTusBVAL2p3MXAPgMDYCv+ek0IfuGQnWKpD33q32vfwQbpRLAGb73t/F2mDpUjXdsYj2b6BbCnbienaKoj7A74iaGI4vj08cHFR9fsRjhAVAQ6wHM5eZha4Idetafw/4xx5g/ceJOdapjT2a8h4LMgtMMXeM2qSAWUsnbzGQluxSkwhDjt9tyd/IgdLevuQHdfgFG6McpEbf1Yi6KCE0xe7oNjxbC3I/QDoYM1f3OGESMVxijIPjDEPRByHQ997XqxGCe9BnZrotwjT5L7vdBuOlKurHMSekz541QMPzEqkRILrADWIIhFudww3XS3TWd7YsIl8sJK2mxaO1ENCJwqDOpCbRgJsncOHc4qNIIcenW8E3TJD2JkGjuisgRUzV9R7K555kE7vHUMKIzhPOcfKWALyCFEAlnwwfOEIBFZw9E0lm41YMmMu3HpXMN0YLM+HdcbXaIQCE6IC4fmL7TCJfrmeFY4OJuuJ7CHbve0ABrO4+leBArxAOp4WRBBHAdV4gHC8GFOsUiKyIkHviLM3iCCmFWSGfU+MChzSJE0hk79rtHqM4K6cBgQ4GM/FathK6lriC8izECEccdnFzttgOpHGdsjJOgLzUsJ1BwfItH9qvnVTqFDJN+qYoE0N3fx/pAcy796AMQr69wF7jmk1E4AeSg3/XrfkEssEPbfct3l5V0CSdyOD6cCFdylCC5Z7uTv+Zlp4/jFEXnOb9ZYjDQyuFN/zy0C4TPk8MHw+PgE4zxfPv07vNxGL0quq9cGwRQH5tWH5OjYW5dx2/xqwnJyN7gt52+T2xf8aXdwuDu6zoJNH6BKCuYO8Or7TQMpCCYO311A0FS2JCI+7i8w9r9IWDKX077iUDRTpiARQDi9EvHLl7jnL2aQak0KyUU1il7NUZSedftpH2fl4jH3VfR8eZ64IS92SlVOmzzUnUSUYmA51jJmBZB5ECgAHeHO0MILsr+QFJaaW0Pn8XZn7XCOkY4CPMj0s7dOH0xm12IGPJ12XKPcFOJtx9zHzAxsGQwHjcMFMNVpbHSd/bLyg0x44SQ90CQNJZuMC74Ta79xERwQAsIdx3jdzTkCJn9nvJ+ogyEvPkEbgADvGe7Nzg8LVCPn+sQkVinq8tx73KDxSsbGLkEcww38fgV89t9L5pierrC+UzZnn+6weRLiRiXi9FH5DhwNTaJ5w/V7hZ3HDCEK7GxrjnoOpgQOX4yrQEg14ET3Yk+V04hHl+Ox0CYovDIncannAPZoJ1GjcSKckye07NQ6dYQCkOSuCNmH26NJFfxQ+v7NcXwDJOHc7IzRMycjPUEoQRfg51Mp5vLY8tbgyU2Z4lrvxZJgYzui0QG0rKst7o8UgIBpmVyF+7KbZoW7GBeAuSIx0c6PVb+hLyJLRJmZUT2/DZ35StpwfCcym3uyjHDiLDFEhm3uSs3T8cYD9ZFnyBPHM+Pne7FOQSQKLYnWjhdF6LyUgVjn89JbucsFf78BmOHzlB/riePbktMMeUd5npBPNOV8vMwckb8b0tJ4BkkEHRMu40k8EhFCYTym5Dm+VNPcpEi2Zl21xHoaOIuEHx3FTHbTdMUub/bo26yPYajOR4EjJQFwaG35rz49gixnignZ5MtyLuJvj+X7yQK1vWJ7RnSt0Fo+sTxaFiqijtqPvz20JDlZsuumUUBt0ciePkXLI9gCaXb3VzBpDM9O0VDb3dRoNSCK7OLYr07woWMsNrKKxLhfijZ/518SGLuJSU393LCdYeLo1g403L1qBOhbPCIiH2X8BHDBcI83g2TZNWDO37RD1CYtcNgDBcJ7iMYDOgD/MyI5HzyVVhp515eDIRRVRT17nibrZlS8JVnKrNlUTms94ig685gIOXj7KV80jjlerYbb7OyuJoz1CMX0KrC5DvDgWzkhulNmeK408UBgYU3avrCennK/mFeFs54RnI+ygegEpr/vKIoC9QbrQ4Rb46ebEXlHG6uaHVrZGUbD8rNFW6sKRrDG37G4yxMYsSDZMwQiaOv6bkiHKxF2jdT9OXBYPNg/a7fuS2rj9kPS6DdRNC7MQLbrPFGSDTT88LMCu2KEZubyPljRypV2ajh1CL2y3Qj5LePWBH4riSUEr7WjI/v9L7KhDMZ1sJJXCqwSsgy8P4XQt7hL3IfgBcuNlInnclLd264YjsVTLpbRhYhJhGapEWcD9uwChHJdMliIJY5jQhqiiUhLGUAQ30qx6erYqOlRxb4cpxU/bv2jpokzPA9HCdX/4ZYHfwtFVJukPU7niTxiQpF0wcu8txf9Ry0V9XZyN2ei/9+doCG1vAh2zPxw9LYTSVRHZDn4Z/ZVSxlDvueyMI/SfIJmqSc7j2eg/9aifb4UDOH+vbx2Grc1zDu/Akf/hwNCoiwMxxWJeQ8w7BuItk3VddRpSgKJywNdFNpfXuew3IWiEFSXf08H71Dnn5ofqmCEdkelUwejSfeUXVoXmCCopzI7+LCtCpY5f4jvSu11eWEOiZyTkXMpadCiCYZ1dk5i5eNCM/u2kYSIy1CyDqdbEuPLzWT853Yqm38ELGihvDzrjsLIYGclDc0RBfipCQzi/tMBxFwX1PcLQDczn1VQyJIOcrq3htSGSkfUc8nDj3EYraa1c+GNGgrpYRYYNmQemcGuhHyTt2P8HGOUNIEYTiCeRTbJxEjMiKy1mZhvqoPwPwPZRmRCEI84z7j0OFjvtEKIPvliNWOQ4MOiDuOVCBA0FkEuaGO7hAA/4aABnHPgcsx/sOxhT+fESDitmm5Uaxqv+f6RgVfSCq1I6zHLei0rYS4S09pFTF2QpJTl9bJmRAxws/5TlBpJmC+HOEarufYcC6115zvQvoFCEVa1aP70uJAUp0d5gfdTOW1zptcdkVCFVG8PhJOHSH+cD9VOhH15fZhL1ntJqK9GhzM1DGtYZcxldkZoQwnd9+QD4iWM0msMDIoAECpE89dtCSSKLDX4b0tYHW4p1Ffb9EYMdAdw/HlXucn9e1pm0HWZ8owmuNoApCDkN3ky6ddkoMgRZpodOMulYM3T44P0bdwBlEQM/qw10hF6ahFQUa0CcG2nGc8KPo9nodqvocg6IuE101PLwR073Fy9avgqgSMV+PGLR6N4t721EFnEW2f4QS1CleOG6SROT9NsjOttIYyuKXB8ZM60izPbxzBt3FzXhLhx0GIYuPigjC+FtKjqM3Pe6fWVXiBU1ocbqnTqzMPq68HO2/Lw1c1kmG85ZwofnWEdU/MvBYvpIH+YAWJmC+mJ7w9UN+YubXprtRnKsTKR7xuZKQvAeotDQulkjYBFMJSN6c3ZXKcUljbQwKqJ8Ylhd2l+eR4Gq8pl2miUL4ewzzGXRX9sjr5DM/I6K6PhND8IgcXZwXGhQhVCS4O22o3fDJOxPjISRxjWyyZagLl171WdcgEOb48zyoVYtJDG5ADT1RnvhHn1d8Ed1XDJCU2DsK/UhGuAxFtcpPlM0/FcyNRMB8RkF5SbSBacu8qXS6ynS8ikXGEgKDrasQCwufw5Jz2RkCPwF+tlyXuw3q57R9VKVc4ifW7RQ7anaKQFfmR641HiO40Q+RHFIBGaBdl8u5NrHTKHL7pQu8T/APDrg9PiRZmARECh7Ntv5Hs0trkG0zRUDQ4vrZoezeuNne42Mvx5nUYXrZ+wgUnzl2nJyzVjEjESDlHXiVvDARipmiyIg2cY6+dfIPNwXReBNg71AJByB0pGBDkpNC3MlztKTLaUaktdqn7iHZeTuKcA4yCvObGbdlBOuaJUis9+iFEhAF9L6uH48WWz7/RjhKWH+7uBO1A8FGX16YhvRO0A46qS5AIHFIB3dRiFrcoiMDOuwdLlnZ/5C9y2/vw3Pf8yF0kYm4JSROtrQX27ryMlYpWzX3ueUWzSsaF8uRW9xKMqfoFT43uKrr5x3yK9moi443oHiYmZl5KpGzq9vDmvuxU1Emmx2CJmCM1ecRkjNwDkR6HKCge0EgCZmqGr9uDmDE4vjxR7A1Q153bjSrg/chzb4+eotMdRcIlSRcNX4+VPUvY07hn0pc71fKRR7kRxF4F7JAj9yYzdXwD6BZ8frsWz6Hbfs2t2uGOxWEZuaNW/wuvx2GmpqiE40PCMgfHRwoKpiXzJ5pEMDzzW4l+1QbUNL5+XjVZ5fJ9+p3a9mgBN+IiAs6rQcRy7+pRBEB2aUY7ed+6XbWEWVqTeQhGXY1HEEYJxu96HNsIcJOkHK3U0RZZk0EqHjQgRmsEtMhxKHy2A9N4f3n74RFwPwRIAdipNv2GCjCj71TPGIrb75oINuZdHtfYNaupk93OOFKgRWs8r+R/Kw4oBNxXU6OHYrS5NbFrNAHv+opIag1RlH+z7KYstYYwSnonxvdLttVEr2+PRAz7uxchLborPSzQiyQgfpUyUukFmteSVbrOANmXopXMaP2P/A0ffwCxEk+/15OnUKBg2eZDTrw+wfVvmVJIaKwycPym3zf178NzUbW5adKTk09q8QOg5vcx1iY09LIEgHcDz0pEzqboqYDRghh5FFh7evBRcl1JAUrtKZYanohAxss/MlLgXsk03UjXRLNBUOxAhju7mlg2e9FGZLKF2K8p+npnh2rqZGcN2g2+b/XGqywlqRhRc6uRguIxB+IQUV/vK2akwoxf1NTIrmTI66nHFzJeQlEEQBrTiXg/w2FXzHTeXXNFfEdfLmKlqlletK/NV6c6EDM9ydwjPxPD+ErPyFQTcuubquH9o146JvUuASfpWOLWvoubSAhSM13+twbrRvpzRM6QJykNOc4TCjtPOC7avbUGVSee8Zl5RoSqzqFqMG9w2ZWeJUwCXkmo/AoIe5i5xOQdRnEE03t3LADwnuT0+2rEfQpcNfo4jafU6dW9rxNuSh+3MCPTy3tWPizHGWPXB1jdQ5Xt/DJDM6p7VqQ7YouQwfTq3tzAqJSh32Z9zpaHvJhHh3V5dW8+PPI0j7A2h/nble9xPeLmcPfKZlXzgIhEXd43BXbnqTHc/vW+/fYDkVrl8PK8ldp8I/5Kte4241aUDDg5eY8S3RSLqanQbZa9TXGEyu9H78GM2J9d1ADUVwGheEUftzdVuKe7e4swEIGm8nb30mM6v1TcVhtTajIHoIexq6ofJQ2N2rZZYouge4Ql+1fiaSq3CYO8c95JycepvFaNF+VwWGOJi2H/+/fzv728lUqf9waQ9e6J0AKiXtTctUEQnMFlcIZ7rtY8dtshgzNZ0PMbY1OcVCrSixTlAAVor2zS6Km5dROSE05VtDO5rvyefEWdVbSSkPl6qTpWvIQq+tDKHbCoKe7mBvH8biQ8oQCNgEiHe0spwo5KwE13Qp2vadyf+A/PJJtJ6RrX64wSz5Pz/P4447EFyGOo6v4dv7m91n3Wdq2JB+RipJ5G0DhdwklXjylTAobnmr0+uZXxr8GoVzzTLJ7xr8Go14pq33Lmd38lwxEqXbcbQam9ZU/jjePXC1w/YtbFDVdwakSB20vkMBvxdhyvbpqzpsP11dffRqjRGXYeCbToOl8jWk1MYpPj7eV0u/Tecs0tOs57JPunH2ALZj2uVA3BtYbfamAmEfN3ApzQ1Uh3bGlsK6/80g8dlxk2B5xXXJ5Yp47gvijfmU75OEPu/fghf4GP4SEEoVPXhNTRbHPLZewwj8ySPjajJXZdorbF6KYR4XMMf2hFQtOJeKeYmM6rXkFriWLjRuG9y1KM0RLFHjJyWedaotgl/vqIOeOpnaS/qOBcxjzi0Tfe7YJ1e7lta6/39fc22k0NcgYIj31oOXW9OEN9dwWg8lkV0LRMsbenVHASdi8SwcajMQsq0T43CHB6fbyMbtzSslItsWs0sCDixJ/TaESM1ytKxvVVRr619IKpEaHwuhKwXss8Hnp07WMnz6sXi56ZbKndHPVA+2W0LQjgWfH4CypMvgAQKY8bT4dOWkN0eizvTbIrgaub+DV6IpB48tpsS/R6H/8zL4iwLgHt9ZdTqqcCLEnQMrvuclnDi8et59f6DXRzNYUfLVrMx8Yzc7aSDf22HurPJ3S/H6bcLX5q3alDe152fwNgXhdf20nMYk48RNTxdX/Et57Jiz8TNC/QujcQzOez1j50blq5P89n5yGLTLtzODKvFN2xP2Vikw/PvG42QXNj93JYrQf2evLo6QQOdThtqCotWJ90pdyDOF9ljfgO0+LKlmizbFJl0bRxfOU/ZIS/wXKVem1Bm/nswFOz/v2TX12hAhdnl55+1+pmeXlPQRu/Exnmm5a3RrSR8hjR8rBoA/4ffzwna7xKAAA=\'\nINDICES_CSV_SHA256 = \'b6f444fbec48681c1e5b46a5c22aa197ad91c7fbbdaff82c3ed3a9c63f7c15a4\'\nSCHEMA_CFSV2 = \'worcap_cfsv2_pentad_media_auditada_v1\'\nPOLITICA_CFSV2 = \'ensemble_completo_ou_excecao_201908_m17_auditada_v2\'\nFONTE_CFSV2 = \'https://iridl.ldeo.columbia.edu/SOURCES/.Models/.NMME/.NCEP-CFSv2/.HINDCAST/.PENTAD_SAMPLES_FULL/.prec/\'\nFEATURE_SEAS5 = \'seas5_tp_prevista_media_T_emitida_Tmenos1\'\nFEATURE_SEAS5_ANOM = \'seas5_tp_anomalia_mensal_treino_fold\'\nFEATURE_CFSV2 = \'cfsv2_tp_prevista_media_T_emitida_Tmenos1\'\nFEATURE_CFSV2_ANOM = \'cfsv2_tp_anomalia_mensal_treino_fold\'\nNOMES_MODELOS = [\'arvores_seas5\', \'arvores_multissistema\']\nFASES_CFSV2 = dict(desenvolvimento=(\'1982-02-01\', \'2020-12-01\', 467), somente_ajuste_final=(\'2021-01-01\', \'2022-12-01\', 24), teste=(\'2023-01-01\', \'2024-12-01\', 24))\nMOS_ALPHA = 0.1\nMOS_FRACAO = 0.25\n\ndef exigir(condicao, mensagem):\n    if not bool(condicao):\n        raise ValueError(mensagem)\n\ndef sha256(arquivo):\n    digest = hashlib.sha256()\n    with Path(arquivo).open(\'rb\') as stream:\n        for bloco in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(bloco)\n    return digest.hexdigest()\n\ndef salvar_json(nome, conteudo):\n    caminho = SAIDA / nome\n    caminho.write_text(json.dumps(conteudo, indent=2, ensure_ascii=False, default=str, allow_nan=False), encoding=\'utf-8\')\n    return caminho\n\ndef localizar_dados(pasta_manual=None):\n    obrigatorios = (\'treino_tp.nc\', \'treino_tp_alvo.nc\', \'teste_features.nc\', \'sample_submission.csv\')\n    if pasta_manual is not None:\n        candidatas = [Path(pasta_manual)]\n    elif Path(\'/kaggle/input\').is_dir():\n        candidatas = sorted({p.parent for p in Path(\'/kaggle/input\').rglob(obrigatorios[0])})\n    else:\n        candidatas = [Path.cwd() / \'data\' / \'raw\', Path.cwd().parent / \'data\' / \'raw\']\n    validas = sorted({p.resolve() for p in candidatas if all(((p / nome).is_file() for nome in obrigatorios))})\n    exigir(len(validas) == 1, f\'Exactly one folder containing the four official files is required. Attach the competition data or set PASTA_DADOS_MANUAL. Complete folders found: {validas}\')\n    return validas[0]\n\ndef conferir_campo(campo, referencia, datas):\n    exigir(set(campo.dims) == {\'time\', \'lat\', \'lon\'}, \'Unexpected atmospheric dimensions.\')\n    for eixo in (\'lat\', \'lon\'):\n        exigir(np.array_equal(campo[eixo].values, referencia[eixo].values), f\'Misaligned atmospheric grid: {eixo}.\')\n    exigir(pd.DatetimeIndex(campo.time.values).equals(pd.DatetimeIndex(datas)), \'Misaligned atmospheric dates.\')\n\ndef ajustar_clima_atmosfera(campos, origens):\n    origens = pd.DatetimeIndex(origens)\n    resultado = {}\n    for nome in VARIAVEIS:\n        campo = campos[nome]\n        indices = pd.DatetimeIndex(campo.time.values).get_indexer(origens)\n        exigir((indices >= 0).all(), \'Missing atmospheric fitting months.\')\n        mapas = []\n        for mes in range(1, 13):\n            ids = indices[origens.month == mes]\n            exigir(len(ids) > 0, \'Calendar month with no atmospheric history.\')\n            valores = campo.values[ids]\n            exigir(np.isfinite(valores).all(), f\'{nome}: nonfinite training values.\')\n            mapas.append(valores.mean(axis=0, dtype=np.float64).astype(np.float32))\n        resultado[nome] = np.stack(mapas)\n    return resultado\n\ndef reconstruir_chuva(clima_tp, anomalias, peso):\n    peso = float(peso)\n    exigir(np.isfinite(peso) and peso >= 0, \'The weight must be finite and nonnegative.\')\n    meses = pd.DatetimeIndex(anomalias.time.values).month.to_numpy() - 1\n    valores = np.maximum(0, clima_tp.values[meses] + peso * anomalias.values)\n    exigir(np.isfinite(valores).all(), \'Rainfall reconstruction produced invalid values.\')\n    return xr.DataArray(valores.astype(np.float32), dims=anomalias.dims, coords=anomalias.coords, name=\'tp_mm_day\', attrs={\'units\': \'mm/day\'})\n\ndef auditar_chuva_desenv():\n    with xr.open_dataset(PASTA / \'treino_tp.nc\') as ds, xr.open_dataset(PASTA / \'treino_tp_alvo.nc\') as da:\n        bruto, alvo = (ds.tp, da.tp_alvo)\n        esperado = pd.date_range(\'1940-01-01\', CORTE_FINAL, freq=\'MS\')\n        exigir(set(bruto.dims) == set(alvo.dims) == {\'time\', \'lat\', \'lon\'}, \'Unexpected precipitation dimensions.\')\n        bruto, alvo = (bruto.transpose(\'time\', \'lat\', \'lon\'), alvo.transpose(\'time\', \'lat\', \'lon\'))\n        exigir(bruto.shape == alvo.shape == (996, 301, 261), \'Unexpected grid or calendar.\')\n        exigir(pd.DatetimeIndex(bruto.time.values).equals(esperado), \'Official calendar mismatch.\')\n        exigir(bruto.attrs.get(\'units\') == alvo.attrs.get(\'units\') == \'mm/day\', \'Rainfall must use mm/day; no conversion is applied.\')\n        for eixo in [\'time\', \'lat\', \'lon\']:\n            exigir(np.array_equal(bruto[eixo].values, alvo[eixo].values), f\'Misaligned target at {eixo}.\')\n        for eixo in [\'lat\', \'lon\']:\n            exigir(np.all(np.diff(bruto[eixo].values) == 0.25), f\'Invalid grid: {eixo}.\')\n        chuva = bruto.sel(time=slice(None, FIM_DESENVOLVIMENTO)).load()\n        exigir(np.isfinite(chuva.values).all() and (chuva.values >= 0).all(), \'Invalid historical precipitation.\')\n        for i in range(0, chuva.sizes[\'time\'] - 1, 24):\n            j = min(i + 24, chuva.sizes[\'time\'] - 1)\n            exigir(np.array_equal(alvo.isel(time=slice(i, j)).values, chuva.values[i + 1:j + 1]), \'tp_alvo[M] does not match tp[M+1].\')\n    return chuva\n\ndef carregar_atmosfera(referencia, inicio, fim):\n    resultado, metadados = ({}, [])\n    esperado_completo = pd.date_range(\'1940-01-01\', CORTE_FINAL, freq=\'MS\')\n    esperado_recorte = pd.date_range(inicio, fim, freq=\'MS\')\n    for nome in VARIAVEIS:\n        with xr.open_dataset(PASTA / f\'treino_{nome}.nc\') as ds:\n            bruto = ds[nome].transpose(\'time\', \'lat\', \'lon\')\n            conferir_campo(bruto, referencia, esperado_completo)\n            campo = bruto.sel(time=slice(inicio, fim)).astype(\'float32\').load()\n            conferir_campo(campo, referencia, esperado_recorte)\n            exigir(np.isfinite(campo.values).all(), f\'{nome}: nonfinite atmospheric data.\')\n            resultado[nome] = campo\n            metadados.append(dict(variavel=nome, atributos=dict(bruto.attrs), inicio=str(inicio), fim=str(fim), minimo=float(campo.min()), maximo=float(campo.max())))\n        print(\'Loaded:\', nome, campo.shape, flush=True)\n    return (resultado, metadados)\n\ndef carregar_indices_incorporados():\n    dados = gzip.decompress(base64.b64decode(INDICES_CSV_GZIP_BASE64))\n    exigir(hashlib.sha256(dados).hexdigest() == INDICES_CSV_SHA256, \'Embedded NOAA snapshot hash mismatch.\')\n    tabela = pd.read_csv(io.BytesIO(dados), parse_dates=[\'time_origem\'])\n    exigir(tabela.columns.tolist() == [\'time_origem\'] + INDICES_NOMES, \'Ocean index columns differ from the reference.\')\n    tabela = tabela.set_index(\'time_origem\')\n    conferir_indices(tabela)\n    esperado = pd.date_range(FONTES_INDICES[\'inicio\'], FONTES_INDICES[\'fim\'], freq=\'MS\')\n    exigir(tabela.index.equals(esperado), \'The NOAA snapshot has missing or altered dates.\')\n    return (tabela, dados)\n\ndef conferir_indices(tabela):\n    exigir(tabela.columns.tolist() == INDICES_NOMES, \'The order of the four ocean indices changed.\')\n    datas = pd.DatetimeIndex(tabela.index)\n    exigir(len(datas) > 0 and datas.is_unique and datas.is_monotonic_increasing, \'NOAA dates are empty, duplicated or unordered.\')\n    exigir(datas.equals(pd.date_range(datas.min(), datas.max(), freq=\'MS\')), \'NOAA dates must be consecutive first-of-month timestamps.\')\n    exigir(np.isfinite(tabela.to_numpy(dtype=np.float64)).all(), \'Missing or infinite NOAA index. Do not interpolate or fill from future months.\')\n\ndef auditar_dataset(ds, esperado=None):\n    exigir(ds.attrs.get(\'schema\') == SCHEMA, \'Unknown SEAS5 artifact version.\')\n    exigir(ds.attrs.get(\'dataset\') == DATASET and str(ds.attrs.get(\'system\')) == SYSTEM, \'Invalid SEAS5 provenance.\')\n    exigir(ds.attrs.get(\'product_type\') == \'monthly_mean\' and int(ds.attrs.get(\'leadtime_month\')) == 2, \'Incorrect product or lead time.\')\n    exigir(ds.seas5_tp_media.attrs.get(\'units\') == \'mm/day\', \'SEAS5 must use mm/day after conversion.\')\n    exigir(ds.seas5_tp_media.dims == (\'time\', \'lat\', \'lon\'), \'Unexpected SEAS5 dimensions.\')\n    datas = pd.DatetimeIndex(ds.time.values)\n    exigir(datas.is_unique and datas.is_monotonic_increasing, \'Duplicate or unordered targets.\')\n    exigir(datas.equals(pd.date_range(datas.min(), datas.max(), freq=\'MS\')), \'Missing SEAS5 target months.\')\n    if esperado is not None:\n        exigir(datas.equals(pd.DatetimeIndex(esperado)), \'SEAS5 dates do not match the request.\')\n    exigir(pd.DatetimeIndex(ds.time_origem.values).equals((datas.to_period(\'M\') - 1).to_timestamp()), \'SEAS5 must be initialized exactly in T-1.\')\n    exigir(np.isin(ds.n_membros.values, [25, 51]).all(), \'Incomplete ensemble member set.\')\n    for eixo in [\'lat\', \'lon\']:\n        v = ds[eixo].values\n        exigir(len(v) > 1 and np.all(np.diff(v) == 1), f\'Incorrect native one-degree grid: {eixo}.\')\n    exigir(ds.lat.min() <= -60 and ds.lat.max() >= 15 and (ds.lon.min() <= -90) and (ds.lon.max() >= -25), \'The input domain does not cover the official grid. Do not extrapolate.\')\n    exigir(np.isfinite(ds.seas5_tp_media.values).all() and (ds.seas5_tp_media.values >= 0).all(), \'Invalid SEAS5 values.\')\n\ndef localizar_seas5():\n    if PASTA_SEAS5_MANUAL:\n        candidatos = [Path(PASTA_SEAS5_MANUAL) / \'seas5_51_manifesto.json\']\n    else:\n        candidatos = []\n        for raiz in [Path(\'/kaggle/input\'), Path(\'/kaggle/working/worcap_SEAS5_dados\'), Path.cwd() / \'outputs/worcap_SEAS5_dados\']:\n            if raiz.is_dir():\n                candidatos.extend(raiz.rglob(\'seas5_51_manifesto.json\'))\n    candidatos = sorted(set((p.resolve() for p in candidatos if p.is_file())))\n    exigir(len(candidatos) == 1, \'Attach the prepared SEAS5 dataset or set PASTA_SEAS5_MANUAL. Exactly one SEAS5 manifest is required.\')\n    caminho = candidatos[0]\n    m = json.loads(caminho.read_text(encoding=\'utf-8\'))\n    exigir(m.get(\'schema\') == SCHEMA and m.get(\'system\') == SYSTEM and (m.get(\'dataset\') == DATASET), \'Incompatible SEAS5 manifest.\')\n    exigir(m.get(\'leadtime_month\') == 2 and m.get(\'paramId\') == 172228 and (m.get(\'product_type\') == \'monthly_mean\') and (m.get(\'variable\') == \'total_precipitation\'), \'SEAS5 product does not match the experiment.\')\n    return (caminho.parent, m)\n\ndef carregar_seas5(uso, referencia):\n    item = MANIFESTO_SEAS5[\'arquivos\'][uso]\n    caminho = PASTA_SEAS5 / item[\'arquivo\']\n    exigir(caminho.is_file() and sha256(caminho) == item[\'sha256\'], f\'Missing or altered SEAS5: {uso}.\')\n    with xr.open_dataset(caminho) as ds:\n        bruto = ds.load()\n    auditar_dataset(bruto, pd.date_range(item[\'inicio\'], item[\'fim\'], freq=\'MS\'))\n    exigir(bruto.attrs.get(\'uso\') == uso, \'SEAS5 file partition mismatch.\')\n    grade = bruto.seas5_tp_media.interp(lat=referencia.lat, lon=referencia.lon, method=\'linear\').astype(np.float32)\n    exigir(np.isfinite(grade.values).all() and (grade.values >= 0).all(), \'Invalid SEAS5 interpolation.\')\n    for coord in [\'lat\', \'lon\']:\n        exigir(np.array_equal(grade[coord].values, referencia[coord].values), \'Misaligned SEAS5 grid.\')\n    grade.attrs.update(units=\'mm/day\', system=SYSTEM, leadtime_month=2)\n    return grade\n\ndef valores_seas5(seas, destino, pontos=None):\n    destino = pd.Timestamp(destino)\n    exigir(pd.DatetimeIndex(seas.time.values).is_unique, \'Duplicate SEAS5 targets.\')\n    i = pd.DatetimeIndex(seas.time.values).get_indexer([destino])[0]\n    exigir(i >= 0, f\'Missing SEAS5 prediction for {destino.date()}.\')\n    origem = pd.Timestamp(seas.time_origem.values[i])\n    exigir(origem == (destino.to_period(\'M\') - 1).to_timestamp() and origem < destino, \'SEAS5 initialization is incorrect or beyond the cutoff.\')\n    v = seas.values[i].reshape(-1)\n    if pontos is not None:\n        v = v[pontos]\n    exigir(np.isfinite(v).all(), \'Nonfinite SEAS5 feature.\')\n    return v\n\ndef amostrar_m5_documentado(chuva, atmosfera, seas, corte):\n    destinos = calendario_pareado(corte)\n    origens = (destinos.to_period(\'M\') - 1).to_timestamp()\n    ref_origens, ref_alvos = calendario_pares(atmosfera[VARIAVEIS[0]].time.values, corte, 30)\n    clima, somas, contagens = estatisticas_climatologia(chuva, corte, 30)\n    exigir(np.all(contagens == 30), \'Rainfall climatology must contain exactly 30 years.\')\n    ca = ajustar_clima_atmosfera(atmosfera, ref_origens)\n    ca[\'_clima_indices\'] = ajustar_clima_indices(INDICES_OC, ref_origens, corte)\n    exigir(set(destinos).issubset(set(ref_alvos)), \'Example outside the reference window.\')\n    ngrade = chuva.sizes[\'lat\'] * chuva.sizes[\'lon\']\n    n = min(PONTOS_POR_MES, ngrade)\n    X = np.empty((len(destinos) * n, 28), dtype=np.float32)\n    y = np.empty(len(X), dtype=np.float32)\n    ids = np.empty((len(destinos), n), dtype=np.int32)\n    for i, t in enumerate(destinos):\n        pontos = np.random.default_rng(np.random.SeedSequence([SEMENTE, t.year, t.month])).choice(ngrade, size=n, replace=False)\n        ids[i] = pontos\n        basicas = matriz_mes(atmosfera, ca, clima, t, pontos)\n        obs = chuva.sel(time=t).values.reshape(-1)[pontos]\n        basicas[:, 22] = ((somas[t.month - 1].reshape(-1)[pontos] - obs.astype(np.float64)) / 29).astype(np.float32)\n        sl = slice(i * n, (i + 1) * n)\n        X[sl, :27] = basicas\n        X[sl, 27] = valores_seas5(seas, t, pontos)\n        y[sl] = obs - basicas[:, 22]\n    exigir(np.isfinite(X).all() and np.isfinite(y).all(), \'Nonfinite training data.\')\n    exigir((origens < destinos).all() and destinos.max() <= pd.Timestamp(corte), \'Temporal cutoff violated.\')\n    return (X, y, ids, destinos, clima, ca, somas, contagens)\n\ndef valores_anomalia_seas5(seas, clima_seas, destino, pontos=None):\n    for eixo in [\'lat\', \'lon\']:\n        exigir(np.array_equal(seas[eixo].values, clima_seas[eixo].values), \'Misaligned SEAS5 climatology.\')\n    bruto = valores_seas5(seas, destino, pontos)\n    referencia = clima_seas.sel(month=pd.Timestamp(destino).month).values.reshape(-1)\n    if pontos is not None:\n        referencia = referencia[pontos]\n    anomalia = (bruto - referencia).astype(np.float32)\n    exigir(np.isfinite(anomalia).all(), \'Nonfinite SEAS5 anomaly.\')\n    return anomalia\n\ndef amostrar_arvores_seas5(chuva, atmosfera, seas, corte):\n    x5, y, ids, destinos, clima, ca, somas, contagens = amostrar_m5_documentado(chuva, atmosfera, seas, corte)\n    clima_seas = ajustar_clima_seas5(seas, corte)\n    X = np.empty((len(x5), 29), dtype=np.float32)\n    X[:, :28] = x5\n    n = ids.shape[1]\n    for i, t in enumerate(destinos):\n        X[i * n:(i + 1) * n, 28] = valores_anomalia_seas5(seas, clima_seas, t, ids[i])\n    exigir(np.array_equal(X[:, :28], x5), \'The 28 base SEAS5 features changed.\')\n    exigir(np.isfinite(X).all() and np.isfinite(y).all(), \'Nonfinite training data.\')\n    del x5\n    return (X, y, ids, destinos, clima, ca, somas, contagens, clima_seas)\n\ndef validar_manifesto_cfsv2(m):\n    exigir(m.get(\'schema\') == SCHEMA_CFSV2 and m.get(\'modelo\') == \'NCEP-CFSv2\' and (m.get(\'fonte\') == FONTE_CFSV2), \'Incompatible CFSv2 manifest.\')\n    exigir(m.get(\'lead_iri\') == 1.5 and m.get(\'unidade\') == \'mm/day\' and (m.get(\'chuva_observada_utilizada\') is False), \'Incorrect CFSv2 product or temporal alignment.\')\n    exigir(m.get(\'meses\') == 515 and m.get(\'inicio_alvos\') == \'1982-02-01\' and (m.get(\'fim_alvos\') == \'2024-12-01\'), \'CFSv2 coverage mismatch.\')\n    exigir(m.get(\'politica_membros\') == POLITICA_CFSV2 and m.get(\'excecoes_permitidas\') == {\'2019-08-01\': [17]}, \'CFSv2 ensemble policy mismatch.\')\n    aplicadas = m.get(\'excecoes_aplicadas\', [])\n    exigir(isinstance(aplicadas, list) and len(aplicadas) <= 1, \'Unspecified CFSv2 exceptions.\')\n    if aplicadas:\n        r = aplicadas[0]\n        exigir(r.get(\'time_origem\') == \'2019-08-01\' and r.get(\'time_alvo\') == \'2019-09-01\' and (r.get(\'membros\') == 23) and (r.get(\'membros_esperados\') == 24) and (r.get(\'membros_indisponiveis_ids\') == [17]), \'Incorrect CFSv2 exception.\')\n        a = r.get(\'auditoria_excecao\', {})\n        exigir(a.get(\'membros_indisponiveis_ids\') == [17] and np.isfinite(a.get(\'maximo_delta_media_mm_day\', np.nan)) and (0 <= a[\'maximo_delta_media_mm_day\'] <= 1e-05), \'Exception missing the individual-member mean check.\')\n    for fase, (inicio, fim, n) in FASES_CFSV2.items():\n        item = m.get(\'arquivos\', {}).get(fase, {})\n        exigir(item.get(\'arquivo\') == f\'cfsv2_{fase}.nc\' and item.get(\'inicio_alvos\') == inicio and (item.get(\'fim_alvos\') == fim) and (item.get(\'meses\') == n), \'Incorrect CFSv2 partition.\')\n\ndef localizar_cfsv2():\n    if PASTA_CFSV2_MANUAL:\n        candidatos = [Path(PASTA_CFSV2_MANUAL) / \'cfsv2_manifesto.json\']\n    else:\n        candidatos = []\n        for raiz in [Path(\'/kaggle/input\'), Path(\'/kaggle/working/worcap_CFSv2_dados\'), Path.cwd() / \'outputs/worcap_CFSv2_dados\']:\n            if raiz.is_dir():\n                candidatos.extend(raiz.rglob(\'cfsv2_manifesto.json\'))\n    candidatos = sorted(set((p.resolve() for p in candidatos if p.is_file())))\n    exigir(len(candidatos) == 1, \'Attach the prepared CFSv2 output or set PASTA_CFSV2_MANUAL. Exactly one CFSv2 manifest is required.\')\n    caminho = candidatos[0]\n    m = json.loads(caminho.read_text(encoding=\'utf-8\'))\n    validar_manifesto_cfsv2(m)\n    return (caminho.parent, m)\n\ndef auditar_cfsv2(ds, uso, m):\n    inicio, fim, n = FASES_CFSV2[uso]\n    datas = pd.date_range(inicio, fim, freq=\'MS\')\n    origens = (datas.to_period(\'M\') - 1).to_timestamp()\n    exigir(ds.attrs.get(\'schema\') == SCHEMA_CFSV2 and ds.attrs.get(\'modelo\') == \'NCEP-CFSv2\' and (ds.attrs.get(\'fonte\') == FONTE_CFSV2) and (ds.attrs.get(\'lead_iri\') == 1.5), \'CFSv2 NetCDF provenance mismatch.\')\n    exigir(ds.attrs.get(\'politica_membros\') == POLITICA_CFSV2, \'CFSv2 NetCDF is missing the corrected ensemble policy.\')\n    exigir(ds.cfsv2_tp_media.dims == (\'time\', \'lat\', \'lon\') and ds.cfsv2_tp_media.attrs.get(\'units\') == \'mm/day\', \'Invalid CFSv2 dimensions or units.\')\n    exigir(pd.DatetimeIndex(ds.time.values).equals(datas), \'Missing, extra or unordered CFSv2 dates.\')\n    exigir(pd.DatetimeIndex(ds.time_origem.values).equals(origens), \'CFSv2 nominal issue month is not T-1.\')\n    exigir(np.array_equal(ds.lat.values, np.arange(-61, 17)) and np.array_equal(ds.lon.values, np.arange(-91, -23)), \'CFSv2 native grid mismatch.\')\n    for var in [\'n_membros\', \'inicializacao_mais_antiga\', \'inicializacao_mais_recente\', \'fase_fonte\', \'time_origem\']:\n        exigir(ds[var].dims == (\'time\',), f\'Invalid CFSv2 coordinate: {var}.\')\n    dmin = pd.DatetimeIndex(ds.inicializacao_mais_antiga.values)\n    dmax = pd.DatetimeIndex(ds.inicializacao_mais_recente.values)\n    exigir(not dmin.hasnans and (not dmax.hasnans) and (dmin <= dmax).all() and (dmax < datas).all() and (dmin >= origens - pd.Timedelta(days=40)).all() and (dmax <= origens + pd.Timedelta(days=7)).all(), \'CFSv2 initializations are outside the allowed window.\')\n    esperados = np.where(origens.month == 11, 28, 24)\n    if m[\'excecoes_aplicadas\'] and pd.Timestamp(\'2019-09-01\') in datas:\n        esperados[datas == pd.Timestamp(\'2019-09-01\')] = 23\n    exigir(np.array_equal(ds.n_membros.values, esperados), \'CFSv2 member counts differ from the manifest.\')\n    exigir(np.array_equal(ds.fase_fonte.values, (origens >= pd.Timestamp(\'2011-04-01\')).astype(np.int8)), \'Incorrect CFSv2 hindcast/real-time transition.\')\n    v = ds.cfsv2_tp_media.values\n    exigir(np.isfinite(v).all() and (v >= 0).all(), \'Invalid CFSv2 predicted rainfall.\')\n\ndef carregar_cfsv2(uso, referencia):\n    item = MANIFESTO_CFSV2[\'arquivos\'][uso]\n    path = PASTA_CFSV2 / item[\'arquivo\']\n    exigir(path.is_file() and sha256(path) == item[\'sha256\'], f\'Missing or altered CFSv2: {uso}.\')\n    with xr.open_dataset(path) as d:\n        bruto = d.load()\n    auditar_cfsv2(bruto, uso, MANIFESTO_CFSV2)\n    grade = bruto.cfsv2_tp_media.interp(lat=referencia.lat, lon=referencia.lon, method=\'linear\').astype(np.float32)\n    exigir(np.isfinite(grade.values).all() and (grade.values >= 0).all(), \'Invalid CFSv2 interpolation; do not extrapolate.\')\n    for eixo in [\'lat\', \'lon\']:\n        exigir(np.array_equal(grade[eixo].values, referencia[eixo].values), \'Misaligned CFSv2 grid.\')\n    for coord in [\'time_origem\', \'n_membros\', \'inicializacao_mais_antiga\', \'inicializacao_mais_recente\', \'fase_fonte\']:\n        grade = grade.assign_coords({coord: (\'time\', bruto[coord].values)})\n    grade.attrs.update(units=\'mm/day\', lead_iri=1.5)\n    salvar_json(f\'auditoria_CFSv2_{uso}.json\', dict(arquivo=str(path), sha256=item[\'sha256\'], meses=bruto.sizes[\'time\'], minimo=float(grade.min()), maximo=float(grade.max()), membros_usados=np.unique(bruto.n_membros.values).tolist(), interpolacao=\'bilinear without extrapolation\'))\n    return grade\n\ndef ajustar_clima_previsao(previsoes, corte, nome):\n    treino = calendario_pareado(corte)\n    datas = pd.DatetimeIndex(previsoes.time.values)\n    exigir(datas.is_unique and datas.is_monotonic_increasing and treino.isin(datas).all(), f\'Incomplete or unordered dates: {nome}.\')\n    sel = previsoes.sel(time=treino)\n    exigir(pd.DatetimeIndex(sel.time_origem.values).equals((treino.to_period(\'M\') - 1).to_timestamp()), f\'Incorrect origin in climatology {nome}.\')\n    v = sel.values\n    exigir(np.isfinite(v).all() and (v >= 0).all(), f\'Invalid values in climatology {nome}.\')\n    n = np.array([(treino.month == m).sum() for m in range(1, 13)], dtype=np.int16)\n    exigir((n > 0).all() and n.max() <= 30, \'Forecast climatology exceeds 30 years or has a missing calendar month.\')\n    medias = np.stack([v[treino.month == m].mean(axis=0, dtype=np.float64) for m in range(1, 13)]).astype(np.float32)\n    return xr.DataArray(medias, dims=(\'month\', \'lat\', \'lon\'), coords={\'month\': np.arange(1, 13), \'lat\': previsoes.lat, \'lon\': previsoes.lon, \'n_meses\': (\'month\', n)}, name=f\'climatologia_{nome}\', attrs=dict(units=\'mm/day\', inicio=str(treino[0].date()), corte=str(pd.Timestamp(corte).date()), meses_treino=len(treino), anos_min=int(n.min()), anos_max=int(n.max()), referencia=\'training-month predictors only; one ensemble mean per year and month\'))\n\ndef ajustar_clima_seas5(seas, corte):\n    return ajustar_clima_previsao(seas, corte, \'SEAS5\')\n\ndef valores_cfsv2(cfs, destino, pontos=None):\n    destino = pd.Timestamp(destino)\n    exigir(pd.Timestamp(cfs.inicializacao_mais_recente.sel(time=destino).values) < destino, \'A real CFSv2 initialization extends into the target month.\')\n    return valores_seas5(cfs, destino, pontos)\n\ndef valores_anomalia_cfsv2(cfs, clima_cfs, destino, pontos=None):\n    for eixo in [\'lat\', \'lon\']:\n        exigir(np.array_equal(cfs[eixo], clima_cfs[eixo]), \'Misaligned CFSv2 climatology.\')\n    bruto = valores_cfsv2(cfs, destino, pontos)\n    ref = clima_cfs.sel(month=pd.Timestamp(destino).month).values.ravel()\n    if pontos is not None:\n        ref = ref[pontos]\n    anom = (bruto - ref).astype(np.float32)\n    exigir(np.isfinite(anom).all(), \'Invalid CFSv2 anomaly.\')\n    return anom\n\n# Both tree components receive exactly the same sampled pixels and residual targets.\ndef amostrar_arvores_multissistema(chuva, atmosfera, seas, cfs, corte):\n    x6, y, ids, destinos, clima, ca, somas, contagens, cs = amostrar_arvores_seas5(chuva, atmosfera, seas, corte)\n    cc = ajustar_clima_previsao(cfs, corte, \'CFSv2\')\n    X = np.empty((len(x6), 31), dtype=np.float32)\n    X[:, :29] = x6\n    n = ids.shape[1]\n    for i, t in enumerate(destinos):\n        sl = slice(i * n, (i + 1) * n)\n        X[sl, 29] = valores_cfsv2(cfs, t, ids[i])\n        X[sl, 30] = valores_anomalia_cfsv2(cfs, cc, t, ids[i])\n    exigir(np.array_equal(X[:, :29], x6), \'The 29 control features changed.\')\n    exigir(np.isfinite(X).all() and np.isfinite(y).all(), \'Invalid training matrix.\')\n    return (X, y, ids, destinos, clima, ca, somas, contagens, cs, cc)\n\ndef prever_par(modelo, nome, atmosfera, seas, cfs, ca, clima, cs, cc, destinos):\n    destinos = pd.DatetimeIndex(destinos)\n    exigir(nome in NOMES_MODELOS, \'Unknown model.\')\n    exigir((destinos > pd.Timestamp(cs.attrs[\'corte\'])).all() and cs.attrs[\'corte\'] == cc.attrs[\'corte\'], \'Inference must follow the training cutoff.\')\n    out = np.empty((len(destinos), clima.sizes[\'lat\'], clima.sizes[\'lon\']), dtype=np.float32)\n    for i, t in enumerate(destinos):\n        x = np.column_stack([matriz_mes(atmosfera, ca, clima, t), valores_seas5(seas, t), valores_anomalia_seas5(seas, cs, t)]).astype(np.float32)\n        if nome == \'arvores_multissistema\':\n            x = np.column_stack([x, valores_cfsv2(cfs, t), valores_anomalia_cfsv2(cfs, cc, t)]).astype(np.float32)\n        exigir(x.shape[1] == (29 if nome == \'arvores_seas5\' else 31) and np.isfinite(x).all(), \'Invalid inference.\')\n        out[i] = modelo.predict(x).reshape(out.shape[1:])\n    exigir(np.isfinite(out).all(), \'Invalid predictions.\')\n    return xr.DataArray(out, dims=(\'time\', \'lat\', \'lon\'), coords={\'time\': destinos, \'lat\': clima.lat, \'lon\': clima.lon}, attrs={\'units\': \'mm/day\'})\n\n# Fit a local two-predictor ridge with training-only monthly means and scales.\ndef mos_fit(chuva, seas, cfs, corte):\n    dates = calendario_pareado(corte)\n    r = chuva.sel(time=dates).transpose(\'time\', \'lat\', \'lon\').values.reshape(len(dates), -1)\n    s = seas.sel(time=dates).transpose(\'time\', \'lat\', \'lon\').values.reshape(len(dates), -1)\n    c = cfs.sel(time=dates).transpose(\'time\', \'lat\', \'lon\').values.reshape(len(dates), -1)\n    for pre in [seas, cfs]:\n        assert pd.DatetimeIndex(pre.sel(time=dates).time_origem.values).equals((dates.to_period(\'M\') - 1).to_timestamp())\n    assert (pd.DatetimeIndex(cfs.sel(time=dates).inicializacao_mais_recente.values) < dates).all()\n    baseline = estatisticas_climatologia(chuva, corte, 30)[0]\n    shape = baseline.shape[1:]\n    ng = r.shape[1]\n    coef = np.zeros((2, ng), np.float64)\n    mu = np.zeros((2, 12, ng), np.float64)\n    for lo in range(0, ng, 4096):\n        hi = min(lo + 4096, ng)\n        y = r[:, lo:hi].astype(np.float64)\n        x = s[:, lo:hi].astype(np.float64)\n        z = c[:, lo:hi].astype(np.float64)\n        assert np.isfinite(y).all() and np.isfinite(x).all() and np.isfinite(z).all()\n        for m in range(1, 13):\n            ix = dates.month == m\n            mu[0, m - 1, lo:hi] = x[ix].mean(0)\n            mu[1, m - 1, lo:hi] = z[ix].mean(0)\n            x[ix] -= mu[0, m - 1, lo:hi]\n            z[ix] -= mu[1, m - 1, lo:hi]\n            y[ix] -= y[ix].mean(0)\n        sx = np.sqrt(np.mean(x * x, axis=0))\n        sz = np.sqrt(np.mean(z * z, axis=0))\n        sx = np.where(sx > 1e-12, sx, 1.0)\n        sz = np.where(sz > 1e-12, sz, 1.0)\n        x /= sx\n        z /= sz\n        a = np.mean(x * x, axis=0) + MOS_ALPHA\n        b = np.mean(x * z, axis=0)\n        d = np.mean(z * z, axis=0) + MOS_ALPHA\n        u = np.mean(x * y, axis=0)\n        v = np.mean(z * y, axis=0)\n        det = a * d - b * b\n        assert (det > 0).all()\n        coef[0, lo:hi] = (d * u - b * v) / det / sx\n        coef[1, lo:hi] = (a * v - b * u) / det / sz\n    assert np.isfinite(coef).all()\n    return dict(coef=coef.reshape(2, *shape), mu=mu.reshape(2, 12, *shape), baseline=baseline.values, lat=baseline.lat.values, lon=baseline.lon.values, corte=str(pd.Timestamp(corte).date()), inicio=str(dates[0].date()), meses=len(dates))\n\ndef mos_predict(fit, seas, cfs, dates):\n    dates = pd.DatetimeIndex(dates)\n    assert (dates > pd.Timestamp(fit[\'corte\'])).all()\n    for pre in [seas, cfs]:\n        assert np.array_equal(pre.lat, fit[\'lat\']) and np.array_equal(pre.lon, fit[\'lon\'])\n    out = []\n    for t in dates:\n        s = valores_seas5(seas, t).reshape(fit[\'baseline\'].shape[1:]).astype(np.float64)\n        c = valores_cfsv2(cfs, t).reshape(s.shape).astype(np.float64)\n        anom = fit[\'coef\'][0] * (s - fit[\'mu\'][0, t.month - 1]) + fit[\'coef\'][1] * (c - fit[\'mu\'][1, t.month - 1])\n        out.append(np.maximum(0, fit[\'baseline\'][t.month - 1].astype(np.float64) + anom))\n    return np.stack(out)\n\ndef matriz_mes(campos, clima_atmos, clima_tp, destino, pontos=None):\n    """Build features at the specified source month. The full training rainfall climatology is used here; the sampler replaces its training column with the leave-one-out value."""\n    destino = pd.Timestamp(destino)\n    origem = origem_mensal([destino], LAG_ATMOSFERA)[0]\n    nlat = clima_tp.sizes[\'lat\']\n    nlon = clima_tp.sizes[\'lon\']\n    pontos = np.arange(nlat * nlon) if pontos is None else np.asarray(pontos)\n    exigir(pontos.ndim == 1 and np.issubdtype(pontos.dtype, np.integer), \'Invalid indices.\')\n    exigir(((pontos >= 0) & (pontos < nlat * nlon)).all(), \'Point outside the grid.\')\n    X = np.empty((len(pontos), len(FEATURES)), dtype=np.float32)\n    for j, nome in enumerate(VARIAVEIS):\n        campo = campos[nome]\n        i = pd.DatetimeIndex(campo.time.values).get_indexer([origem])[0]\n        exigir(i >= 0, f\'Origin {origem.date()} missing in {nome}.\')\n        valores = campo.values[i].reshape(-1)[pontos]\n        clima_origem = clima_atmos[nome][origem.month - 1].reshape(-1)[pontos]\n        X[:, j] = valores\n        X[:, j + len(VARIAVEIS)] = valores - clima_origem\n    X[:, 18] = clima_tp.lat.values[pontos // nlon]\n    X[:, 19] = clima_tp.lon.values[pontos % nlon]\n    X[:, 20] = np.sin(2 * np.pi * destino.month / 12)\n    X[:, 21] = np.cos(2 * np.pi * destino.month / 12)\n    X[:, 22] = clima_tp.values[destino.month - 1].reshape(-1)[pontos]\n    exigir(origem < destino, \'Temporal violation: input origin is not earlier than its target.\')\n    valores_oceano = valores_indices_mes(INDICES_OC, clima_atmos[\'_clima_indices\'], destino)\n    X[:, 23:] = valores_oceano[None, :]\n    exigir(np.isfinite(X).all(), \'Features contain missing or infinite values.\')\n    return X\nSST_COMPONENTES = 8\nSST_NOME = \'ersstv5_4graus_198201_202411.nc\'\nSST_HASH = \'908034f6418833302ea432f25850ad2982d1114b747b7b298e26a350eaf2891d\'\n\ndef sst_auditar_arquivo(path):\n    with xr.open_dataset(path) as ds:\n        exigir(\'sst\' in ds and ds.sst.dims == (\'time\', \'lat\', \'lon\'), \'Incorrect SST variable or dimensions.\')\n        exigir(ds.sst.attrs.get(\'units\') == \'degC\', \'SST must use degrees Celsius.\')\n        exigir(\'ERSSTv5\' in str(ds.attrs.get(\'title\', \'\')) and str(ds.attrs.get(\'product_version\', \'\')) == \'Version 5\', \'Product differs from ERSSTv5.\')\n        t = pd.DatetimeIndex(ds.time.values)\n        exigir(t.equals(pd.date_range(\'1982-01-01\', \'2024-11-01\', freq=\'MS\')), \'Incomplete or different SST calendar.\')\n        exigir(np.array_equal(ds.lat.values, np.arange(60, -61, -4)) and np.array_equal(ds.lon.values, np.arange(0, 360, 4)), \'SST must use the 60S-60N subset on a four-degree grid.\')\n        sst = ds.sst.astype(\'float32\').load()\n        atributos = {k: str(v) for k, v in ds.attrs.items()}\n    v = sst.values\n    validos = v[np.isfinite(v)]\n    exigir(validos.size > 0 and (not np.isinf(v).any()) and (validos.min() >= -1.801) and (validos.max() <= 45), "SST values outside the product\'s physical range.")\n    return (sst, dict(arquivo=Path(path).name, sha256=sha256(path), bytes=Path(path).stat().st_size, inicio_origens=str(t[0].date()), fim_origens=str(t[-1].date()), meses=len(t), grade=[31, 90], minimo=float(validos.min()), maximo=float(validos.max()), atributos=atributos, referencia=\'https://psl.noaa.gov/data/gridded/data.noaa.ersst.v5.html\', origem=\'NOAA PSL / ERSSTv5; retrospective snapshot, without verified historical operational vintages\'))\n\n# Fit the ocean mask, monthly anomalies and PCA using training SST only.\ndef sst_ajustar_pca(sst, alvos_treino):\n    alvos_treino = pd.DatetimeIndex(alvos_treino)\n    exigir(alvos_treino.is_unique and alvos_treino.equals(pd.date_range(alvos_treino.min(), alvos_treino.max(), freq=\'MS\')), \'Invalid PCA fitting calendar.\')\n    origens = origem_mensal(alvos_treino, LAG_SST)\n    treino = sst.sel(time=origens).transpose(\'time\', \'lat\', \'lon\')\n    bruto = treino.values.reshape(len(origens), -1).astype(np.float64)\n    mascara = np.isfinite(bruto).all(axis=0)\n    exigir(mascara.any(), \'No complete training ocean grid cells.\')\n    mensal = np.full((12, bruto.shape[1]), np.nan, dtype=np.float64)\n    contagens = np.array([(origens.month == m).sum() for m in range(1, 13)])\n    exigir((contagens >= 2).all(), \'Insufficient monthly SST climatology.\')\n    for m in range(1, 13):\n        mensal[m - 1, mascara] = bruto[origens.month == m][:, mascara].mean(axis=0)\n    anom = bruto[:, mascara] - mensal[origens.month - 1][:, mascara]\n    ativos = np.flatnonzero(mascara)\n    mascara[ativos[np.std(anom, axis=0) <= 1e-08]] = False\n    latitudes = np.repeat(sst.lat.values, sst.sizes[\'lon\'])\n    pesos = np.sqrt(np.cos(np.deg2rad(latitudes[mascara]))).astype(np.float64)\n    exigir(int(mascara.sum()) > SST_COMPONENTES and len(origens) > SST_COMPONENTES, \'Insufficient dimensions for PCA.\')\n    matriz = (bruto[:, mascara] - mensal[origens.month - 1][:, mascara]) * pesos\n    exigir(np.isfinite(matriz).all(), \'Invalid PCA matrix.\')\n    with threadpool_limits(limits=1):\n        pca = PCA(n_components=SST_COMPONENTES, svd_solver=\'full\').fit(matriz)\n    return dict(mascara=mascara, clima_mensal=mensal[:, mascara], pesos_area=pesos, centro=pca.mean_, componentes=pca.components_, variancia_explicada=pca.explained_variance_ratio_, contagens_mensais=contagens, origens_treino=origens.values, lat=sst.lat.values.copy(), lon=sst.lon.values.copy())\n\ndef sst_transformar(sst, estado, alvos):\n    alvos = pd.DatetimeIndex(alvos)\n    origens = origem_mensal(alvos, LAG_SST)\n    exigir(np.array_equal(sst.lat, estado[\'lat\']) and np.array_equal(sst.lon, estado[\'lon\']), \'SST grid changed.\')\n    v = sst.sel(time=origens).values.reshape(len(origens), -1).astype(np.float64)[:, estado[\'mascara\']]\n    exigir(np.isfinite(v).all(), \'Missing SST in the training-selected ocean mask; do not impute from the future.\')\n    x = (v - estado[\'clima_mensal\'][origens.month - 1]) * estado[\'pesos_area\'] - estado[\'centro\']\n    with threadpool_limits(limits=1):\n        scores = x @ estado[\'componentes\'].T\n    exigir(scores.shape == (len(alvos), SST_COMPONENTES) and np.isfinite(scores).all(), \'Invalid PCA scores.\')\n    return scores.astype(np.float32)\n\ndef pesquisa_prever_sst(modelo, nome, atmosfera, seas, cfs, ca, clima, cs, cc, datas, scores):\n    datas = pd.DatetimeIndex(datas)\n    exigir((datas > pd.Timestamp(cs.attrs[\'corte\'])).all(), \'Inference overlaps training.\')\n    exigir(scores.shape == (len(datas), 8), \'Misaligned SST scores.\')\n    out = np.empty((len(datas), clima.sizes[\'lat\'], clima.sizes[\'lon\']), dtype=np.float32)\n    expected = (FEATURES_SEAS5 if nome == \'arvores_seas5\' else FEATURES_MULTISSISTEMA) + SST_FEATURES\n    exigir(modelo.feature_name() == expected, \'Incorrect SST feature order.\')\n    for i, t in enumerate(datas):\n        x = np.column_stack([matriz_mes(atmosfera, ca, clima, t), valores_seas5(seas, t), valores_anomalia_seas5(seas, cs, t)]).astype(np.float32)\n        if nome == \'arvores_multissistema\':\n            x = np.column_stack([x, valores_cfsv2(cfs, t), valores_anomalia_cfsv2(cfs, cc, t)]).astype(np.float32)\n        x = np.column_stack([x, np.broadcast_to(scores[i], (len(x), 8))]).astype(np.float32)\n        exigir(x.shape[1] == len(expected) and np.isfinite(x).all(), \'Invalid SST features.\')\n        out[i] = modelo.predict(x).reshape(out.shape[1:])\n    return xr.DataArray(out, dims=(\'time\', \'lat\', \'lon\'), coords=dict(time=datas, lat=clima.lat, lon=clima.lon), attrs=dict(units=\'mm/day\'))\n\ndef pesquisa_diagnosticos(obs, previsoes, bloco):\n    """Complete fields, without pixel subsampling or hiding errors."""\n    linhas = []\n    lat = obs.lat.values\n    regioes = {\'dominio_inteiro\': np.ones(lat.size, dtype=bool), \'lat_menor_que_menos35\': lat < -35, \'lat_menos35_a_menos15\': (lat >= -35) & (lat < -15), \'lat_maior_igual_menos15\': lat >= -15}\n    for modelo, pred in previsoes.items():\n        obs, pred = xr.align(obs, pred, join=\'exact\')\n        for i, t in enumerate(pd.DatetimeIndex(obs.time.values)):\n            erro = pred.values[i].astype(np.float64) - obs.values[i].astype(np.float64)\n            exigir(np.isfinite(erro).all(), \'Nonfinite diagnostic error.\')\n            for regiao, mask in regioes.items():\n                if not mask.any():\n                    continue\n                e = erro[mask]\n                w = np.broadcast_to(np.cos(np.deg2rad(lat[mask]))[:, None], e.shape)\n                linhas.append(dict(modelo=modelo, bloco=bloco, mes_alvo=str(t.date()), ano=t.year, mes=t.month, regiao=regiao, n=e.size, sse=float(np.square(e).sum()), soma_erro=float(e.sum()), soma_erro_absoluto=float(np.abs(e).sum()), sse_area=float((w * np.square(e)).sum()), peso_area=float(w.sum())))\n    return pd.DataFrame(linhas)\n\ndef pesquisa_resumir(mensais, grupos):\n    cols = [\'n\', \'sse\', \'soma_erro\', \'soma_erro_absoluto\', \'sse_area\', \'peso_area\']\n    r = mensais.groupby(grupos, as_index=False)[cols].sum()\n    r[\'rmse\'] = np.sqrt(r.sse / r.n)\n    r[\'mae\'] = r.soma_erro_absoluto / r.n\n    r[\'vies\'] = r.soma_erro / r.n\n    r[\'rmse_area\'] = np.sqrt(r.sse_area / r.peso_area)\n    return r\n\'Candidate lag policy; not evidence of historical publication dates or vintages.\'\nLAG_ATMOSFERA = 4\nLAG_INDICES = 3\nLAG_SST = 2\nLAG_CHUVA_TREINO = 4\nINICIO_COMUM = pd.Timestamp(\'1982-03-01\')\nFEATURES = [v + \'_lag4\' for v in VARIAVEIS] + [v + \'_lag4_anomalia\' for v in VARIAVEIS] + [\'latitude\', \'longitude\', \'mes_alvo_sin\', \'mes_alvo_cos\', \'clima_tp_alvo\'] + [n + \'_lag3_anomalia_fold\' for n in INDICES_NOMES]\nFEATURES_SEAS5 = FEATURES + [FEATURE_SEAS5, FEATURE_SEAS5_ANOM]\nFEATURES_MULTISSISTEMA = FEATURES_SEAS5 + [FEATURE_CFSV2, FEATURE_CFSV2_ANOM]\nSST_FEATURES = [f\'sst_lag2_pc_{i:02d}_treino_fold\' for i in range(1, 9)]\n\ndef origem_mensal(alvos, lag):\n    datas = pd.DatetimeIndex(alvos)\n    exigir(lag >= 1 and datas.equals(datas.to_period(\'M\').to_timestamp()), \'Invalid dates or lag.\')\n    return (datas.to_period(\'M\') - lag).to_timestamp()\n\ndef janela_referencia(corte, anos=30):\n    corte = pd.Timestamp(corte)\n    exigir(corte.day == 1, \'The cutoff must identify a month.\')\n    return pd.date_range(end=corte, periods=12 * anos, freq=\'MS\')\n\ndef calendario_pareado(corte):\n    datas = janela_referencia(corte)\n    datas = datas[datas >= INICIO_COMUM]\n    exigir(24 <= len(datas) <= 360, \'Invalid common window.\')\n    return datas\n\ndef calendario_pares(datas_disponiveis, corte, anos=30):\n    alvos = janela_referencia(corte, anos)\n    origens = origem_mensal(alvos, LAG_ATMOSFERA)\n    disponiveis = pd.DatetimeIndex(datas_disponiveis)\n    exigir(disponiveis.is_unique and origens.isin(disponiveis).all(), \'Missing atmospheric origins.\')\n    return (origens, alvos)\n\n# Monthly rainfall sums and counts support leave-one-out training references.\ndef estatisticas_climatologia(tp, corte, anos=30):\n    datas = janela_referencia(corte, anos)\n    treino = tp.sel(time=datas).transpose(\'time\', \'lat\', \'lon\')\n    exigir(np.isfinite(treino.values).all(), \'Incomplete training rainfall.\')\n    somas = np.stack([treino.values[datas.month == m].sum(axis=0, dtype=np.float64) for m in range(1, 13)])\n    contagens = np.array([(datas.month == m).sum() for m in range(1, 13)], dtype=np.int64)\n    exigir((contagens == anos).all(), \'Climatology requires every calendar month.\')\n    clima = xr.DataArray((somas / contagens[:, None, None]).astype(np.float32), dims=(\'month\', \'lat\', \'lon\'), coords=dict(month=np.arange(1, 13), lat=tp.lat, lon=tp.lon), attrs=dict(units=\'mm/day\', ajuste_inicio=str(datas[0].date()), ajuste_fim=str(datas[-1].date())))\n    return (clima, somas, contagens)\n\ndef ajustar_clima_indices(tabela, origens_atmosfera, corte):\n    alvos = janela_referencia(corte)\n    exigir(pd.DatetimeIndex(origens_atmosfera).equals(origem_mensal(alvos, LAG_ATMOSFERA)), \'Atmospheric reference outside the protocol.\')\n    origens = origem_mensal(alvos, LAG_INDICES)\n    valores = tabela.loc[origens, INDICES_NOMES].to_numpy(dtype=np.float64)\n    exigir(np.isfinite(valores).all(), \'Incomplete training indices.\')\n    return np.stack([valores[origens.month == m].mean(axis=0, dtype=np.float64) for m in range(1, 13)])\n\ndef valores_indices_mes(tabela, clima_indices, destino):\n    origem = origem_mensal([destino], LAG_INDICES)[0]\n    valores = tabela.loc[origem, INDICES_NOMES].to_numpy(dtype=np.float64)\n    exigir(np.isfinite(valores).all(), \'Unavailable index; do not fill.\')\n    return (valores - clima_indices[origem.month - 1]).astype(np.float32)\n\ndef exigir_disponibilidade_real(registros, emitido_em, fontes_necessarias):\n    """Validate supplied availability metadata for a future production gate. Receipt records must identify the actual bytes. This check alone cannot authenticate a historical publication timestamp."""\n    emissao = pd.Timestamp(emitido_em)\n    exigir(emissao.tzinfo is not None, \'Issue time must include a timezone.\')\n    exigir(len(registros) == len(fontes_necessarias), \'Incorrect source count.\')\n    exigir({r.get(\'fonte\') for r in registros} == set(fontes_necessarias), \'Missing or duplicate source.\')\n    for r in registros:\n        for chave in [\'disponivel_em\', \'recebido_em\', \'url\', \'sha256\', \'versao\']:\n            exigir(bool(r.get(chave)), \'Unknown availability: \' + r.get(\'fonte\', \'?\') + \'/\' + chave)\n        recebido, publicado = (pd.Timestamp(r[\'recebido_em\']), pd.Timestamp(r[\'disponivel_em\']))\n        exigir(recebido.tzinfo is not None and publicado.tzinfo is not None, \'Dates must include a timezone.\')\n        exigir(publicado <= recebido <= emissao, \'A source arrived after issue time or its chronology is invalid.\')\n        exigir(len(r[\'sha256\']) == 64 and set(r[\'sha256\']) <= set(\'0123456789abcdef\'), \'Invalid SHA-256.\')\n    return True\n\ndef calendario_emissoes(datas, corte):\n    datas = pd.DatetimeIndex(datas)\n    limite = origem_mensal([datas[0]], LAG_CHUVA_TREINO)[0]\n    exigir(pd.Timestamp(corte) <= limite, \'Training rainfall is too recent for the first issue date.\')\n    return pd.DataFrame(dict(time_alvo=datas, limite_emissao=datas - pd.Timedelta(nanoseconds=1), atmosfera=origem_mensal(datas, LAG_ATMOSFERA), indices=origem_mensal(datas, LAG_INDICES), sst=origem_mensal(datas, LAG_SST), inicializacao_sazonal=origem_mensal(datas, 1), ultimo_alvo_treino=pd.Timestamp(corte), disponibilidade_historica_comprovada=False))\n\ndef auditar_invariancia_defasada(chuva, atmosfera, seas, cfs, sst, corte, datas, clima, ca, cs, cc, pca):\n    """Mutate small copies of real data; values beyond each cutoff must not affect T."""\n    primeiro = pd.Timestamp(datas[0])\n    pontos = np.array([0, 1, chuva.sizes[\'lon\']], dtype=np.int32)\n    ch = chuva.isel(lat=slice(0, 2), lon=slice(0, 2)).copy(deep=True)\n    at = {k: v.isel(lat=slice(0, 2), lon=slice(0, 2)).copy(deep=True) for k, v in atmosfera.items()}\n    alvos_ref = janela_referencia(corte)\n    orig_at = origem_mensal(alvos_ref, LAG_ATMOSFERA)\n    c0, _, _ = estatisticas_climatologia(ch, corte)\n    a0 = ajustar_clima_atmosfera(at, orig_at)\n    a0[\'_clima_indices\'] = ajustar_clima_indices(INDICES_OC, orig_at, corte)\n    x0 = matriz_mes(at, a0, c0, primeiro)\n    ch.values[pd.DatetimeIndex(ch.time.values) > pd.Timestamp(corte)] = 123456\n    c1, _, _ = estatisticas_climatologia(ch, corte)\n    exigir(np.array_equal(c0.values, c1.values), \'Post-cutoff rainfall changed the climatology.\')\n    for v in at.values():\n        v.values[pd.DatetimeIndex(v.time.values) > origem_mensal([primeiro], LAG_ATMOSFERA)[0]] = 7654321\n    a1 = ajustar_clima_atmosfera(at, orig_at)\n    a1[\'_clima_indices\'] = a0[\'_clima_indices\']\n    exigir(all((np.array_equal(a0[k], a1[k]) for k in a0)), \'Future data changed atmospheric climatology.\')\n    exigir(np.array_equal(x0, matriz_mes(at, a1, c1, primeiro)), \'Recent or future atmospheric data changed T.\')\n    ni = INDICES_OC.copy(deep=True)\n    ni.loc[ni.index > origem_mensal([primeiro], LAG_INDICES)[0], :] = 99999\n    ci = ajustar_clima_indices(ni, orig_at, corte)\n    exigir(np.array_equal(ci, a0[\'_clima_indices\']), \'Future indices changed the fit.\')\n    exigir(np.array_equal(valores_indices_mes(ni, ci, primeiro), x0[0, 23:]), \'A future index changed the feature.\')\n    for pre, nome in [(seas, \'SEAS5\'), (cfs, \'CFSv2\')]:\n        pequeno = pre.isel(lat=slice(0, 2), lon=slice(0, 2)).copy(deep=True)\n        antes = ajustar_clima_previsao(pequeno, corte, nome)\n        pequeno.values[pd.DatetimeIndex(pequeno.time.values) > pd.Timestamp(corte)] = 12345\n        exigir(np.array_equal(antes, ajustar_clima_previsao(pequeno, corte, nome)), \'Future data changed seasonal climatology.\')\n    copia = sst.copy(deep=True)\n    copia.values[pd.DatetimeIndex(copia.time.values) > origem_mensal([primeiro], LAG_SST)[0]] = 12345\n    novo = sst_ajustar_pca(copia, calendario_pareado(corte))\n    for k in [\'mascara\', \'clima_mensal\', \'centro\', \'componentes\', \'origens_treino\']:\n        exigir(np.array_equal(novo[k], pca[k]), \'Future SST changed PCA: \' + k)\n    exigir(np.array_equal(sst_transformar(copia, novo, [primeiro]), sst_transformar(sst, pca, [primeiro])), \'SST more recent than T-2 changed the prediction for T.\')\n    return dict(chuva_pos_corte_invariante=True, atmosfera_apos_Tmenos4_invariante=True, indices_apos_Tmenos3_invariantes=True, sst_apos_Tmenos2_invariante=True, climas_sazonais_somente_treino=True, publicacao_historica_comprovada=False)\n', 'research/lagged_sources/ocean_indices.csv': 'time_origem,nino34,nino12,tna,tsa\n1976-07-01,0.150000000,2.020000000,-0.640000000,-0.930000000\n1976-08-01,0.310000000,1.810000000,-0.270000000,-0.680000000\n1976-09-01,0.570000000,1.230000000,-0.240000000,-0.470000000\n1976-10-01,0.860000000,0.760000000,-0.250000000,-0.580000000\n1976-11-01,0.880000000,0.440000000,-0.270000000,-0.630000000\n1976-12-01,0.640000000,0.250000000,-0.600000000,-0.470000000\n1977-01-01,0.840000000,0.290000000,-0.400000000,-0.530000000\n1977-02-01,0.550000000,-0.140000000,-0.400000000,-0.270000000\n1977-03-01,0.400000000,-0.380000000,-0.270000000,-0.340000000\n1977-04-01,-0.250000000,-0.680000000,-0.250000000,-0.340000000\n1977-05-01,-0.030000000,-0.450000000,-0.280000000,-0.390000000\n1977-06-01,0.360000000,0.080000000,-0.200000000,-0.410000000\n1977-07-01,0.450000000,0.030000000,-0.420000000,-0.400000000\n1977-08-01,0.250000000,-0.520000000,-0.620000000,-0.190000000\n1977-09-01,0.440000000,-0.500000000,-0.610000000,-0.120000000\n1977-10-01,0.890000000,-0.190000000,-0.660000000,-0.310000000\n1977-11-01,1.030000000,-0.290000000,-0.570000000,-0.270000000\n1977-12-01,1.090000000,-0.560000000,-0.440000000,-0.340000000\n1978-01-01,0.720000000,-0.070000000,-0.130000000,-0.520000000\n1978-02-01,0.510000000,-0.350000000,0.020000000,-0.560000000\n1978-03-01,0.060000000,-0.770000000,0.090000000,-0.800000000\n1978-04-01,-0.370000000,-0.370000000,0.100000000,-0.960000000\n1978-05-01,-0.500000000,-0.540000000,-0.130000000,-0.990000000\n1978-06-01,-0.450000000,-0.860000000,-0.390000000,-0.780000000\n1978-07-01,-0.480000000,-0.400000000,-0.600000000,-0.330000000\n1978-08-01,-0.620000000,-0.640000000,-0.660000000,-0.400000000\n1978-09-01,-0.600000000,-0.300000000,-0.590000000,-0.500000000\n1978-10-01,-0.340000000,-0.430000000,-0.350000000,-0.610000000\n1978-11-01,-0.150000000,0.040000000,-0.300000000,-0.610000000\n1978-12-01,0.070000000,0.230000000,-0.480000000,-0.430000000\n1979-01-01,0.050000000,0.230000000,-0.080000000,-0.490000000\n1979-02-01,0.090000000,-0.580000000,0.100000000,-0.510000000\n1979-03-01,0.130000000,-0.250000000,-0.010000000,-0.650000000\n1979-04-01,0.170000000,0.510000000,0.140000000,-0.300000000\n1979-05-01,-0.090000000,0.270000000,0.150000000,-0.270000000\n1979-06-01,0.040000000,0.760000000,0.180000000,-0.210000000\n1979-07-01,-0.200000000,0.450000000,0.010000000,-0.120000000\n1979-08-01,0.210000000,0.500000000,-0.250000000,0.070000000\n1979-09-01,0.480000000,1.020000000,-0.270000000,0.140000000\n1979-10-01,0.320000000,0.670000000,-0.320000000,-0.170000000\n1979-11-01,0.350000000,0.120000000,-0.200000000,-0.120000000\n1979-12-01,0.690000000,-0.340000000,-0.030000000,-0.280000000\n1980-01-01,0.550000000,-0.100000000,0.150000000,-0.480000000\n1980-02-01,0.330000000,-0.250000000,0.230000000,-0.610000000\n1980-03-01,0.090000000,0.020000000,0.020000000,-0.610000000\n1980-04-01,0.200000000,-0.000000000,0.400000000,-0.420000000\n1980-05-01,0.130000000,0.250000000,0.500000000,-0.380000000\n1980-06-01,0.370000000,0.210000000,0.230000000,-0.390000000\n1980-07-01,0.020000000,-0.440000000,-0.080000000,-0.310000000\n1980-08-01,-0.230000000,-0.440000000,-0.040000000,-0.160000000\n1980-09-01,-0.120000000,-0.110000000,-0.170000000,-0.040000000\n1980-10-01,-0.110000000,-0.520000000,-0.380000000,-0.250000000\n1980-11-01,0.100000000,-0.680000000,-0.280000000,-0.430000000\n1980-12-01,0.360000000,-1.130000000,-0.310000000,-0.860000000\n1981-01-01,-0.460000000,-1.290000000,0.030000000,-0.610000000\n1981-02-01,-0.450000000,-0.830000000,-0.010000000,-0.730000000\n1981-03-01,-0.020000000,-0.190000000,0.450000000,-0.990000000\n1981-04-01,-0.170000000,-0.310000000,0.180000000,-0.830000000\n1981-05-01,-0.110000000,-0.220000000,0.160000000,-0.750000000\n1981-06-01,-0.150000000,-0.060000000,-0.060000000,-0.530000000\n1981-07-01,-0.430000000,-0.560000000,-0.120000000,-0.480000000\n1981-08-01,-0.180000000,-0.550000000,-0.240000000,-0.070000000\n1981-09-01,-0.070000000,-0.240000000,-0.230000000,0.050000000\n1981-10-01,-0.030000000,-0.520000000,-0.390000000,0.120000000\n1981-11-01,-0.270000000,-0.550000000,-0.180000000,0.080000000\n1981-12-01,0.070000000,-0.170000000,-0.020000000,-0.160000000\n1982-01-01,-0.040000000,-0.120000000,-0.060000000,-0.300000000\n1982-02-01,-0.130000000,-0.350000000,-0.240000000,-0.470000000\n1982-03-01,-0.020000000,-0.720000000,-0.230000000,-0.560000000\n1982-04-01,0.240000000,-0.510000000,-0.320000000,-0.660000000\n1982-05-01,0.650000000,0.080000000,-0.240000000,-0.710000000\n1982-06-01,0.920000000,0.270000000,-0.400000000,-0.760000000\n1982-07-01,0.640000000,0.600000000,-0.540000000,-0.700000000\n1982-08-01,0.930000000,1.090000000,-0.640000000,-0.770000000\n1982-09-01,1.390000000,2.010000000,-0.660000000,-0.460000000\n1982-10-01,2.000000000,2.280000000,-0.770000000,-0.330000000\n1982-11-01,2.070000000,3.080000000,-0.770000000,-0.460000000\n1982-12-01,2.330000000,3.200000000,-0.790000000,-0.430000000\n1983-01-01,2.430000000,2.730000000,-0.610000000,-0.010000000\n1983-02-01,2.220000000,2.030000000,-0.190000000,0.110000000\n1983-03-01,1.690000000,2.340000000,0.280000000,-0.480000000\n1983-04-01,1.120000000,3.360000000,0.290000000,-0.560000000\n1983-05-01,1.120000000,4.100000000,-0.030000000,-0.800000000\n1983-06-01,0.620000000,4.560000000,0.100000000,-0.760000000\n1983-07-01,-0.110000000,3.930000000,-0.050000000,-0.560000000\n1983-08-01,-0.130000000,3.110000000,-0.330000000,-0.280000000\n1983-09-01,-0.500000000,2.070000000,-0.560000000,-0.110000000\n1983-10-01,-1.030000000,1.020000000,-0.580000000,-0.310000000\n1983-11-01,-1.130000000,0.560000000,-0.260000000,-0.120000000\n1983-12-01,-0.950000000,0.220000000,-0.060000000,0.120000000\n1984-01-01,-0.800000000,-0.290000000,-0.300000000,0.350000000\n1984-02-01,-0.530000000,-0.770000000,-0.380000000,0.380000000\n1984-03-01,-0.480000000,-0.410000000,-0.250000000,0.440000000\n1984-04-01,-0.600000000,-0.250000000,-0.340000000,0.190000000\n1984-05-01,-0.640000000,-1.110000000,-0.360000000,-0.080000000\n1984-06-01,-0.900000000,-0.920000000,-0.610000000,0.140000000\n1984-07-01,-0.400000000,-0.540000000,-0.630000000,0.290000000\n1984-08-01,-0.400000000,-0.520000000,-0.620000000,0.570000000\n1984-09-01,-0.350000000,-0.360000000,-0.620000000,0.690000000\n1984-10-01,-0.830000000,-0.610000000,-0.920000000,0.360000000\n1984-11-01,-1.180000000,-0.180000000,-0.770000000,0.130000000\n1984-12-01,-1.530000000,-0.500000000,-0.790000000,-0.010000000\n1985-01-01,-0.890000000,-0.770000000,-0.680000000,-0.020000000\n1985-02-01,-0.910000000,-1.130000000,-0.550000000,0.250000000\n1985-03-01,-0.960000000,-0.590000000,-0.770000000,-0.210000000\n1985-04-01,-0.980000000,-1.380000000,-0.810000000,-0.160000000\n1985-05-01,-0.900000000,-1.410000000,-0.910000000,-0.100000000\n1985-06-01,-0.760000000,-1.250000000,-0.540000000,-0.270000000\n1985-07-01,-0.610000000,-1.040000000,-0.320000000,0.020000000\n1985-08-01,-0.490000000,-1.020000000,-0.420000000,-0.100000000\n1985-09-01,-0.570000000,-0.830000000,-0.540000000,-0.130000000\n1985-10-01,-0.640000000,-0.820000000,-0.570000000,0.050000000\n1985-11-01,-0.430000000,-0.920000000,-0.390000000,0.060000000\n1985-12-01,-0.400000000,-0.750000000,-0.540000000,0.030000000\n1986-01-01,-0.790000000,-0.590000000,-0.860000000,-0.140000000\n1986-02-01,-0.640000000,-0.350000000,-0.800000000,0.200000000\n1986-03-01,-0.510000000,-0.840000000,-0.760000000,0.040000000\n1986-04-01,-0.410000000,-0.860000000,-0.800000000,-0.030000000\n1986-05-01,-0.470000000,-0.820000000,-0.740000000,-0.080000000\n1986-06-01,-0.080000000,-0.900000000,-0.840000000,0.100000000\n1986-07-01,0.160000000,-0.140000000,-0.650000000,-0.060000000\n1986-08-01,0.240000000,0.170000000,-0.740000000,0.010000000\n1986-09-01,0.590000000,-0.130000000,-0.340000000,0.160000000\n1986-10-01,0.900000000,0.140000000,-0.630000000,-0.080000000\n1986-11-01,1.040000000,0.280000000,-0.910000000,-0.350000000\n1986-12-01,0.980000000,0.530000000,-0.980000000,-0.240000000\n1987-01-01,1.190000000,0.770000000,-0.570000000,-0.320000000\n1987-02-01,1.170000000,1.020000000,-0.350000000,-0.190000000\n1987-03-01,1.240000000,1.150000000,0.060000000,-0.180000000\n1987-04-01,0.890000000,1.510000000,0.170000000,-0.270000000\n1987-05-01,0.910000000,1.350000000,0.170000000,-0.060000000\n1987-06-01,1.240000000,1.060000000,0.370000000,0.110000000\n1987-07-01,1.340000000,1.190000000,0.210000000,0.250000000\n1987-08-01,1.480000000,0.910000000,0.220000000,0.240000000\n1987-09-01,1.530000000,1.310000000,0.020000000,0.170000000\n1987-10-01,1.360000000,1.500000000,0.020000000,0.190000000\n1987-11-01,1.290000000,0.900000000,0.050000000,0.450000000\n1987-12-01,1.050000000,0.890000000,0.220000000,0.130000000\n1988-01-01,0.690000000,0.090000000,-0.090000000,0.300000000\n1988-02-01,0.350000000,-0.270000000,-0.200000000,0.390000000\n1988-03-01,0.290000000,-0.520000000,0.120000000,0.430000000\n1988-04-01,-0.490000000,-0.900000000,0.220000000,0.280000000\n1988-05-01,-1.050000000,-1.080000000,0.080000000,0.370000000\n1988-06-01,-1.460000000,-1.960000000,0.100000000,0.420000000\n1988-07-01,-1.540000000,-1.540000000,-0.000000000,0.200000000\n1988-08-01,-1.440000000,-1.590000000,-0.310000000,0.300000000\n1988-09-01,-1.330000000,-1.430000000,-0.320000000,0.160000000\n1988-10-01,-2.090000000,-1.260000000,-0.630000000,0.150000000\n1988-11-01,-2.180000000,-0.650000000,-0.540000000,-0.050000000\n1988-12-01,-1.980000000,-0.970000000,-0.520000000,-0.070000000\n1989-01-01,-1.950000000,-0.280000000,-0.670000000,0.080000000\n1989-02-01,-1.370000000,0.050000000,-0.660000000,-0.040000000\n1989-03-01,-1.330000000,-0.450000000,-0.820000000,0.000000000\n1989-04-01,-1.110000000,-0.490000000,-1.040000000,-0.020000000\n1989-05-01,-0.800000000,-1.160000000,-0.860000000,-0.010000000\n1989-06-01,-0.640000000,-0.730000000,-0.420000000,0.000000000\n1989-07-01,-0.470000000,-0.440000000,-0.030000000,0.000000000\n1989-08-01,-0.580000000,-0.460000000,0.050000000,0.190000000\n1989-09-01,-0.380000000,-0.470000000,-0.320000000,0.060000000\n1989-10-01,-0.400000000,-0.290000000,-0.300000000,0.060000000\n1989-11-01,-0.300000000,-0.380000000,-0.350000000,-0.240000000\n1989-12-01,-0.130000000,-0.430000000,-0.070000000,-0.510000000\n1990-01-01,0.020000000,-0.330000000,-0.330000000,-0.680000000\n1990-02-01,0.360000000,-0.270000000,-0.280000000,-0.140000000\n1990-03-01,0.200000000,-0.410000000,-0.200000000,-0.030000000\n1990-04-01,0.260000000,-0.330000000,-0.100000000,0.040000000\n1990-05-01,0.320000000,-0.230000000,-0.070000000,-0.400000000\n1990-06-01,0.000000000,-0.050000000,-0.270000000,-0.230000000\n1990-07-01,0.150000000,-0.870000000,-0.220000000,-0.370000000\n1990-08-01,0.170000000,-0.610000000,-0.040000000,-0.330000000\n1990-09-01,0.120000000,-0.280000000,0.020000000,-0.220000000\n1990-10-01,0.130000000,-0.730000000,-0.010000000,-0.130000000\n1990-11-01,0.080000000,-0.550000000,0.020000000,-0.010000000\n1990-12-01,0.320000000,-0.440000000,-0.120000000,-0.040000000\n1991-01-01,0.510000000,-0.090000000,-0.390000000,-0.210000000\n1991-02-01,0.320000000,-0.100000000,-0.290000000,-0.200000000\n1991-03-01,0.100000000,0.000000000,-0.370000000,-0.210000000\n1991-04-01,0.210000000,-0.680000000,-0.540000000,-0.000000000\n1991-05-01,0.440000000,-0.010000000,-0.620000000,0.140000000\n1991-06-01,0.650000000,0.240000000,-0.670000000,0.130000000\n1991-07-01,0.620000000,0.600000000,-0.670000000,-0.200000000\n1991-08-01,0.430000000,0.000000000,-0.620000000,-0.520000000\n1991-09-01,0.220000000,0.160000000,-0.520000000,-0.510000000\n1991-10-01,0.910000000,0.420000000,-0.530000000,-0.590000000\n1991-11-01,1.140000000,0.640000000,-0.460000000,-0.520000000\n1991-12-01,1.620000000,0.450000000,-0.470000000,-0.340000000\n1992-01-01,1.610000000,0.360000000,-0.240000000,-0.610000000\n1992-02-01,1.620000000,0.440000000,-0.140000000,-0.530000000\n1992-03-01,1.500000000,0.920000000,-0.340000000,-0.740000000\n1992-04-01,1.410000000,1.720000000,-0.440000000,-0.880000000\n1992-05-01,1.270000000,1.830000000,-0.390000000,-1.010000000\n1992-06-01,0.530000000,0.710000000,-0.360000000,-1.110000000\n1992-07-01,0.270000000,-0.050000000,-0.290000000,-0.860000000\n1992-08-01,-0.150000000,-0.240000000,-0.560000000,-0.860000000\n1992-09-01,-0.140000000,-0.500000000,-0.580000000,-0.700000000\n1992-10-01,-0.350000000,-0.260000000,-0.480000000,-0.740000000\n1992-11-01,-0.140000000,-0.250000000,-0.610000000,-0.970000000\n1992-12-01,0.040000000,-0.310000000,-0.520000000,-0.630000000\n1993-01-01,0.280000000,0.030000000,-0.500000000,-0.590000000\n1993-02-01,0.420000000,0.200000000,-0.430000000,-0.390000000\n1993-03-01,0.470000000,0.530000000,-0.380000000,-0.440000000\n1993-04-01,0.920000000,0.830000000,-0.220000000,-0.110000000\n1993-05-01,0.930000000,1.130000000,-0.250000000,-0.320000000\n1993-06-01,0.640000000,0.880000000,-0.180000000,-0.340000000\n1993-07-01,0.330000000,0.320000000,-0.440000000,-0.210000000\n1993-08-01,0.160000000,0.210000000,-0.450000000,-0.160000000\n1993-09-01,0.260000000,0.120000000,-0.500000000,0.300000000\n1993-10-01,0.360000000,0.320000000,-0.560000000,0.470000000\n1993-11-01,0.280000000,-0.230000000,-0.650000000,0.570000000\n1993-12-01,0.190000000,-0.290000000,-0.700000000,0.240000000\n1994-01-01,0.040000000,-0.070000000,-0.830000000,-0.010000000\n1994-02-01,-0.160000000,-0.330000000,-0.780000000,-0.140000000\n1994-03-01,-0.060000000,-1.130000000,-0.760000000,-0.100000000\n1994-04-01,0.030000000,-1.220000000,-0.620000000,-0.130000000\n1994-05-01,0.140000000,-0.910000000,-0.700000000,-0.230000000\n1994-06-01,0.270000000,-0.590000000,-0.670000000,-0.490000000\n1994-07-01,0.160000000,-0.820000000,-0.670000000,-0.520000000\n1994-08-01,0.520000000,-0.900000000,-0.630000000,-0.370000000\n1994-09-01,0.400000000,-0.050000000,-0.620000000,-0.300000000\n1994-10-01,0.900000000,0.650000000,-0.500000000,-0.270000000\n1994-11-01,1.140000000,0.590000000,-0.340000000,-0.110000000\n1994-12-01,1.210000000,0.680000000,-0.420000000,-0.070000000\n1995-01-01,1.100000000,0.910000000,-0.300000000,0.070000000\n1995-02-01,0.870000000,0.240000000,-0.220000000,0.340000000\n1995-03-01,0.490000000,-0.570000000,-0.210000000,0.390000000\n1995-04-01,0.270000000,-0.990000000,-0.040000000,0.290000000\n1995-05-01,0.030000000,-1.010000000,0.100000000,0.240000000\n1995-06-01,0.080000000,-0.530000000,0.230000000,0.330000000\n1995-07-01,0.030000000,-0.250000000,0.310000000,0.090000000\n1995-08-01,-0.380000000,-0.520000000,0.250000000,-0.260000000\n1995-09-01,-0.570000000,-0.280000000,0.030000000,-0.190000000\n1995-10-01,-0.720000000,-0.410000000,-0.060000000,-0.270000000\n1995-11-01,-0.780000000,-0.260000000,0.200000000,-0.060000000\n1995-12-01,-0.720000000,-0.690000000,0.170000000,-0.260000000\n1996-01-01,-0.650000000,-0.610000000,0.490000000,-0.150000000\n1996-02-01,-0.660000000,-0.320000000,0.220000000,0.160000000\n1996-03-01,-0.480000000,-0.070000000,0.070000000,0.120000000\n1996-04-01,-0.340000000,-1.280000000,0.150000000,0.340000000\n1996-05-01,-0.400000000,-0.810000000,0.160000000,0.430000000\n1996-06-01,-0.120000000,-1.070000000,-0.060000000,0.560000000\n1996-07-01,-0.140000000,-0.980000000,-0.150000000,0.410000000\n1996-08-01,-0.300000000,-0.460000000,-0.360000000,0.150000000\n1996-09-01,-0.340000000,-0.530000000,-0.220000000,-0.010000000\n1996-10-01,-0.280000000,-0.800000000,-0.340000000,-0.100000000\n1996-11-01,-0.300000000,-0.790000000,-0.280000000,-0.060000000\n1996-12-01,-0.430000000,-1.030000000,-0.150000000,-0.330000000\n1997-01-01,-0.430000000,-0.560000000,0.060000000,-0.550000000\n1997-02-01,-0.240000000,-0.030000000,0.160000000,-0.600000000\n1997-03-01,-0.060000000,0.820000000,-0.000000000,-0.820000000\n1997-04-01,0.340000000,0.980000000,0.120000000,-0.810000000\n1997-05-01,0.870000000,2.160000000,0.230000000,-0.890000000\n1997-06-01,1.150000000,3.170000000,0.290000000,-0.980000000\n1997-07-01,1.600000000,3.630000000,0.060000000,-0.530000000\n1997-08-01,1.940000000,3.980000000,-0.170000000,-0.290000000\n1997-09-01,2.100000000,4.050000000,-0.120000000,0.060000000\n1997-10-01,2.290000000,4.000000000,0.160000000,0.470000000\n1997-11-01,2.420000000,4.120000000,0.270000000,0.600000000\n1997-12-01,2.300000000,4.270000000,0.150000000,0.760000000\n1998-01-01,2.420000000,3.550000000,0.190000000,0.380000000\n1998-02-01,2.080000000,2.880000000,0.590000000,0.470000000\n1998-03-01,1.490000000,2.880000000,0.590000000,0.340000000\n1998-04-01,0.900000000,3.000000000,0.430000000,0.230000000\n1998-05-01,0.680000000,3.010000000,0.390000000,0.200000000\n1998-06-01,-0.390000000,2.000000000,0.420000000,0.350000000\n1998-07-01,-0.730000000,1.310000000,0.290000000,0.590000000\n1998-08-01,-0.830000000,0.940000000,0.330000000,0.400000000\n1998-09-01,-0.820000000,0.310000000,0.130000000,0.260000000\n1998-10-01,-1.190000000,0.120000000,0.070000000,-0.010000000\n1998-11-01,-1.230000000,-0.210000000,0.120000000,0.140000000\n1998-12-01,-1.510000000,-0.140000000,0.000000000,-0.150000000\n1999-01-01,-1.530000000,-0.230000000,-0.350000000,-0.200000000\n1999-02-01,-1.410000000,0.050000000,-0.380000000,-0.220000000\n1999-03-01,-0.920000000,0.300000000,-0.380000000,-0.020000000\n1999-04-01,-0.810000000,-0.780000000,-0.280000000,0.140000000\n1999-05-01,-0.870000000,-0.430000000,-0.090000000,0.180000000\n1999-06-01,-0.950000000,-0.550000000,-0.100000000,0.180000000\n1999-07-01,-0.840000000,-0.650000000,-0.100000000,0.380000000\n1999-08-01,-0.980000000,-0.510000000,0.010000000,0.350000000\n1999-09-01,-0.840000000,-0.970000000,0.010000000,0.090000000\n1999-10-01,-1.030000000,-0.760000000,-0.180000000,0.050000000\n1999-11-01,-1.410000000,-1.110000000,-0.180000000,0.150000000\n1999-12-01,-1.540000000,-1.060000000,-0.020000000,0.100000000\n2000-01-01,-1.790000000,-0.890000000,-0.420000000,-0.120000000\n2000-02-01,-1.530000000,-0.470000000,-0.280000000,0.120000000\n2000-03-01,-1.260000000,-0.560000000,-0.090000000,0.050000000\n2000-04-01,-0.800000000,0.460000000,-0.170000000,0.060000000\n2000-05-01,-0.800000000,-0.290000000,-0.270000000,-0.050000000\n2000-06-01,-0.750000000,-0.650000000,-0.400000000,-0.110000000\n2000-07-01,-0.570000000,-0.570000000,-0.270000000,-0.100000000\n2000-08-01,-0.360000000,-0.590000000,-0.190000000,-0.000000000\n2000-09-01,-0.390000000,-0.340000000,-0.360000000,-0.130000000\n2000-10-01,-0.550000000,-0.490000000,-0.420000000,-0.120000000\n2000-11-01,-0.750000000,-0.700000000,-0.430000000,-0.110000000\n2000-12-01,-0.920000000,-0.550000000,-0.490000000,-0.270000000\n2001-01-01,-0.880000000,-0.760000000,-0.380000000,-0.170000000\n2001-02-01,-0.630000000,-0.330000000,-0.320000000,-0.100000000\n2001-03-01,-0.480000000,0.350000000,-0.010000000,-0.060000000\n2001-04-01,-0.300000000,0.420000000,-0.110000000,0.070000000\n2001-05-01,-0.300000000,-0.640000000,-0.140000000,0.120000000\n2001-06-01,-0.110000000,-0.860000000,-0.010000000,0.040000000\n2001-07-01,0.010000000,-0.700000000,0.020000000,-0.110000000\n2001-08-01,-0.070000000,-0.770000000,0.000000000,-0.060000000\n2001-09-01,-0.280000000,-1.200000000,0.130000000,-0.040000000\n2001-10-01,-0.260000000,-1.140000000,0.060000000,-0.030000000\n2001-11-01,-0.280000000,-0.940000000,0.310000000,-0.020000000\n2001-12-01,-0.460000000,-1.120000000,0.370000000,-0.250000000\n2002-01-01,-0.140000000,-0.940000000,0.530000000,-0.190000000\n2002-02-01,0.000000000,-0.300000000,0.350000000,-0.330000000\n2002-03-01,0.110000000,0.620000000,0.210000000,-0.060000000\n2002-04-01,0.140000000,0.330000000,-0.180000000,-0.000000000\n2002-05-01,0.210000000,0.180000000,-0.400000000,0.150000000\n2002-06-01,0.680000000,-0.400000000,-0.380000000,0.020000000\n2002-07-01,0.570000000,-0.440000000,-0.330000000,0.220000000\n2002-08-01,0.700000000,-0.260000000,-0.380000000,-0.190000000\n2002-09-01,0.820000000,0.040000000,-0.310000000,-0.230000000\n2002-10-01,1.160000000,0.570000000,-0.070000000,-0.090000000\n2002-11-01,1.410000000,0.780000000,-0.090000000,-0.250000000\n2002-12-01,1.410000000,0.690000000,-0.090000000,0.080000000\n2003-01-01,0.980000000,0.100000000,0.020000000,0.220000000\n2003-02-01,0.640000000,-0.210000000,-0.170000000,0.300000000\n2003-03-01,0.480000000,-0.490000000,-0.110000000,0.280000000\n2003-04-01,-0.030000000,-0.900000000,-0.150000000,0.110000000\n2003-05-01,-0.520000000,-1.350000000,-0.140000000,0.020000000\n2003-06-01,-0.190000000,-0.910000000,-0.020000000,-0.210000000\n2003-07-01,0.140000000,-0.580000000,0.090000000,0.140000000\n2003-08-01,0.050000000,0.080000000,0.260000000,0.300000000\n2003-09-01,0.150000000,-0.430000000,0.280000000,0.340000000\n2003-10-01,0.460000000,-0.070000000,0.380000000,0.310000000\n2003-11-01,0.390000000,0.290000000,0.170000000,0.360000000\n2003-12-01,0.320000000,0.310000000,0.200000000,0.300000000\n2004-01-01,0.260000000,-0.030000000,0.250000000,0.070000000\n2004-02-01,0.170000000,-0.200000000,0.340000000,-0.050000000\n2004-03-01,-0.100000000,-0.340000000,0.200000000,-0.360000000\n2004-04-01,0.060000000,-0.450000000,0.300000000,-0.570000000\n2004-05-01,0.100000000,-0.960000000,-0.030000000,-0.470000000\n2004-06-01,0.140000000,-0.690000000,0.020000000,-0.390000000\n2004-07-01,0.410000000,-0.700000000,0.180000000,-0.180000000\n2004-08-01,0.660000000,-0.440000000,0.350000000,0.020000000\n2004-09-01,0.670000000,-0.100000000,0.210000000,0.180000000\n2004-10-01,0.730000000,0.110000000,0.200000000,-0.080000000\n2004-11-01,0.620000000,0.390000000,0.320000000,0.060000000\n2004-12-01,0.710000000,0.270000000,0.240000000,0.170000000\n2005-01-01,0.560000000,0.030000000,0.220000000,0.280000000\n2005-02-01,0.260000000,-0.610000000,0.200000000,0.220000000\n2005-03-01,0.280000000,-1.150000000,0.530000000,0.260000000\n2005-04-01,0.280000000,-0.360000000,0.650000000,-0.020000000\n2005-05-01,0.300000000,0.220000000,0.810000000,-0.400000000\n2005-06-01,0.220000000,-0.150000000,0.670000000,-0.570000000\n2005-07-01,-0.010000000,-0.060000000,0.570000000,-0.570000000\n2005-08-01,-0.040000000,-0.300000000,0.410000000,-0.190000000\n2005-09-01,-0.080000000,-0.610000000,0.350000000,-0.230000000\n2005-10-01,-0.150000000,-1.050000000,0.120000000,-0.220000000\n2005-11-01,-0.440000000,-1.410000000,0.090000000,-0.260000000\n2005-12-01,-0.750000000,-1.190000000,0.260000000,-0.290000000\n2006-01-01,-0.980000000,-0.690000000,0.160000000,-0.300000000\n2006-02-01,-0.710000000,0.330000000,0.010000000,-0.040000000\n2006-03-01,-0.730000000,-0.270000000,0.020000000,0.060000000\n2006-04-01,-0.300000000,-0.890000000,0.260000000,-0.000000000\n2006-05-01,-0.110000000,-0.390000000,0.230000000,-0.110000000\n2006-06-01,0.090000000,-0.190000000,0.280000000,0.100000000\n2006-07-01,0.030000000,0.180000000,0.090000000,0.050000000\n2006-08-01,0.370000000,0.510000000,0.200000000,0.130000000\n2006-09-01,0.620000000,0.780000000,0.310000000,0.130000000\n2006-10-01,0.760000000,1.200000000,0.310000000,0.010000000\n2006-11-01,0.980000000,0.910000000,0.400000000,-0.100000000\n2006-12-01,1.100000000,0.620000000,0.180000000,0.050000000\n2007-01-01,0.590000000,0.570000000,0.210000000,-0.090000000\n2007-02-01,0.120000000,0.060000000,0.380000000,-0.060000000\n2007-03-01,-0.150000000,-0.650000000,0.170000000,-0.310000000\n2007-04-01,-0.160000000,-1.050000000,0.240000000,-0.070000000\n2007-05-01,-0.390000000,-1.470000000,0.010000000,-0.060000000\n2007-06-01,-0.160000000,-1.400000000,-0.080000000,0.120000000\n2007-07-01,-0.370000000,-0.940000000,-0.180000000,-0.020000000\n2007-08-01,-0.570000000,-1.360000000,-0.330000000,-0.040000000\n2007-09-01,-1.040000000,-1.340000000,-0.270000000,0.120000000\n2007-10-01,-1.410000000,-1.650000000,-0.190000000,-0.110000000\n2007-11-01,-1.580000000,-1.950000000,-0.070000000,-0.080000000\n2007-12-01,-1.610000000,-1.630000000,-0.210000000,-0.060000000\n2008-01-01,-1.790000000,-0.800000000,-0.100000000,-0.140000000\n2008-02-01,-1.700000000,-0.160000000,-0.000000000,0.090000000\n2008-03-01,-1.170000000,0.330000000,0.120000000,0.250000000\n2008-04-01,-0.890000000,0.500000000,-0.060000000,0.330000000\n2008-05-01,-0.640000000,-0.320000000,0.070000000,0.470000000\n2008-06-01,-0.440000000,0.260000000,0.150000000,0.390000000\n2008-07-01,-0.040000000,0.470000000,0.170000000,0.400000000\n2008-08-01,-0.040000000,0.670000000,0.120000000,0.310000000\n2008-09-01,-0.280000000,0.320000000,0.200000000,0.130000000\n2008-10-01,-0.300000000,-0.170000000,0.010000000,0.050000000\n2008-11-01,-0.370000000,-0.210000000,-0.040000000,0.130000000\n2008-12-01,-0.900000000,-0.330000000,-0.030000000,0.150000000\n2009-01-01,-1.000000000,-0.170000000,-0.170000000,0.110000000\n2009-02-01,-0.710000000,-0.310000000,-0.550000000,0.080000000\n2009-03-01,-0.720000000,-0.850000000,-0.490000000,0.200000000\n2009-04-01,-0.250000000,0.000000000,-0.470000000,0.210000000\n2009-05-01,0.170000000,0.170000000,-0.430000000,0.440000000\n2009-06-01,0.490000000,0.540000000,-0.190000000,0.280000000\n2009-07-01,0.690000000,0.700000000,0.170000000,0.140000000\n2009-08-01,0.620000000,0.660000000,0.070000000,0.040000000\n2009-09-01,0.680000000,0.220000000,-0.030000000,0.040000000\n2009-10-01,0.960000000,0.220000000,0.200000000,0.230000000\n2009-11-01,1.490000000,0.140000000,0.200000000,0.110000000\n2009-12-01,1.810000000,0.490000000,0.250000000,0.430000000\n2010-01-01,1.430000000,0.360000000,0.550000000,0.460000000\n2010-02-01,1.180000000,0.040000000,0.900000000,0.410000000\n2010-03-01,1.070000000,-0.040000000,1.120000000,0.540000000\n2010-04-01,0.560000000,0.510000000,1.250000000,0.370000000\n2010-05-01,-0.150000000,0.390000000,1.150000000,0.270000000\n2010-06-01,-0.620000000,0.140000000,0.870000000,0.540000000\n2010-07-01,-0.890000000,-1.050000000,0.740000000,0.350000000\n2010-08-01,-1.330000000,-1.230000000,0.630000000,0.290000000\n2010-09-01,-1.560000000,-1.290000000,0.370000000,0.080000000\n2010-10-01,-1.650000000,-1.530000000,0.350000000,0.210000000\n2010-11-01,-1.570000000,-1.390000000,0.390000000,0.220000000\n2010-12-01,-1.630000000,-1.030000000,0.670000000,0.150000000\n2011-01-01,-1.700000000,-0.610000000,0.730000000,0.120000000\n2011-02-01,-1.260000000,0.070000000,0.350000000,0.320000000\n2011-03-01,-0.980000000,-0.430000000,0.250000000,0.380000000\n2011-04-01,-0.740000000,-0.050000000,0.170000000,0.180000000\n2011-05-01,-0.530000000,0.250000000,0.200000000,-0.040000000\n2011-06-01,-0.250000000,0.360000000,0.350000000,-0.110000000\n2011-07-01,-0.230000000,0.270000000,0.150000000,0.020000000\n2011-08-01,-0.660000000,-0.200000000,0.150000000,0.000000000\n2011-09-01,-0.760000000,-1.010000000,-0.010000000,0.090000000\n2011-10-01,-0.930000000,-0.710000000,0.020000000,0.060000000\n2011-11-01,-1.090000000,-0.800000000,-0.100000000,-0.230000000\n2011-12-01,-1.050000000,-0.680000000,-0.060000000,-0.740000000\n2012-01-01,-0.930000000,-0.390000000,-0.090000000,-0.660000000\n2012-02-01,-0.610000000,0.400000000,-0.190000000,-0.780000000\n2012-03-01,-0.480000000,-0.020000000,-0.410000000,-0.430000000\n2012-04-01,-0.290000000,0.480000000,-0.330000000,-0.640000000\n2012-05-01,-0.180000000,0.430000000,-0.110000000,-0.380000000\n2012-06-01,0.140000000,0.990000000,0.070000000,-0.270000000\n2012-07-01,0.440000000,0.820000000,0.020000000,-0.290000000\n2012-08-01,0.660000000,0.270000000,0.190000000,-0.280000000\n2012-09-01,0.440000000,0.040000000,0.340000000,-0.130000000\n2012-10-01,0.230000000,-0.390000000,0.350000000,-0.190000000\n2012-11-01,0.330000000,-0.460000000,0.440000000,-0.160000000\n2012-12-01,-0.130000000,-0.610000000,0.260000000,-0.090000000\n2013-01-01,-0.420000000,-0.700000000,0.190000000,0.050000000\n2013-02-01,-0.400000000,-0.320000000,0.220000000,-0.020000000\n2013-03-01,-0.140000000,-0.380000000,0.550000000,-0.090000000\n2013-04-01,-0.080000000,-1.320000000,0.370000000,0.040000000\n2013-05-01,-0.280000000,-1.440000000,0.440000000,0.130000000\n2013-06-01,-0.330000000,-1.440000000,0.070000000,-0.030000000\n2013-07-01,-0.280000000,-1.360000000,0.060000000,-0.030000000\n2013-08-01,-0.290000000,-0.760000000,0.220000000,0.050000000\n2013-09-01,-0.090000000,-0.770000000,0.230000000,-0.120000000\n2013-10-01,-0.240000000,-0.160000000,0.230000000,-0.180000000\n2013-11-01,-0.020000000,-0.500000000,0.020000000,-0.260000000\n2013-12-01,-0.090000000,-0.250000000,-0.020000000,-0.160000000\n2014-01-01,-0.420000000,0.370000000,-0.120000000,-0.230000000\n2014-02-01,-0.450000000,-0.670000000,-0.250000000,-0.050000000\n2014-03-01,-0.070000000,-0.510000000,-0.500000000,-0.130000000\n2014-04-01,0.280000000,-0.250000000,-0.360000000,-0.080000000\n2014-05-01,0.450000000,0.740000000,-0.270000000,0.140000000\n2014-06-01,0.480000000,1.170000000,-0.290000000,0.150000000\n2014-07-01,0.130000000,1.040000000,-0.130000000,-0.090000000\n2014-08-01,0.140000000,0.670000000,-0.100000000,0.050000000\n2014-09-01,0.370000000,0.150000000,0.130000000,0.110000000\n2014-10-01,0.480000000,0.340000000,0.130000000,-0.060000000\n2014-11-01,0.890000000,0.570000000,0.040000000,-0.500000000\n2014-12-01,0.770000000,0.140000000,0.050000000,-0.380000000\n2015-01-01,0.590000000,-0.160000000,-0.080000000,0.100000000\n2015-02-01,0.570000000,-0.620000000,-0.060000000,0.150000000\n2015-03-01,0.480000000,-0.010000000,-0.270000000,-0.110000000\n2015-04-01,0.900000000,0.640000000,-0.400000000,0.050000000\n2015-05-01,1.040000000,1.590000000,-0.130000000,0.110000000\n2015-06-01,1.280000000,2.190000000,-0.310000000,0.070000000\n2015-07-01,1.560000000,2.320000000,-0.120000000,-0.190000000\n2015-08-01,1.870000000,2.080000000,0.090000000,-0.310000000\n2015-09-01,2.010000000,2.480000000,0.350000000,-0.190000000\n2015-10-01,2.210000000,2.380000000,0.510000000,0.060000000\n2015-11-01,2.570000000,2.210000000,0.180000000,0.190000000\n2015-12-01,2.560000000,2.090000000,0.130000000,0.440000000\n2016-01-01,2.560000000,1.820000000,0.250000000,0.470000000\n2016-02-01,2.110000000,1.090000000,0.280000000,0.320000000\n2016-03-01,1.600000000,1.100000000,0.240000000,0.250000000\n2016-04-01,1.050000000,0.290000000,0.170000000,0.080000000\n2016-05-01,0.450000000,0.450000000,0.180000000,0.010000000\n2016-06-01,0.060000000,0.520000000,0.150000000,0.370000000\n2016-07-01,-0.250000000,-0.060000000,0.100000000,0.370000000\n2016-08-01,-0.480000000,0.190000000,0.150000000,0.330000000\n2016-09-01,-0.460000000,0.250000000,0.010000000,0.110000000\n2016-10-01,-0.750000000,-0.030000000,0.120000000,0.240000000\n2016-11-01,-0.630000000,-0.080000000,0.150000000,0.100000000\n2016-12-01,-0.510000000,0.020000000,0.230000000,0.410000000\n2017-01-01,-0.340000000,0.330000000,0.240000000,0.250000000\n2017-02-01,-0.010000000,0.930000000,0.060000000,-0.180000000\n2017-03-01,-0.090000000,1.560000000,-0.030000000,-0.060000000\n2017-04-01,0.220000000,0.490000000,0.310000000,0.090000000\n2017-05-01,0.300000000,0.250000000,0.380000000,0.120000000\n2017-06-01,0.220000000,0.200000000,0.350000000,0.260000000\n2017-07-01,0.220000000,-0.460000000,0.320000000,0.100000000\n2017-08-01,-0.180000000,-0.760000000,0.330000000,0.390000000\n2017-09-01,-0.560000000,-1.280000000,0.150000000,0.120000000\n2017-10-01,-0.520000000,-1.320000000,0.030000000,0.190000000\n2017-11-01,-0.840000000,-1.440000000,0.220000000,0.020000000\n2017-12-01,-0.850000000,-1.540000000,0.250000000,-0.090000000\n2018-01-01,-0.860000000,-1.250000000,0.030000000,0.290000000\n2018-02-01,-0.730000000,-0.430000000,-0.270000000,-0.270000000\n2018-03-01,-0.730000000,-1.300000000,-0.130000000,-0.180000000\n2018-04-01,-0.360000000,-0.810000000,-0.330000000,-0.030000000\n2018-05-01,-0.120000000,-0.460000000,-0.600000000,-0.070000000\n2018-06-01,0.120000000,-0.470000000,-0.720000000,0.150000000\n2018-07-01,0.270000000,-0.080000000,-0.480000000,0.120000000\n2018-08-01,0.050000000,-0.540000000,-0.320000000,0.210000000\n2018-09-01,0.300000000,0.120000000,-0.040000000,0.210000000\n2018-10-01,0.840000000,0.300000000,-0.130000000,0.310000000\n2018-11-01,1.000000000,0.690000000,-0.320000000,0.210000000\n2018-12-01,0.970000000,0.790000000,-0.330000000,0.400000000\n2019-01-01,0.510000000,0.670000000,-0.170000000,0.410000000\n2019-02-01,0.620000000,0.660000000,-0.040000000,0.410000000\n2019-03-01,0.810000000,-0.180000000,-0.150000000,0.670000000\n2019-04-01,0.670000000,0.360000000,-0.170000000,0.340000000\n2019-05-01,0.610000000,0.240000000,-0.080000000,0.310000000\n2019-06-01,0.660000000,-0.110000000,0.060000000,0.270000000\n2019-07-01,0.410000000,-0.360000000,0.090000000,0.240000000\n2019-08-01,0.190000000,-0.290000000,0.090000000,0.220000000\n2019-09-01,0.110000000,-0.680000000,0.200000000,0.330000000\n2019-10-01,0.550000000,-0.710000000,0.010000000,0.250000000\n2019-11-01,0.740000000,0.020000000,-0.050000000,0.870000000\n2019-12-01,0.510000000,-0.190000000,-0.040000000,0.800000000\n2020-01-01,0.640000000,-0.120000000,0.010000000,0.950000000\n2020-02-01,0.480000000,-0.030000000,0.320000000,0.570000000\n2020-03-01,0.360000000,-0.440000000,0.580000000,0.310000000\n2020-04-01,0.490000000,0.030000000,0.460000000,0.410000000\n2020-05-01,-0.120000000,-0.240000000,0.320000000,0.530000000\n2020-06-01,-0.210000000,-0.430000000,0.470000000,0.290000000\n2020-07-01,-0.040000000,-1.090000000,0.380000000,0.290000000\n2020-08-01,-0.420000000,-0.430000000,0.250000000,0.300000000\n2020-09-01,-0.660000000,-0.850000000,0.150000000,0.090000000\n', 'research/lagged_sources/official_hashes.json': '[\n  {\n    "nome": "treino_tp.nc",\n    "sha256": "012bebbe0e38aefdf4f9e2432c3ca22f71d94ea1a1b5421da9d9e5048632cab0",\n    "bytes": 186685299\n  },\n  {\n    "nome": "treino_tp_alvo.nc",\n    "sha256": "915f67f2627e37399830d1dd92b209ce1160082b0645662675eab17df957abe4",\n    "bytes": 186509303\n  },\n  {\n    "nome": "teste_features.nc",\n    "sha256": "70bcb5c9bb9c011c8901660834b42dc57ff7e54a4f34161d489ec5e9f05c1f23",\n    "bytes": 39057097\n  },\n  {\n    "nome": "sample_submission.csv",\n    "sha256": "34fe449ddb309b0579a87786f76049863833d82c26f183d18e943dc76f1e4c8c",\n    "bytes": 48145117\n  },\n  {\n    "nome": "treino_cloud_cover.nc",\n    "sha256": "607ee1210615a0ca8a5c954737617e668a5f016eb881554cb1468fda74b4fde1",\n    "bytes": 190610590\n  },\n  {\n    "nome": "treino_geopotential_850.nc",\n    "sha256": "ca3baca0177e2e90b42901803f348a4b4affe88bee202b76af3f293fa7035c17",\n    "bytes": 159707312\n  },\n  {\n    "nome": "treino_rel_hum_850.nc",\n    "sha256": "90f62b44b1b461259e7539249914e26cf09993f3bcd572ba4871931e45264b19",\n    "bytes": 180398147\n  },\n  {\n    "nome": "treino_shum_850.nc",\n    "sha256": "35cf8071b68d1e924ad87e0f8bd801d0a66d1d78ae75fc3cb623bde85c3f47bb",\n    "bytes": 175651693\n  },\n  {\n    "nome": "treino_surface_pressure.nc",\n    "sha256": "f00a5e44a03ea923ac0f693b975ef71cd347f8ad3d07692c602e3ac9a25c10fb",\n    "bytes": 160228025\n  },\n  {\n    "nome": "treino_t2.nc",\n    "sha256": "7d8c53b438289cddf87a11414ad10c8167f52ff33c6eff459a3288975079ef23",\n    "bytes": 175952066\n  },\n  {\n    "nome": "treino_temperature_850.nc",\n    "sha256": "1db7b4fdeba87b6a25f7fb6f1e40abec50964234dd9cc918cd34f902b8e41bac",\n    "bytes": 172474226\n  },\n  {\n    "nome": "treino_u_850.nc",\n    "sha256": "73c898f27bea25cf2ab0914bbe5c29acc9794a93f8b6b3affa39854bebccf761",\n    "bytes": 188451602\n  },\n  {\n    "nome": "treino_v_850.nc",\n    "sha256": "7bab583972e5718bb0e0fe0fe6a056bd56c968af1f1497205182f7621e26c241",\n    "bytes": 197354831\n  }\n]', 'research/spatial_unet/__init__.py': '"""Monthly spatial rainfall experiment."""\n', 'research/spatial_unet/network.py': '"""A compact U-Net for continuous rainfall anomalies, not image classification."""\nimport random\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\ndef seed_everything(seed=42):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    # Some CUDA interpolation backward kernels have no deterministic implementation.\n    # Keep the warning visible and archive the environment; do not promise bit equality.\n    torch.use_deterministic_algorithms(True, warn_only=True)\n\n\nclass Block(nn.Sequential):\n    def __init__(self, inputs, outputs):\n        super().__init__(nn.Conv2d(inputs, outputs, 3, padding=1, padding_mode=\'replicate\'),\n                         nn.GroupNorm(4, outputs), nn.SiLU(),\n                         nn.Conv2d(outputs, outputs, 3, padding=1, padding_mode=\'replicate\'),\n                         nn.GroupNorm(4, outputs), nn.SiLU())\n\n\nclass RainfallUNet(nn.Module):\n    def __init__(self, inputs=31, widths=(16, 32, 64)):\n        super().__init__()\n        a, b, c = widths\n        self.encoder1 = Block(inputs, a)\n        self.encoder2 = Block(a, b)\n        self.bottleneck = Block(b, c)\n        self.decoder2 = Block(c + b, b)\n        self.decoder1 = Block(b + a, a)\n        self.output = nn.Conv2d(a, 1, 1)\n        # Start at the climatology. Negative anomalies remain learnable.\n        nn.init.zeros_(self.output.weight)\n        nn.init.zeros_(self.output.bias)\n\n    def forward(self, x):\n        e1 = self.encoder1(x)\n        e2 = self.encoder2(F.avg_pool2d(e1, 2))\n        b = self.bottleneck(F.avg_pool2d(e2, 2))\n        d2 = self.decoder2(torch.cat([F.interpolate(b, size=e2.shape[-2:], mode=\'bilinear\', align_corners=False), e2], 1))\n        d1 = self.decoder1(torch.cat([F.interpolate(d2, size=e1.shape[-2:], mode=\'bilinear\', align_corners=False), e1], 1))\n        return self.output(d1)[:, 0]\n\n\ndef restore(path, device=\'cpu\'):\n    checkpoint = torch.load(path, map_location=device, weights_only=True)\n    model = RainfallUNet(widths=tuple(checkpoint[\'widths\'])).to(device)\n    model.load_state_dict(checkpoint[\'state_dict\'])\n    model.eval()\n    return model, checkpoint\n', 'research/spatial_unet/protocol.json': '{\n  "id": "spatial_unet_v1",\n  "status": "development_experiment_not_independent_holdout",\n  "sources": "same 31 available features as the lagged multisystem component; no SST or new source",\n  "lags": {"atmosphere": 4, "indices": 3, "seasonal_initialization": 1, "training_rainfall": 4},\n  "evaluation": ["2007-01", "2020-12"],\n  "max_training_months": 360,\n  "seed": 42,\n  "architecture": {"channels": [16, 32, 64], "convolutions_per_block": 2, "kernel": 3, "padding": "replicate", "activation": "SiLU", "normalization": "GroupNorm_4", "upsampling": "bilinear_align_corners_false", "output": "one_unbounded_rainfall_anomaly"},\n  "optimizer": {"name": "AdamW", "learning_rate": 0.001, "weight_decay": 0.0001, "gradient_clip_norm": 1.0},\n  "batch_size": 2,\n  "max_epochs": 40,\n  "patience": 6,\n  "inner_validation_months": 24,\n  "inner_label_lag": 4,\n  "selection": "lowest reconstructed inner RMSE; earliest epoch on ties; refit from seed on full outer training window for that epoch count",\n  "loss": "unweighted pixel MSE in mm/day anomalies; mean over complete monthly maps",\n  "scaling": "per-channel mean and standard deviation fitted on training maps only; standard deviation floor 1e-6",\n  "augmentation": "none: do not flip geography or mix months across temporal boundaries",\n  "reconstruction": "maximum(0, training_rainfall_climatology + predicted_anomaly); no tuned residual scale",\n  "baseline": {"architecture": "two_LightGBM_and_local_ridge", "weights": [0.375, 0.375, 0.25], "residual_scales": [0.9, 0.875]},\n  "blend": "none",\n  "outer_metrics": ["RMSE", "MAE", "bias", "area_weighted_RMSE", "year", "block", "latitude_band"],\n  "comparison_limit": "same source information and target months; U-Net uses all pixels in monthly maps, baseline trees use 5000 sampled pixels per month",\n  "prospective_registration": "separate future protocol after historical development; never alter the existing hybrid registry"\n}\n', 'research/spatial_unet/run_experiment.py': '"""Compare a monthly-map U-Net to the unchanged lagged hybrid on development years."""\nfrom pathlib import Path\nimport argparse\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\nimport sys\nimport tempfile\nimport numpy as np\nimport pandas as pd\nimport torch\nimport xarray as xr\n\nROOT = Path(__file__).resolve().parents[2]\nif str(ROOT) not in sys.path:\n    sys.path.insert(0, str(ROOT))\nfrom research.common.inputs import Inputs, Context, require, write_json, sha256\nfrom research.common import baseline\nfrom research.spatial_unet.training import MonthlyMaps, fit_network, inference_maps, split_inner\n\n\ndef signature(data, protocol, device):\n    code = {}\n    for folder in [\'common\', \'spatial_unet\']:\n        for p in sorted((ROOT / \'research\' / folder).glob(\'*.py\')):\n            code[p.relative_to(ROOT).as_posix()] = sha256(p)\n    return dict(inputs=data.audit, protocol=protocol, code=code,\n        python=platform.python_version(), torch=torch.__version__, device=str(device),\n        gpu=torch.cuda.get_device_name(device) if device.type == \'cuda\' else None,\n        cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version())\n\n\ndef run_block(data, block, config, directory, device):\n    m = data.lib\n    start = pd.Timestamp(block[\'inicio\'])\n    cutoff = m.origem_mensal([start], 4)[0]\n    dates = pd.date_range(start, periods=24, freq=\'MS\')\n    training = m.calendario_pareado(cutoff)\n    reference = m.janela_referencia(cutoff)\n    inner_train, inner_reference, inner_dates = split_inner(training, reference, config[\'inner_validation_months\'])\n    directory.mkdir(parents=True, exist_ok=True)\n    m.calendario_emissoes(dates, cutoff).to_csv(directory / \'source_calendar.csv\', index=False)\n    write_json(directory / \'split.json\', dict(train=training.strftime(\'%Y-%m\').tolist(),\n        inner_train=inner_train.strftime(\'%Y-%m\').tolist(), inner_reference=inner_reference.strftime(\'%Y-%m\').tolist(),\n        inner_validation=inner_dates.strftime(\'%Y-%m\').tolist(), outer=dates.strftime(\'%Y-%m\').tolist(),\n        independent_holdout=False, historical_publication_vintages_verified=False))\n    # Verify the inexpensive control before spending GPU time on the neural fit.\n    state = baseline.fit(data, cutoff, directory / \'hybrid\')\n    hybrid = baseline.predict(data, state, dates)\n    control_metrics = m.pesquisa_diagnosticos(data.rain.sel(time=dates), {\'hybrid\': hybrid}, block[\'nome\'])\n    reference_metrics = pd.read_csv(ROOT / \'research/lagged_sources/evidence/metricas_blocos.csv\')\n    old = reference_metrics[(reference_metrics.bloco == block[\'nome\']) & (reference_metrics.modelo == \'controle_defasado\')].iloc[0]\n    now = m.pesquisa_resumir(control_metrics[control_metrics.regiao == \'dominio_inteiro\'], [\'modelo\']).iloc[0]\n    for key in [\'rmse\', \'mae\', \'vies\']:\n        require(abs(float(now[key]) - float(old[key])) < 1e-6, f\'Control did not reproduce archived {key}. Stop comparison.\')\n    write_json(directory / \'baseline_check.json\', dict(reproduced=True, tolerance=1e-6,\n        rmse_archived=float(old.rmse), rmse_current=float(now.rmse)))\n    print(block[\'nome\'], \'fixed hybrid reproduced; starting neural fit.\', flush=True)\n    baseline_climate = state[\'climate\']\n    del state\n    gc.collect()\n    scratch = directory / \'temporary_maps\'\n    scratch.mkdir(exist_ok=True)\n    with tempfile.TemporaryDirectory(dir=scratch) as temp:\n        temp = Path(temp)\n        inner = Context(data, inner_reference, inner_train)\n        inner.save(directory / \'inner\')\n        train = MonthlyMaps(inner, inner_train, temp / \'inner_train\', training=True)\n        valid = MonthlyMaps(inner, inner_dates, temp / \'inner_valid\', scaling=train.scaling)\n        try:\n            net, epochs = fit_network(train, config, directory / \'inner\', device, validation=valid)\n        finally:\n            train.close()\n            valid.close()\n        del train, valid, inner, net\n        gc.collect()\n        if device.type == \'cuda\':\n            torch.cuda.empty_cache()\n\n        # Final preprocessing is refitted after epoch selection, using only outer training.\n        context = Context(data, reference, training)\n        context.save(directory / \'unet\')\n        train = MonthlyMaps(context, training, temp / \'outer_train\', training=True)\n        try:\n            net, _ = fit_network(train, config, directory / \'unet\', device, epochs=epochs)\n            scaling = train.scaling\n        finally:\n            train.close()\n        valid = MonthlyMaps(context, dates, temp / \'outer_eval\', scaling=scaling)\n        try:\n            prediction = inference_maps(net, valid, device, config[\'batch_size\'])\n        finally:\n            valid.close()\n        coords = dict(time=dates, lat=data.rain.lat, lon=data.rain.lon)\n        predictions = dict(unet=xr.DataArray(prediction, dims=(\'time\', \'lat\', \'lon\'), coords=coords, attrs=dict(units=\'mm/day\')))\n        del net, train, valid\n        gc.collect()\n        if device.type == \'cuda\':\n            torch.cuda.empty_cache()\n\n    require(np.array_equal(baseline_climate.values, context.climate.values), \'Baseline and U-Net rainfall references differ.\')\n    predictions[\'hybrid\'] = hybrid\n    predictions[\'climatology\'] = xr.DataArray(context.climate.values[dates.month - 1], dims=(\'time\', \'lat\', \'lon\'), coords=coords, attrs=dict(units=\'mm/day\'))\n    metrics = m.pesquisa_diagnosticos(data.rain.sel(time=dates), predictions, block[\'nome\'])\n    xr.Dataset(predictions, attrs=dict(training_end=str(cutoff.date()), independent_holdout=\'false\', chosen_epochs=epochs)).to_netcdf(\n        directory / \'predictions.nc\', engine=\'h5netcdf\', encoding={k: dict(zlib=True, complevel=4) for k in predictions})\n    metrics.to_csv(directory / \'monthly_metrics.csv\', index=False)\n    return metrics\n\n\ndef summarize(data, directory, frames):\n    all_metrics = pd.concat(frames, ignore_index=True)\n    require(not all_metrics.duplicated([\'modelo\', \'mes_alvo\', \'regiao\']).any(), \'Duplicate evaluation rows.\')\n    all_metrics.to_csv(directory / \'monthly_metrics.csv\', index=False)\n    domain = all_metrics[all_metrics.regiao == \'dominio_inteiro\']\n    for name, frame, groups in [(\'global\', domain, [\'modelo\']), (\'blocks\', domain, [\'modelo\', \'bloco\']),\n                                (\'years\', domain, [\'modelo\', \'ano\']), (\'regions\', all_metrics, [\'modelo\', \'regiao\'])]:\n        summary = data.lib.pesquisa_resumir(frame, groups)\n        summary.to_csv(directory / (name + \'.csv\'), index=False)\n    totals = pd.read_csv(directory / \'global.csv\').set_index(\'modelo\')\n    complete = len(domain.bloco.unique()) == 7\n    result = dict(complete_seven_blocks=complete, blocks=sorted(domain.bloco.unique().tolist()),\n        unet_rmse=float(totals.loc[\'unet\', \'rmse\']), hybrid_rmse=float(totals.loc[\'hybrid\', \'rmse\']),\n        delta_rmse=float(totals.loc[\'unet\', \'rmse\'] - totals.loc[\'hybrid\', \'rmse\']),\n        model_promoted=False, forecast_issued=False, independent_holdout=False,\n        interpretation=\'Development comparison on previously consulted years. A pilot is not the seven-block result.\')\n    write_json(directory / \'summary.json\', result)\n    print(json.dumps(result, indent=2), flush=True)\n    return result\n\n\ndef execute(output, blocks=(\'H1\',), official=None, seas5=None, cfsv2=None, device=None):\n    config = json.loads(Path(__file__).with_name(\'protocol.json\').read_text())\n    require(config[\'id\'] == \'spatial_unet_v1\' and config[\'lags\'] == dict(atmosphere=4, indices=3, seasonal_initialization=1, training_rainfall=4), \'Different temporal protocol.\')\n    require(config[\'inner_label_lag\'] == 4 and config[\'max_training_months\'] == 360, \'Different training protocol.\')\n    device = torch.device(device or (\'cuda\' if torch.cuda.is_available() else \'cpu\'))\n    if device.type == \'cpu\':\n        torch.set_num_threads(4)\n        print(\'CPU selected: full-grid training may be slow. Prefer a Kaggle GPU.\', flush=True)\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    data = Inputs(official, seas5, cfsv2, output=output)\n    run_signature = signature(data, config, device)\n    signature_file = output / \'signature.json\'\n    if signature_file.exists():\n        require(json.loads(signature_file.read_text()) == run_signature, \'Changed inputs, code or device. Preserve this run and choose another output directory.\')\n    else:\n        write_json(signature_file, run_signature)\n    selected = [b for b in data.lib.BLOCOS if b[\'nome\'] in blocks]\n    require(len(selected) == len(set(blocks)) and selected, \'Unknown or empty block selection.\')\n    frames = []\n    for block in selected:\n        directory = output / block[\'nome\']\n        complete = directory / \'complete.json\'\n        if complete.exists():\n            record = json.loads(complete.read_text())\n            require(record[\'signature\'] == sha256(signature_file), \'Changed completed-block signature.\')\n            for name, digest in record[\'files\'].items():\n                require(sha256(directory / name) == digest, f\'Changed block artifact: {name}\')\n            metrics = pd.read_csv(directory / \'monthly_metrics.csv\')\n            print(block[\'nome\'], \'verified cached result\', flush=True)\n        else:\n            require(not directory.exists(), f\'Incomplete run in {directory}. Preserve it and use a new output directory.\')\n            metrics = run_block(data, block, config, directory, device)\n            files = {p.relative_to(directory).as_posix(): sha256(p) for p in directory.rglob(\'*\') if p.is_file()}\n            write_json(complete, dict(signature=sha256(signature_file), files=files))\n        frames.append(metrics)\n    return summarize(data, output, frames)\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--output\', required=True)\n    parser.add_argument(\'--blocks\', nargs=\'+\', default=[\'H1\'])\n    parser.add_argument(\'--official\')\n    parser.add_argument(\'--seas5\')\n    parser.add_argument(\'--cfsv2\')\n    parser.add_argument(\'--device\', choices=[\'cpu\', \'cuda\'])\n    args = parser.parse_args()\n    execute(**vars(args))\n', 'research/spatial_unet/test_spatial.py': '"""Synthetic temporal, spatial and training checks; never a rainfall skill benchmark."""\nfrom pathlib import Path\nimport copy\nimport json\nimport sys\nimport tempfile\nfrom types import SimpleNamespace\nimport unittest\nimport numpy as np\nimport pandas as pd\nimport torch\nimport xarray as xr\n\nROOT = Path(__file__).resolve().parents[2]\nsys.path.insert(0, str(ROOT))\nfrom research.common.inputs import Context, library\nfrom research.spatial_unet.network import RainfallUNet, seed_everything, restore\nfrom research.spatial_unet.training import MonthlyMaps, split_inner, fit_network, inference_maps\n\n\nfrom research.common.synthetic_fixture import fixture\n\n\nclass SpatialTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        torch.set_num_threads(2)\n\n    def setUp(self):\n        self.data = fixture()\n        self.reference = pd.date_range(\'1976-10-01\', \'2006-09-01\', freq=\'MS\')\n        self.training = self.reference[self.reference >= \'1982-03-01\']\n\n    def test_inner_selection_has_label_gap_and_no_new_history(self):\n        train, reference, valid = split_inner(self.training, self.reference)\n        self.assertEqual(len(valid), 24)\n        self.assertEqual(train[-1], valid[0] - pd.DateOffset(months=4))\n        self.assertEqual(reference[0], self.reference[0])\n        self.assertTrue(reference.max() < valid.min())\n\n    def test_future_observations_do_not_change_context_or_features(self):\n        a = Context(self.data, self.reference, self.training)\n        expected = a.features(\'2007-01-01\')\n        self.data.rain.loc[dict(time=slice(\'2006-10-01\', None))] = 9000\n        for field in self.data.atmosphere.values():\n            field.loc[dict(time=slice(\'2006-10-01\', None))] = 9000\n        self.data.lib.INDICES_OC.loc[\'2006-11-01\':] = 9000\n        self.data.seas.loc[dict(time=slice(\'2007-02-01\', None))] = 9000\n        self.data.cfs.loc[dict(time=slice(\'2007-02-01\', None))] = 9000\n        b = Context(self.data, self.reference, self.training)\n        np.testing.assert_array_equal(a.climate, b.climate)\n        np.testing.assert_array_equal(expected, b.features(\'2007-01-01\'))\n\n    def test_training_climatology_excludes_entire_target_map(self):\n        t = self.training[36]\n        a = Context(self.data, self.reference, self.training).features(t, training=True)[22]\n        self.data.rain.loc[dict(time=t)] += np.arange(117).reshape(9, 13)\n        b = Context(self.data, self.reference, self.training).features(t, training=True)[22]\n        np.testing.assert_allclose(a, b, rtol=0, atol=1e-6)\n\n    def test_context_matches_existing_baseline_transform(self):\n        m = self.data.lib\n        ctx = Context(self.data, self.reference, self.training)\n        climate, _, _ = m.estatisticas_climatologia(self.data.rain, self.reference[-1])\n        ca = m.ajustar_clima_atmosfera(self.data.atmosphere, m.origem_mensal(self.reference, 4))\n        ca[\'_clima_indices\'] = m.ajustar_clima_indices(m.INDICES_OC, m.origem_mensal(self.reference, 4), self.reference[-1])\n        cs = m.ajustar_clima_seas5(self.data.seas, self.reference[-1])\n        cc = m.ajustar_clima_previsao(self.data.cfs, self.reference[-1], \'CFSv2\')\n        t = pd.Timestamp(\'2007-01-01\')\n        x = np.column_stack([m.matriz_mes(self.data.atmosphere, ca, climate, t),\n            m.valores_seas5(self.data.seas, t), m.valores_anomalia_seas5(self.data.seas, cs, t),\n            m.valores_cfsv2(self.data.cfs, t), m.valores_anomalia_cfsv2(self.data.cfs, cc, t)])\n        np.testing.assert_array_equal(ctx.features(t).reshape(31, -1).T, x)\n\n    def test_odd_grid_shape_and_initial_climatology(self):\n        seed_everything()\n        net = RainfallUNet()\n        with torch.inference_mode():\n            result = net(torch.zeros(1, 31, 301, 261))\n        self.assertEqual(tuple(result.shape), (1, 301, 261))\n        self.assertEqual(float(result.abs().max()), 0.)\n\n    def test_gradients_and_checkpoint_roundtrip(self):\n        seed_everything()\n        net = RainfallUNet(widths=(4, 8, 16))\n        x = torch.randn(2, 31, 9, 13)\n        optimizer = torch.optim.AdamW(net.parameters(), lr=.001)\n        loss = ((net(x) - 1) ** 2).mean()\n        loss.backward()\n        self.assertTrue(torch.isfinite(net.output.weight.grad).all())\n        optimizer.step()\n        self.assertFalse(torch.equal(net(x), torch.zeros(2, 9, 13)))\n        with tempfile.TemporaryDirectory() as folder:\n            path = Path(folder) / \'weights.pt\'\n            torch.save(dict(state_dict=net.state_dict(), widths=[4, 8, 16]), path)\n            loaded, _ = restore(path)\n            torch.testing.assert_close(net(x), loaded(x), rtol=0, atol=0)\n\n    def test_training_loop_and_scaler_use_only_training_maps(self):\n        config = json.loads(Path(__file__).with_name(\'protocol.json\').read_text())\n        config.update(max_epochs=2, patience=2)\n        config[\'architecture\'][\'channels\'] = [4, 8, 16]\n        ctx = Context(self.data, self.reference, self.training)\n        with tempfile.TemporaryDirectory() as folder:\n            folder = Path(folder)\n            train = MonthlyMaps(ctx, self.training[-4:], folder / \'train\', training=True)\n            valid = MonthlyMaps(ctx, pd.date_range(\'2007-01-01\', periods=2, freq=\'MS\'), folder / \'valid\', scaling=train.scaling)\n            try:\n                np.testing.assert_array_equal(valid.scaling[\'mean\'], train.scaling[\'mean\'])\n                model, epochs = fit_network(train, config, folder / \'model\', torch.device(\'cpu\'), validation=valid)\n                self.assertIn(epochs, [1, 2])\n                pred = inference_maps(model, valid, torch.device(\'cpu\'))\n                self.assertTrue(np.isfinite(pred).all() and (pred >= 0).all())\n                self.assertEqual(pred.shape, (2, 9, 13))\n            finally:\n                train.close()\n                valid.close()\n\n    def test_invalid_month_cannot_be_used_for_training(self):\n        ctx = Context(self.data, self.reference, self.training)\n        with self.assertRaises(ValueError):\n            ctx.features(\'2007-01-01\', training=True)\n        with self.assertRaises(ValueError):\n            ctx.features(\'2006-09-01\', training=False)\n\n\nif __name__ == \'__main__\':\n    unittest.main(argv=[__file__])\n', 'research/spatial_unet/training.py': '"""Monthly map datasets and chronological epoch selection."""\nfrom pathlib import Path\nimport time\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import Dataset, DataLoader\nfrom .network import RainfallUNet, seed_everything\nfrom research.common.inputs import require, write_json\n\n\ndef split_inner(training, reference, validation_months=24):\n    training, reference = pd.DatetimeIndex(training), pd.DatetimeIndex(reference)\n    validation = training[-validation_months:]\n    cutoff = validation[0] - pd.DateOffset(months=4)\n    fit = training[training <= cutoff]\n    refs = reference[reference <= cutoff]\n    require(len(fit) >= 24 and fit[-1] < validation[0], \'Insufficient chronological inner training.\')\n    return fit, refs, validation\n\n\nclass MonthlyMaps(Dataset):\n    def __init__(self, context, dates, directory, training=False, scaling=None):\n        self.dates = pd.DatetimeIndex(dates)\n        self.directory = Path(directory)\n        self.directory.mkdir(parents=True, exist_ok=False)\n        d = context.data\n        n, h, w = len(self.dates), d.rain.sizes[\'lat\'], d.rain.sizes[\'lon\']\n        self.x = np.lib.format.open_memmap(self.directory / \'features.npy\', mode=\'w+\', dtype=\'float32\', shape=(n, 31, h, w))\n        self.y = np.lib.format.open_memmap(self.directory / \'target.npy\', mode=\'w+\', dtype=\'float32\', shape=(n, h, w))\n        self.base = np.lib.format.open_memmap(self.directory / \'climate.npy\', mode=\'w+\', dtype=\'float32\', shape=(n, h, w))\n        totals, squares, pixels = np.zeros(31), np.zeros(31), 0\n        for i, t in enumerate(self.dates):\n            x = context.features(t, training)\n            self.x[i] = x\n            self.base[i] = x[22]\n            self.y[i] = d.rain.sel(time=t).values\n            if scaling is None:\n                require(training, \'Only training maps may fit scaling.\')\n                flat = x.reshape(31, -1).astype(\'float64\')\n                totals += flat.sum(1)\n                squares += np.square(flat).sum(1)\n                pixels += flat.shape[1]\n        if scaling is None:\n            mean = totals / pixels\n            scale = np.sqrt(np.maximum(squares / pixels - mean ** 2, 0))\n            self.scaling = dict(mean=mean.astype(\'float32\'), scale=np.maximum(scale, 1e-6).astype(\'float32\'))\n        else:\n            self.scaling = {k: np.array(v, copy=True) for k, v in scaling.items()}\n        for arr in [self.x, self.y, self.base]:\n            require(all(np.isfinite(arr[i]).all() for i in range(n)), \'Invalid map cache.\')\n            arr.flush()\n\n    def __len__(self):\n        return len(self.dates)\n\n    def __getitem__(self, i):\n        x = (self.x[i] - self.scaling[\'mean\'][:, None, None]) / self.scaling[\'scale\'][:, None, None]\n        return torch.from_numpy(x), torch.from_numpy(self.y[i] - self.base[i]), torch.from_numpy(self.base[i].copy())\n\n    def close(self):\n        # Only files created by this instance; no recursive or input-directory deletion.\n        for arr in [self.x, self.y, self.base]:\n            arr.flush()\n            arr._mmap.close()\n        for name in [\'features.npy\', \'target.npy\', \'climate.npy\']:\n            (self.directory / name).unlink()\n        self.directory.rmdir()\n\n\ndef inference_maps(model, dataset, device, batch_size=2):\n    model.eval()\n    out = []\n    with torch.inference_mode():\n        for x, _, base in DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0):\n            anomaly = model(x.to(device)).cpu()\n            out.append(torch.clamp(base + anomaly, min=0).numpy())\n    result = np.concatenate(out)\n    require(np.isfinite(result).all(), \'Nonfinite network output.\')\n    return result\n\n\ndef fit_network(train, config, folder, device, validation=None, epochs=None):\n    seed_everything(config[\'seed\'])\n    folder = Path(folder)\n    folder.mkdir(parents=True, exist_ok=True)\n    model = RainfallUNet(widths=tuple(config[\'architecture\'][\'channels\'])).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=config[\'optimizer\'][\'learning_rate\'], weight_decay=config[\'optimizer\'][\'weight_decay\'])\n    generator = torch.Generator().manual_seed(config[\'seed\'])\n    loader = DataLoader(train, batch_size=config[\'batch_size\'], shuffle=True, generator=generator, num_workers=0)\n    limit = epochs if epochs is not None else config[\'max_epochs\']\n    history, best, best_epoch, stale = [], float(\'inf\'), 0, 0\n    for epoch in range(1, limit + 1):\n        t0, total, count = time.perf_counter(), 0., 0\n        model.train()\n        for x, y, _ in loader:\n            x, y = x.to(device), y.to(device)\n            optimizer.zero_grad(set_to_none=True)\n            prediction = model(x)\n            loss = torch.mean((prediction - y) ** 2)\n            require(torch.isfinite(loss).item(), \'Training loss diverged.\')\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), config[\'optimizer\'][\'gradient_clip_norm\'], error_if_nonfinite=True)\n            optimizer.step()\n            total += loss.item() * y.numel()\n            count += y.numel()\n        row = dict(epoch=epoch, training_mse=total / count, seconds=time.perf_counter() - t0)\n        if validation is not None:\n            pred = inference_maps(model, validation, device, config[\'batch_size\'])\n            truth = validation.y.astype(\'float64\')\n            rmse = float(np.sqrt(np.mean((pred.astype(\'float64\') - truth) ** 2)))\n            row[\'inner_rmse\'] = rmse\n            if rmse < best:\n                best, best_epoch, stale = rmse, epoch, 0\n            else:\n                stale += 1\n        history.append(row)\n        write_json(folder / \'training_history.json\', history)\n        print(folder.name, row, flush=True)\n        if validation is not None and stale >= config[\'patience\']:\n            break\n    selected = best_epoch if validation is not None else limit\n    # Inner weights are never evaluated on the outer block. Refit starts from seed.\n    torch.save(dict(state_dict=model.state_dict(), widths=config[\'architecture\'][\'channels\'],\n        epochs_run=epoch, selected_epochs=selected, seed=config[\'seed\']), folder / \'network.pt\')\n    np.savez_compressed(folder / \'scaling.npz\', **train.scaling)\n    write_json(folder / \'training.json\', dict(selected_epochs=selected,\n        parameter_count=sum(p.numel() for p in model.parameters()), train_months=len(train),\n        train_start=str(train.dates[0].date()), train_end=str(train.dates[-1].date()),\n        inner_selection=validation is not None, device=str(device)))\n    return model, selected\n', 'research/temporal_extension/__init__.py': '"""Locked retrospective extension of the fixed hybrid / U-Net comparison."""\n', 'research/temporal_extension/experiment.py': '"""Train/freeze first, then score an additional historical block in a separate call."""\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nimport argparse\nimport gc\nimport json\nimport platform\nimport shutil\nimport tempfile\nfrom types import SimpleNamespace\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport xarray as xr\nfrom torch.utils.data import Dataset\n\nfrom research.common.inputs import ROOT, Context, library, preflight, require, sha256, write_json\nfrom research.common import baseline\nfrom research.spatial_unet.training import MonthlyMaps, fit_network, inference_maps, split_inner\n\nPROTOCOL_PATH = Path(__file__).with_name(\'protocol.json\')\nSOURCE_PROTOCOL = ROOT / \'research/spatial_unet/protocol.json\'\nINDEX_SNAPSHOT = ROOT / \'competition/metadata/NOAA/indices_noaa.csv\'\nDEVELOPMENT_INDICES = ROOT / \'research/lagged_sources/ocean_indices.csv\'\nINDEX_SNAPSHOT_SHA256 = \'b6f444fbec48681c1e5b46a5c22aa197ad91c7fbbdaff82c3ed3a9c63f7c15a4\'\nMODELS = (\'hybrid\', \'unet\', \'blend\', \'climatology\')\n\n\ndef utcnow():\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef protocol():\n    plan = json.loads(PROTOCOL_PATH.read_text())\n    require(plan[\'id\'] == \'temporal_extension_2021_2022_v1\', \'Unexpected extension protocol.\')\n    require(plan[\'training_start\'] == \'1990-10-01\' and plan[\'training_end\'] == \'2020-09-01\'\n            and plan[\'evaluation_start\'] == \'2021-01-01\' and plan[\'evaluation_end\'] == \'2022-12-01\', \'Extension dates changed.\')\n    require(plan[\'weights\'] == {\'hybrid\': .75, \'unet\': .25} and plan[\'weight_search\'] is False\n            and plan[\'hyperparameter_search\'] is False, \'This experiment has one fixed candidate.\')\n    require(sha256(SOURCE_PROTOCOL) == plan[\'source_protocol_sha256\'], \'Original U-Net protocol changed.\')\n    return plan, json.loads(SOURCE_PROTOCOL.read_text())\n\n\ndef source_files():\n    paths = [SOURCE_PROTOCOL, PROTOCOL_PATH, ROOT / \'research/lagged_sources/library.py\',\n             ROOT / \'research/lagged_sources/official_hashes.json\', DEVELOPMENT_INDICES, INDEX_SNAPSHOT]\n    for folder in [\'common\', \'spatial_unet\', \'temporal_extension\']:\n        paths += list((ROOT / \'research\' / folder).glob(\'*.py\'))\n    return sorted(set(paths))\n\n\ndef calendar(plan):\n    training = pd.date_range(plan[\'training_start\'], plan[\'training_end\'], freq=\'MS\')\n    targets = pd.date_range(plan[\'evaluation_start\'], plan[\'evaluation_end\'], freq=\'MS\')\n    require(len(training) == 360 and len(targets) == 24 and training[-1] == targets[0] - pd.DateOffset(months=4),\n            \'Invalid four-month label gap or window length.\')\n    return training, targets\n\n\ndef require_index_coverage(table, required):\n    library().conferir_indices(table)\n    missing = pd.DatetimeIndex(required).difference(table.index)\n    require(len(missing) == 0, \'Ocean indices do not cover all training/forecast origins. Missing: \'\n            + \', \'.join(missing.strftime(\'%Y-%m\').tolist()[:12]) + \'. No fitting or imputation is allowed.\')\n\n\ndef index_snapshot(plan):\n    """Extend the calendar using the already archived NOAA table, with exact overlap checks."""\n    require(sha256(INDEX_SNAPSHOT) == INDEX_SNAPSHOT_SHA256, \'Archived NOAA snapshot changed.\')\n    full = pd.read_csv(INDEX_SNAPSHOT, parse_dates=[\'time_origem\']).set_index(\'time_origem\')\n    development = pd.read_csv(DEVELOPMENT_INDICES, parse_dates=[\'time_origem\']).set_index(\'time_origem\')\n    library().conferir_indices(development)\n    shared = development.index.intersection(full.index)\n    require(len(shared) > 0 and full.columns.equals(development.columns)\n            and np.array_equal(full.loc[shared].values, development.loc[shared].values),\n            \'Archived NOAA values differ from the development snapshot on shared months.\')\n    training, targets = calendar(plan)\n    required = pd.date_range(training[0] - pd.DateOffset(months=3), targets[-1] - pd.DateOffset(months=3), freq=\'MS\')\n    require_index_coverage(full, required)\n    audit = dict(source=INDEX_SNAPSHOT.relative_to(ROOT).as_posix(), sha256=sha256(INDEX_SNAPSHOT),\n                 development_sha256=sha256(DEVELOPMENT_INDICES), shared_months=len(shared), overlap_exact=True,\n                 selected_start=str(required[0].date()), selected_end=str(required[-1].date()),\n                 selected_months=len(required), new_download=False, historical_publication_vintages_verified=False)\n    return full.loc[required].copy(), audit\n\n\ndef read_training_rain(path, training):\n    with xr.open_dataset(path) as ds:\n        require(ds.tp.attrs.get(\'units\') == \'mm/day\', \'Unexpected rainfall units.\')\n        rain = ds.tp.sel(time=training).transpose(\'time\', \'lat\', \'lon\').load()\n    require(pd.DatetimeIndex(rain.time.values).equals(training), \'Training rainfall calendar differs.\')\n    require(np.isfinite(rain.values).all() and (rain.values >= 0).all(), \'Invalid training rainfall.\')\n    return rain\n\n\ndef load_inputs(paths, audit, output, plan):\n    training, targets = calendar(plan)\n    m = library()\n    m.PASTA, m.PASTA_SEAS5_MANUAL, m.PASTA_CFSV2_MANUAL = paths[\'official\'], paths[\'seas5\'], paths[\'cfsv2\']\n    m.PASTA_SEAS5, m.MANIFESTO_SEAS5 = m.localizar_seas5()\n    m.PASTA_CFSV2, m.MANIFESTO_CFSV2 = m.localizar_cfsv2()\n    m.SAIDA = output\n    # Only the fitting window is decoded here. No dummy evaluation rainfall is inserted.\n    rain = read_training_rain(paths[\'official\'] / \'treino_tp.nc\', training)\n    require(np.array_equal(rain.lat, np.arange(-60, 15.25, .25)) and np.array_equal(rain.lon, np.arange(-90, -24.75, .25)), \'Different grid.\')\n    atmosphere, _ = m.carregar_atmosfera(rain, training[0] - pd.DateOffset(months=4), targets[-1] - pd.DateOffset(months=4))\n    m.INDICES_OC, indices_audit = index_snapshot(plan)\n    write_json(output / \'indices.json\', indices_audit)\n    phases = [\'desenvolvimento\', \'somente_ajuste_final\']\n    seas = xr.concat([m.carregar_seas5(p, rain) for p in phases], dim=\'time\').sel(time=slice(training[0], targets[-1]))\n    cfs = xr.concat([m.carregar_cfsv2(p, rain) for p in phases], dim=\'time\').sel(time=slice(training[0], targets[-1]))\n    audit = dict(audit, indices=indices_audit, library_sha256=sha256(ROOT / \'research/lagged_sources/library.py\'),\n                 rainfall_decoded_for_fit=[str(training[0].date()), str(training[-1].date())],\n                 evaluation_rainfall_decoded=False, test_partition_read=False, sst_used=False)\n    return SimpleNamespace(lib=m, rain=rain, atmosphere=atmosphere, seas=seas, cfs=cfs, audit=audit)\n\n\nclass ForecastMaps(Dataset):\n    """Inference adapter that never reads target rainfall; zeros are unused loss placeholders."""\n    def __init__(self, context, dates, scaling):\n        self.context, self.dates = context, pd.DatetimeIndex(dates)\n        self.scaling = {k: np.asarray(v).copy() for k, v in scaling.items()}\n        require((self.dates > context.cutoff).all(), \'Forecast targets must follow training.\')\n        require(set(self.scaling) == {\'mean\', \'scale\'} and self.scaling[\'mean\'].shape == (31,)\n                and self.scaling[\'scale\'].shape == (31,), \'Invalid training scaler.\')\n        require(np.isfinite(self.scaling[\'mean\']).all() and np.isfinite(self.scaling[\'scale\']).all()\n                and (self.scaling[\'scale\'] > 0).all(), \'Invalid training scaler values.\')\n\n    def __len__(self):\n        return len(self.dates)\n\n    def __getitem__(self, index):\n        features = self.context.features(self.dates[index], training=False)\n        base = features[22].copy()\n        x = (features - self.scaling[\'mean\'][:, None, None]) / self.scaling[\'scale\'][:, None, None]\n        require(np.isfinite(x).all(), \'Nonfinite forecast features.\')\n        return torch.from_numpy(x), torch.zeros_like(torch.from_numpy(base)), torch.from_numpy(base)\n\n\ndef check(official=None, seas5=None, cfsv2=None, device=\'cuda\'):\n    plan, _ = protocol()\n    _, indices = index_snapshot(plan)\n    paths, audit = preflight(official, seas5, cfsv2, final=True)\n    require(device in {\'cpu\', \'cuda\'}, \'Device must be cpu or cuda.\')\n    require(device != \'cuda\' or torch.cuda.is_available(), \'Enable a Kaggle GPU before fitting, or explicitly select cpu.\')\n    # These are already prepared seasonal inputs; no provider access is needed.\n    print(\'Ready: training files, seasonal partitions and ocean indices checked through\', indices[\'selected_end\'],\n          \'| exact shared NOAA months:\', indices[\'shared_months\'], \'| no model fitted.\', flush=True)\n    return paths, audit\n\n\ndef freeze(output, official=None, seas5=None, cfsv2=None, device=\'cuda\'):\n    plan, config = protocol()\n    paths, audit = check(official, seas5, cfsv2, device)\n    output = Path(output).resolve()\n    require(not output.exists(), \'Output already exists. Preserve it and choose a fresh run directory.\')\n    for path in paths.values():\n        require(not output.is_relative_to(path), \'Output must not be inside an input dataset.\')\n    require(not output.is_relative_to(ROOT / \'research\') and not output.is_relative_to(ROOT / \'competition\'), \'Output must be separate from source code.\')\n    output.mkdir(parents=True)\n    started = utcnow()\n    write_json(output / \'state.json\', dict(status=\'fitting\', started_at=started, forecast_issued=False))\n    try:\n        code = {p.relative_to(ROOT).as_posix(): sha256(p) for p in source_files()}\n        for name in code:\n            target = output / \'code\' / name\n            target.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copy2(ROOT / name, target)\n            require(sha256(target) == code[name], \'Code changed during snapshot.\')\n        write_json(output / \'plan.json\', plan)\n        write_json(output / \'signature.json\', dict(plan=plan, source_protocol=config, code=code, inputs=audit,\n                   python=platform.python_version(), torch=torch.__version__, device=device,\n                   gpu=torch.cuda.get_device_name(0) if device == \'cuda\' else None, started_at=started))\n        data = load_inputs(paths, audit, output, plan)\n        training, targets = calendar(plan)\n        inner_train, inner_reference, inner_dates = split_inner(training, training, config[\'inner_validation_months\'])\n        write_json(output / \'split.json\', dict(training=training.strftime(\'%Y-%m\').tolist(),\n                   inner_training=inner_train.strftime(\'%Y-%m\').tolist(), inner_reference=inner_reference.strftime(\'%Y-%m\').tolist(),\n                   inner_validation=inner_dates.strftime(\'%Y-%m\').tolist(), evaluation=targets.strftime(\'%Y-%m\').tolist(),\n                   independent_holdout=False, evaluation_rainfall_decoded=False))\n        data.lib.calendario_emissoes(targets, training[-1]).to_csv(output / \'source_calendar.csv\', index=False)\n        state = baseline.fit(data, training[-1], output / \'hybrid\')\n        hybrid = baseline.predict(data, state, targets)\n        base_climate = state[\'climate\'].copy()\n        del state\n        gc.collect()\n        torch_device = torch.device(device)\n        if device == \'cpu\':\n            torch.set_num_threads(4)\n        scratch = output / \'scratch\'\n        scratch.mkdir()\n        with tempfile.TemporaryDirectory(dir=scratch) as temporary:\n            temporary = Path(temporary)\n            context = Context(data, inner_reference, inner_train)\n            context.save(output / \'inner\')\n            train = MonthlyMaps(context, inner_train, temporary / \'inner_train\', training=True)\n            valid = MonthlyMaps(context, inner_dates, temporary / \'inner_valid\', scaling=train.scaling)\n            try:\n                net, epochs = fit_network(train, config, output / \'inner\', torch_device, validation=valid)\n            finally:\n                train.close(); valid.close()\n            del net, train, valid, context\n            gc.collect()\n            if device == \'cuda\':\n                torch.cuda.empty_cache()\n            context = Context(data, training, training)\n            context.save(output / \'unet\')\n            np.testing.assert_array_equal(base_climate.values, context.climate.values)\n            train = MonthlyMaps(context, training, temporary / \'outer_train\', training=True)\n            try:\n                net, _ = fit_network(train, config, output / \'unet\', torch_device, epochs=epochs)\n                scaling = {k: v.copy() for k, v in train.scaling.items()}\n            finally:\n                train.close()\n            forecast = inference_maps(net, ForecastMaps(context, targets, scaling), torch_device, config[\'batch_size\'])\n            del net, train\n        coords = dict(time=targets, lat=data.rain.lat, lon=data.rain.lon)\n        unet = xr.DataArray(forecast, dims=(\'time\', \'lat\', \'lon\'), coords=coords, attrs=dict(units=\'mm/day\'))\n        blend = .75 * hybrid.astype(\'float64\') + .25 * unet.astype(\'float64\')\n        blend.attrs[\'units\'] = \'mm/day\'\n        climate = xr.DataArray(context.climate.values[targets.month - 1], dims=unet.dims, coords=coords, attrs=dict(units=\'mm/day\'))\n        predictions = xr.Dataset(dict(hybrid=hybrid, unet=unet, blend=blend, climatology=climate),\n            attrs=dict(protocol=plan[\'id\'], training_end=plan[\'training_end\'], independent_holdout=\'false\', retrospective=\'true\'))\n        for field in predictions.data_vars.values():\n            require(np.isfinite(field).all() and (field >= 0).all(), \'Invalid prediction map.\')\n        predictions.to_netcdf(output / \'predictions.nc\', engine=\'h5netcdf\', encoding={k: dict(zlib=True, complevel=4) for k in predictions})\n        with xr.open_dataset(output / \'predictions.nc\') as restored:\n            xr.testing.assert_identical(restored, predictions)\n        require(all(sha256(ROOT / name) == expected for name, expected in code.items()), \'Source code changed during fitting.\')\n        _, end_audit = preflight(**paths, final=True)\n        require(end_audit == audit, \'Input files changed during fitting.\')\n        frozen = dict(schema=\'rainfall_temporal_extension_frozen_v1\', frozen_at=utcnow(), started_at=started,\n            evaluation_rainfall_decoded=False, independent_holdout=False, model_promoted=False, forecast_issued=False,\n            observation_file_sha256=audit[\'files\'][\'treino_tp.nc\'], selected_epochs=epochs,\n            files={p.relative_to(output).as_posix(): sha256(p) for p in sorted(output.rglob(\'*\')) if p.is_file() and p.name != \'state.json\'})\n        write_json(output / \'frozen.json\', frozen)\n        write_json(output / \'state.json\', dict(status=\'predictions_frozen\', frozen_at=frozen[\'frozen_at\'], forecast_issued=False))\n        print(\'Historical predictions frozen:\', output, \'| selected epochs:\', epochs, flush=True)\n        return output\n    except (Exception, KeyboardInterrupt) as error:\n        write_json(output / \'state.json\', dict(status=\'failed\', error_type=type(error).__name__, error=str(error), forecast_issued=False))\n        raise\n\n\ndef verify_frozen(output):\n    output = Path(output).resolve()\n    require((output / \'frozen.json\').is_file(), \'Freeze predictions before opening evaluation rainfall.\')\n    frozen = json.loads((output / \'frozen.json\').read_text())\n    require(frozen[\'schema\'] == \'rainfall_temporal_extension_frozen_v1\' and frozen[\'evaluation_rainfall_decoded\'] is False,\n            \'Invalid freeze record.\')\n    require({\'predictions.nc\', \'signature.json\', \'plan.json\', \'split.json\'} <= set(frozen[\'files\']), \'Incomplete frozen package.\')\n    for name, digest in frozen[\'files\'].items():\n        path = (output / name).resolve()\n        require(path.is_relative_to(output) and path != output, \'Frozen file path leaves the run.\')\n        require(path.is_file() and sha256(path) == digest, f\'Frozen artifact changed or missing: {name}\')\n    return frozen\n\n\ndef evaluate(output, observations):\n    output, observations = Path(output), Path(observations)\n    frozen = verify_frozen(output)\n    require(sha256(observations) == frozen[\'observation_file_sha256\'], \'Observation snapshot differs from the training archive.\')\n    destination = output / \'evaluation\'\n    require(not destination.exists(), \'This frozen run has already been opened for evaluation. Preserve its report.\')\n    destination.mkdir()\n    write_json(destination / \'opened.json\', dict(opened_at=utcnow(), frozen_record_sha256=sha256(output / \'frozen.json\'),\n               scope=\'retrospective extension; not an independent holdout\', weight_search=False,\n               evaluation_code={p.relative_to(ROOT).as_posix(): sha256(p) for p in source_files()},\n               python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, xarray=xr.__version__))\n    # Observation values are decoded for the first time in this evaluation path.\n    plan = json.loads((output / \'plan.json\').read_text())\n    _, dates = calendar(plan)\n    with xr.open_dataset(observations) as source, xr.open_dataset(output / \'predictions.nc\') as predictions:\n        require(source.tp.attrs.get(\'units\') == \'mm/day\', \'Unexpected verification units.\')\n        obs = source.tp.sel(time=dates).transpose(\'time\', \'lat\', \'lon\').load()\n        predictions.load()\n        require(set(predictions.data_vars) == set(MODELS), \'Unexpected candidate models.\')\n        require(pd.DatetimeIndex(predictions.time.values).equals(dates), \'Wrong prediction calendar.\')\n        obs, predictions = xr.align(obs, predictions, join=\'exact\')\n        require(np.isfinite(obs).all() and (obs >= 0).all(), \'Invalid verification observations.\')\n        for model in MODELS:\n            require(predictions[model].dims == (\'time\', \'lat\', \'lon\') and predictions[model].attrs.get(\'units\') == \'mm/day\', \'Invalid forecast metadata.\')\n        m = library()\n        monthly = m.pesquisa_diagnosticos(obs, {model: predictions[model] for model in MODELS}, \'D_2021_2022\')\n    monthly.to_csv(destination / \'monthly_metrics.csv\', index=False)\n    domain = monthly[monthly.regiao == \'dominio_inteiro\']\n    for name, table, groups in [(\'global\', domain, [\'modelo\']), (\'years\', domain, [\'modelo\', \'ano\']),\n                               (\'calendar\', domain, [\'modelo\', \'mes\']), (\'regions\', monthly, [\'modelo\', \'regiao\'])]:\n        m.pesquisa_resumir(table, groups).to_csv(destination / (name + \'.csv\'), index=False)\n    values = pd.read_csv(destination / \'global.csv\').set_index(\'modelo\')\n    require((values.n == 24 * 301 * 261).all(), \'Incomplete verification.\')\n    delta = float(values.loc[\'blend\', \'rmse\'] - values.loc[\'hybrid\', \'rmse\'])\n    result = dict(status=\'evaluated\', protocol=plan[\'id\'], independent_holdout=False,\n        hybrid_rmse=float(values.loc[\'hybrid\', \'rmse\']), unet_rmse=float(values.loc[\'unet\', \'rmse\']),\n        blend_rmse=float(values.loc[\'blend\', \'rmse\']), delta_rmse=delta,\n        delta_mae=float(values.loc[\'blend\', \'mae\'] - values.loc[\'hybrid\', \'mae\']),\n        delta_absolute_bias=float(abs(values.loc[\'blend\', \'vies\']) - abs(values.loc[\'hybrid\', \'vies\'])),\n        model_promoted=False, forecast_issued=False, evaluated_at=utcnow())\n    write_json(destination / \'summary.json\', result)\n    from .report import render\n    render(destination, result, plan)\n    write_json(destination / \'complete.json\', dict(frozen_record_sha256=sha256(output / \'frozen.json\'),\n        files={p.name: sha256(p) for p in destination.iterdir() if p.is_file()}))\n    print(json.dumps(result, indent=2), flush=True)\n    return result\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    commands = parser.add_subparsers(dest=\'command\', required=True)\n    for command in [\'check\', \'freeze\']:\n        p = commands.add_parser(command)\n        for name in [\'official\', \'seas5\', \'cfsv2\']:\n            p.add_argument(\'--\' + name)\n        p.add_argument(\'--device\', choices=[\'cpu\', \'cuda\'], default=\'cuda\')\n        if command == \'freeze\':\n            p.add_argument(\'--output\', required=True)\n    p = commands.add_parser(\'evaluate\')\n    p.add_argument(\'--output\', required=True)\n    p.add_argument(\'--observations\', required=True)\n    args = vars(parser.parse_args())\n    command = args.pop(\'command\')\n    globals()[command](**args)\n', 'research/temporal_extension/protocol.json': '{\n  "id": "temporal_extension_2021_2022_v1",\n  "purpose": "Evaluate the unchanged models and fixed 75/25 blend on an additional chronological block.",\n  "scope": "retrospective_temporal_extension_not_independent_holdout",\n  "prior_use": "2021-2022 observations have participated in earlier final training. No separate scores for this block were found in the reviewed research records. 2023-2024 leaderboard results and the 2025 retrospective test have already been consulted.",\n  "evaluation_start": "2021-01-01",\n  "evaluation_end": "2022-12-01",\n  "training_start": "1990-10-01",\n  "training_end": "2020-09-01",\n  "training_months": 360,\n  "inner_validation_months": 24,\n  "source_protocol_sha256": "b5819403ce177b1c69cfd698160c0e2339f040d5d9ab0a4ad9608a469872c694",\n  "weights": {"hybrid": 0.75, "unet": 0.25},\n  "lags": {"atmosphere": 4, "indices": 3, "seasonal_initialization": 1, "training_rainfall": 4},\n  "models_fixed_for_entire_evaluation_block": true,\n  "epoch_selection": "Unchanged inner chronological selection before the outer block, followed by a fresh outer fit from the original seed.",\n  "evaluation_labels": "Not decoded by the fit/freeze path; load only after the immutable prediction record verifies.",\n  "primary_metric": "Pooled full-grid RMSE of the fixed blend versus the refitted hybrid",\n  "secondary_metrics": ["MAE", "signed bias", "area-weighted RMSE", "year", "calendar month", "latitude band"],\n  "weight_search": false,\n  "hyperparameter_search": false,\n  "automatic_promotion": false,\n  "forecast_issued": false,\n  "historical_publication_vintages_verified": false\n}\n', 'research/temporal_extension/report.py': '"""A small standalone report for the additional two-year comparison."""\nfrom html import escape\nfrom pathlib import Path\nimport base64\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\n\n\ndef render(output, result, plan):\n    output = Path(output)\n    table = pd.read_csv(output / \'global.csv\')\n    table = table[[\'modelo\', \'rmse\', \'mae\', \'vies\', \'rmse_area\', \'n\']].rename(columns={\n        \'modelo\': \'Model\', \'rmse\': \'RMSE\', \'mae\': \'MAE\', \'vies\': \'Bias\', \'rmse_area\': \'Area-weighted RMSE\', \'n\': \'Samples\'})\n    months = pd.read_csv(output / \'monthly_metrics.csv\')\n    months = months[months.regiao == \'dominio_inteiro\'].copy()\n    months[\'rmse\'] = np.sqrt(months.sse / months.n)\n    colors = dict(hybrid=\'#087e8b\', unet=\'#db654a\', blend=\'#6246a5\', climatology=\'#9299a1\')\n    fig, ax = plt.subplots(figsize=(12, 4.5), constrained_layout=True)\n    for model in [\'hybrid\', \'unet\', \'blend\']:\n        rows = months[months.modelo == model].sort_values(\'mes_alvo\')\n        ax.plot(pd.to_datetime(rows.mes_alvo), rows.rmse, label=model.title(), color=colors[model])\n    ax.set(xlabel=\'Target month\', ylabel=\'Full-grid RMSE (mm/day)\', title=\'Additional chronological block: 2021–2022\')\n    ax.legend(frameon=False)\n    fig.savefig(output / \'monthly_comparison.png\', dpi=150)\n    plt.close(fig)\n    encoded = base64.b64encode((output / \'monthly_comparison.png\').read_bytes()).decode(\'ascii\')\n    delta = result[\'delta_rmse\']\n    parts = [\'<h1>Temporal extension: 2021–2022</h1>\',\n             f\'<p>Fixed 75% hybrid + 25% U-Net. Blend minus hybrid RMSE: <strong>{delta:+.6f} mm/day</strong>.</p>\',\n             \'<p>Retrospective extension, not an independent holdout. These years were used in earlier final training. This run refitted every component only on October 1990–September 2020, with epochs chosen inside that training window. Predictions were saved before evaluation labels were decoded.</p>\',\n             table.to_html(index=False, float_format=lambda value: f\'{value:.6f}\', border=0),\n             f\'<img alt="Monthly comparison" src="data:image/png;base64,{encoded}">\']\n    for name in [\'years\', \'regions\', \'calendar\']:\n        frame = pd.read_csv(output / (name + \'.csv\'))\n        cols = [c for c in [\'modelo\', \'ano\', \'mes\', \'regiao\', \'rmse\', \'mae\', \'vies\', \'rmse_area\', \'n\'] if c in frame]\n        frame = frame[cols].rename(columns={\'modelo\': \'Model\', \'ano\': \'Year\', \'mes\': \'Month\', \'regiao\': \'Region\', \'vies\': \'Bias\'})\n        parts += [f\'<details><summary>{escape(name.title())}</summary>\' + frame.to_html(index=False, float_format=lambda value: f\'{value:.6f}\', border=0) + \'</details>\']\n    parts += [\'<p>Full supplied grid, including ocean. The unweighted pooled RMSE is primary; area-weighted RMSE is secondary. Two years provide limited temporal diversity. No alternative blend weights were tried. No operational forecast was issued and no model was automatically promoted.</p>\',\n              \'<p>Historical inputs are consolidated source snapshots, not reconstructed publication vintages. Saving a prediction before scoring a historical target is an execution safeguard, not proof that the target was previously unknown to the project.</p>\']\n    html = \'<!doctype html><html lang="en"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Rainfall temporal extension</title><style>body{font:16px/1.55 system-ui,sans-serif;max-width:1100px;margin:40px auto;padding:0 24px;color:#172b36}img{max-width:100%}table{display:block;overflow:auto;border-collapse:collapse}th,td{padding:8px 12px;border-bottom:1px solid #ddd;text-align:right}details{margin:20px 0}summary{cursor:pointer;font-weight:600}</style><body>\' + \'\\n\'.join(parts) + \'</body></html>\'\n    (output / \'report.html\').write_text(html, encoding=\'utf-8\', newline=\'\\n\')\n', 'research/temporal_extension/test_extension.py': '"""Synthetic checks for withheld targets, unchanged inference and freeze integrity."""\nfrom pathlib import Path\nfrom contextlib import redirect_stdout\nimport io\nimport json\nimport tempfile\nfrom types import SimpleNamespace\nimport unittest\nfrom unittest.mock import patch\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport xarray as xr\n\nfrom research.common.inputs import Context, write_json, sha256\nfrom research.common.synthetic_fixture import fixture\nfrom research.spatial_unet.network import RainfallUNet, seed_everything\nfrom research.spatial_unet.training import MonthlyMaps, inference_maps, split_inner\nfrom research.temporal_extension.experiment import (\n    ForecastMaps, calendar, protocol, read_training_rain, verify_frozen, evaluate,\n    index_snapshot, require_index_coverage, DEVELOPMENT_INDICES, INDEX_SNAPSHOT,\n)\n\n\nclass ExtensionChecks(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        torch.set_num_threads(2)\n\n    def test_locked_dates_and_inner_label_gap(self):\n        plan, config = protocol()\n        training, targets = calendar(plan)\n        fit, reference, valid = split_inner(training, training, config[\'inner_validation_months\'])\n        self.assertEqual(targets[0], pd.Timestamp(\'2021-01-01\'))\n        self.assertEqual(training[-1], pd.Timestamp(\'2020-09-01\'))\n        self.assertEqual(valid[0], pd.Timestamp(\'2018-10-01\'))\n        self.assertEqual(fit[-1], pd.Timestamp(\'2018-06-01\'))\n        self.assertEqual(reference[0], pd.Timestamp(\'1990-10-01\'))\n\n    def test_archived_indices_cover_first_and_last_forecast_without_changing_training(self):\n        plan, _ = protocol()\n        indices, audit = index_snapshot(plan)\n        self.assertEqual(len(indices), 387)\n        self.assertEqual(indices.index[-1], pd.Timestamp(\'2022-09-01\'))\n        self.assertEqual(audit[\'shared_months\'], 526)\n        self.assertTrue(audit[\'overlap_exact\'])\n        reference = pd.read_csv(DEVELOPMENT_INDICES, parse_dates=[\'time_origem\']).set_index(\'time_origem\')\n        training, targets = calendar(plan)\n        training_origins = training - pd.DateOffset(months=3)\n        pd.testing.assert_frame_equal(indices.loc[training_origins], reference.loc[training_origins])\n        archive = pd.read_csv(INDEX_SNAPSHOT, parse_dates=[\'time_origem\']).set_index(\'time_origem\')\n        for date in [targets[0], targets[-1]]:\n            origin = date - pd.DateOffset(months=3)\n            np.testing.assert_array_equal(indices.loc[origin].values, archive.loc[origin].values)\n\n    def test_old_development_table_rejected_before_fitting(self):\n        table = pd.read_csv(DEVELOPMENT_INDICES, parse_dates=[\'time_origem\']).set_index(\'time_origem\')\n        targets = pd.date_range(\'2021-01-01\', \'2022-12-01\', freq=\'MS\')\n        with self.assertRaisesRegex(ValueError, \'Missing: 2020-10\'):\n            require_index_coverage(table, targets - pd.DateOffset(months=3))\n\n    def test_training_reader_does_not_include_later_values(self):\n        dates = pd.date_range(\'2020-01-01\', periods=12, freq=\'MS\')\n        values = np.ones((12, 2, 3)); values[9:] = np.nan\n        field = xr.DataArray(values, dims=(\'time\', \'lat\', \'lon\'), coords=dict(time=dates, lat=[0, 1], lon=[0, 1, 2]), attrs=dict(units=\'mm/day\'))\n        with tempfile.TemporaryDirectory() as temporary:\n            path = Path(temporary) / \'rain.nc\'\n            xr.Dataset(dict(tp=field)).to_netcdf(path, engine=\'h5netcdf\')\n            selected = read_training_rain(path, dates[:9])\n            self.assertEqual(selected.sizes[\'time\'], 9)\n            self.assertTrue(np.isfinite(selected).all())\n\n    def test_label_free_inference_matches_original_adapter(self):\n        data = fixture()\n        reference = pd.date_range(\'1976-10-01\', \'2006-09-01\', freq=\'MS\')\n        training = reference[reference >= \'1982-03-01\']\n        context = Context(data, reference, training)\n        dates = pd.date_range(\'2007-01-01\', periods=2, freq=\'MS\')\n        scaling = dict(mean=np.arange(31, dtype=\'float32\'), scale=np.arange(1, 32, dtype=\'float32\'))\n        seed_everything()\n        net = RainfallUNet(widths=(4, 8, 16))\n        # Nonzero outputs ensure the comparison exercises feature normalization.\n        with torch.no_grad():\n            net.output.weight.fill_(.02)\n        with tempfile.TemporaryDirectory() as temporary:\n            original = MonthlyMaps(context, dates, Path(temporary) / \'original\', scaling=scaling)\n            try:\n                expected = inference_maps(net, original, torch.device(\'cpu\'))\n                data.rain = data.rain.sel(time=reference)\n                self.assertFalse(dates.isin(data.rain.time.values).any())\n                actual = inference_maps(net, ForecastMaps(context, dates, scaling), torch.device(\'cpu\'))\n                np.testing.assert_array_equal(actual, expected)\n            finally:\n                original.close()\n\n    def test_feature_path_cannot_touch_rainfall(self):\n        class ContextWithoutRain:\n            cutoff = pd.Timestamp(\'2020-09-01\')\n            data = SimpleNamespace()  # No rainfall attribute exists.\n            def features(self, date, training=False):\n                return np.ones((31, 2, 3), dtype=\'float32\')\n        maps = ForecastMaps(ContextWithoutRain(), [\'2021-01-01\'], dict(mean=np.zeros(31), scale=np.ones(31)))\n        x, dummy, base = maps[0]\n        self.assertEqual(tuple(x.shape), (31, 2, 3))\n        self.assertEqual(float(dummy.sum()), 0.)\n        self.assertEqual(float(base.sum()), 6.)\n\n    def test_scoring_requires_unchanged_frozen_predictions_before_opening_targets(self):\n        with tempfile.TemporaryDirectory() as temporary:\n            root = Path(temporary)\n            with patch(\'research.temporal_extension.experiment.xr.open_dataset\', side_effect=AssertionError(\'Targets opened\')):\n                with self.assertRaisesRegex(ValueError, \'Freeze predictions\'):\n                    evaluate(root, root / \'rain.nc\')\n                for name in [\'predictions.nc\', \'signature.json\', \'plan.json\', \'split.json\']:\n                    (root / name).write_text(\'fixture\')\n                record = dict(schema=\'rainfall_temporal_extension_frozen_v1\', evaluation_rainfall_decoded=False,\n                              files={p.name: sha256(p) for p in root.iterdir()})\n                write_json(root / \'frozen.json\', record)\n                verify_frozen(root)\n                (root / \'predictions.nc\').write_text(\'altered\')\n                with self.assertRaisesRegex(ValueError, \'changed or missing\'):\n                    evaluate(root, root / \'rain.nc\')\n\n    def test_complete_evaluation_uses_pooled_errors_and_preserves_freeze(self):\n        # Alternating errors of 1 and 3 have RMSE sqrt(5), not mean RMSE 2.\n        plan, _ = protocol()\n        _, dates = calendar(plan)\n        coords = dict(time=dates, lat=np.arange(-60, 15.25, .25), lon=np.arange(-90, -24.75, .25))\n        observed = xr.DataArray(np.ones((24, 301, 261), dtype=\'float32\'),\n                                dims=(\'time\', \'lat\', \'lon\'), coords=coords, attrs=dict(units=\'mm/day\'))\n        error = xr.DataArray(np.tile([1., 3.], 12), dims=\'time\', coords=dict(time=dates))\n        predictions = xr.Dataset(dict(hybrid=observed + error, unet=observed + 2,\n                                      blend=observed + .75 * error + .5, climatology=observed * 0))\n        for field in predictions.data_vars.values():\n            field.attrs[\'units\'] = \'mm/day\'\n        with tempfile.TemporaryDirectory() as temporary:\n            root = Path(temporary)\n            observed.to_dataset(name=\'tp\').to_netcdf(root / \'rain.nc\', engine=\'h5netcdf\')\n            predictions.to_netcdf(root / \'predictions.nc\', engine=\'h5netcdf\')\n            write_json(root / \'plan.json\', plan)\n            for name in [\'signature.json\', \'split.json\']:\n                write_json(root / name, dict(synthetic=True))\n            write_json(root / \'frozen.json\', dict(schema=\'rainfall_temporal_extension_frozen_v1\',\n                evaluation_rainfall_decoded=False, observation_file_sha256=sha256(root / \'rain.nc\'),\n                files={name: sha256(root / name) for name in [\'plan.json\', \'signature.json\', \'split.json\', \'predictions.nc\']}))\n            original_record = (root / \'frozen.json\').read_bytes()\n            with redirect_stdout(io.StringIO()):\n                result = evaluate(root, root / \'rain.nc\')\n            self.assertAlmostEqual(result[\'hybrid_rmse\'], np.sqrt(5))\n            self.assertAlmostEqual(result[\'blend_rmse\'], np.sqrt((1.25**2 + 2.75**2) / 2))\n            self.assertFalse(result[\'independent_holdout\'])\n            self.assertFalse(result[\'model_promoted\'])\n            self.assertEqual(original_record, (root / \'frozen.json\').read_bytes())\n            report = root / \'evaluation\'\n            self.assertIn(\'not an independent holdout\', (report / \'report.html\').read_text(encoding=\'utf-8\'))\n            for name, digest in json.loads((report / \'complete.json\').read_text())[\'files\'].items():\n                self.assertEqual(sha256(report / name), digest)\n            self.assertEqual(len(pd.read_csv(report / \'years.csv\')), 8)\n            with self.assertRaisesRegex(ValueError, \'already been opened\'):\n                evaluate(root, root / \'rain.nc\')\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n'}
BUNDLE_SHA256 = {'competition/metadata/NOAA/indices_noaa.csv': 'b6f444fbec48681c1e5b46a5c22aa197ad91c7fbbdaff82c3ed3a9c63f7c15a4', 'research/common/__init__.py': '4f6bfaae6ba45669c7d41e1b087df0582970932f06cdea87a4ff3b352f4135f0', 'research/common/baseline.py': '653594bf49fa338304129d7d22e7dc31a4e32647b26ee87af7bea4bad94a4dfa', 'research/common/inputs.py': 'd22140d4ad3db4ac3fecdca26c7ebdd1d780486b51622a1f333fce2a637e6851', 'research/common/synthetic_fixture.py': 'da4de7a662c2c8c25f324c5b543ab04914d490ffde5e316c39187eb8f1640bff', 'research/lagged_sources/library.py': '4b92dd77d868d7806fd0ae37d2837abfcde36e173df75b1bd6d88760593ee9c3', 'research/lagged_sources/ocean_indices.csv': '69226b2e65f783b4d4971a254d1e325d0908447c76ec1df3bf532ecef1f0cf5b', 'research/lagged_sources/official_hashes.json': 'bd5648197c4cf25f85d89a32c701b422f1e0b788421766cc7b8aca99ea42223d', 'research/spatial_unet/__init__.py': 'ee933fd666f8d33250d76a02fcd8b898d3a2dc7b341dde8165411387cfa27c19', 'research/spatial_unet/network.py': 'f1d10bbe1634018857b4fda8ae2753c1cf43d553f221e338bd1d6dcd89f40654', 'research/spatial_unet/protocol.json': 'b5819403ce177b1c69cfd698160c0e2339f040d5d9ab0a4ad9608a469872c694', 'research/spatial_unet/run_experiment.py': '4484f2a6c0b29bee18e003ecfab880f0e16e192f1d1b9c71e90bc486f1cd6412', 'research/spatial_unet/test_spatial.py': '65215314cb5ebd1229ded058c4aab91ab71b9ceccc6af8a83de25668301a8516', 'research/spatial_unet/training.py': '698d27f397a312c3a4a5f832a15fa09fe431095431b74ca328a8034c3f232275', 'research/temporal_extension/__init__.py': '73a12c9b4224de54cff998c920ba7d4d20d896ce3a16b89e85ecffb5a36cd9ba', 'research/temporal_extension/experiment.py': '5172ae29a3575c9bfeb21e5e1396ec09a622a41e34f6fc8dfe7eaf689b9708e7', 'research/temporal_extension/protocol.json': '367fa9da29f45286c5ce2a8358bf87f415a69d35918c16fc129fffe61c5a8006', 'research/temporal_extension/report.py': 'b9ab84a8091d17618702a7a56a85ca054481dd0268af3db2f56cadc8fcb3d179', 'research/temporal_extension/test_extension.py': 'c83ab2d1fd39d7588613bac3cd85a90e085ee82447236e72209741eaab45085d'}
for name, content in BUNDLE.items():
    assert hashlib.sha256(content.encode('utf-8')).hexdigest() == BUNDLE_SHA256[name]
    path = CODE_ROOT/name
    if path.exists() and path.read_text(encoding='utf-8') != content:
        raise RuntimeError('Different source code already exists. Preserve this run and choose a new RUN_ROOT.')
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        path.write_text(content, encoding='utf-8', newline='\n')
sys.path.insert(0, str(CODE_ROOT))
sys.path.insert(0, str(CODE_ROOT/'research/prospective'))
print('Reviewed source bundle:', CODE_ROOT)


## 1. Inputs and preflight
Attach the official dataset and both prepared seasonal datasets. No archived fold models or SST are required. This cell does not train or evaluate the 2021–2022 targets.

In [ ]:
import unittest
from research.temporal_extension.experiment import check, freeze, evaluate, verify_frozen
OFFICIAL = None
SEAS5 = None
CFSV2 = None
DEVICE = "cuda"
checks = unittest.defaultTestLoader.loadTestsFromNames([
    "research.temporal_extension.test_extension", "research.spatial_unet.test_spatial"
])
assert unittest.TextTestRunner(verbosity=2).run(checks).wasSuccessful(), "Synthetic checks failed."
PATHS, INPUT_AUDIT = check(OFFICIAL, SEAS5, CFSV2, DEVICE)
print({name: str(path) for name, path in PATHS.items()})


## 2. Fit and freeze, or restore a complete frozen run
Leave `PREVIOUS_RUN=None` for the first run. To continue after a session reset, attach its saved output and enter the directory containing `frozen.json`. This verifies and restores its package without refitting. Never substitute the final operational model trained through 2022.

In [ ]:
from datetime import datetime, timezone
PREVIOUS_RUN = None  # Example: "/kaggle/input/.../temporal_extension/runs/20260930T..."
if PREVIOUS_RUN is None:
    RUN = RUN_ROOT/"runs"/datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    freeze(RUN, **PATHS, device=DEVICE)
else:
    source = Path(PREVIOUS_RUN).resolve()
    verify_frozen(source)
    RUN = RUN_ROOT/"restored"/source.name
    if RUN.exists():
        verify_frozen(RUN)
        assert (RUN/"frozen.json").read_bytes() == (source/"frozen.json").read_bytes(), "Different restored run."
    else:
        shutil.copytree(source, RUN)
    verify_frozen(RUN)
print("Frozen historical prediction package:", RUN)


## 3. Open the target rainfall and evaluate once
The four forecast fields are already frozen. This cell verifies their package before decoding January 2021–December 2022 observations. It does not select weights, epochs or a winner for operational use.

In [ ]:
from research.common.inputs import sha256
if (RUN/"evaluation"/"complete.json").is_file():
    verify_frozen(RUN)
    completed = json.loads((RUN/"evaluation"/"complete.json").read_text())
    assert completed["frozen_record_sha256"] == sha256(RUN/"frozen.json")
    for name, expected in completed["files"].items():
        artifact = (RUN/"evaluation"/name).resolve()
        assert artifact.is_relative_to((RUN/"evaluation").resolve()), "Invalid evaluation path."
        assert sha256(artifact) == expected, "Changed evaluation artifact: " + name
    RESULT = json.loads((RUN/"evaluation"/"summary.json").read_text())
    print("Reusing the previously completed evaluation; no new scoring.")
else:
    RESULT = evaluate(RUN, PATHS["official"]/"treino_tp.nc")
print(json.dumps(RESULT, indent=2))


## 4. Read and export the results
Download the small report ZIP for review. Save the complete notebook output to retain frozen models and prediction maps; the report ZIP alone cannot restore a run.

In [ ]:
import zipfile
import pandas as pd
from IPython.display import display, FileLink, HTML
display(pd.read_csv(RUN/"evaluation"/"global.csv"))
display(HTML((RUN/"evaluation"/"report.html").read_text(encoding="utf-8")))
REPORT_ZIP = RUN_ROOT/"temporal_extension_reports.zip"
with zipfile.ZipFile(REPORT_ZIP, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in RUN.rglob("*"):
        if path.is_file() and path.suffix in {".json", ".csv", ".png", ".html"} and "code" not in path.relative_to(RUN).parts:
            archive.write(path, path.relative_to(RUN))
display(FileLink(str(REPORT_ZIP.relative_to(BASE))))
display(FileLink(str((RUN/"evaluation"/"report.html").relative_to(BASE))))
print("Save Version with all outputs. The small ZIP excludes model weights and rainfall maps.")
